# ModelCompare_LoadTest - A/B load test of two semantic models

Replays the DAX queries captured with **Power BI Performance Analyzer** against two semantic models
(**A** = baseline, e.g. production; **B** = candidate, e.g. a feature-branch workspace) and shows a clear,
side-by-side comparison. The load is generated by this notebook as soon as you run it; it does not wait for real users.

**Finding the results**: the notebook has 10 steps. Each step has a heading, and its code is hidden, so under each heading you see only that step's result.
- Open **Contents** on the notebook command bar to jump to a step.
- To see the code of a step: cell **...** menu > **Show input**. **Hide input** hides it again.

**Steps** (Steps 3-7 save to the lakehouse as they go, so partial runs are kept)
1. **Parameters**: the only cell you edit (its code stays visible).
2. **Setup**: loads the helper code and finds the lakehouse.
3. **Preflight**: validates parameters, detects the run mode, prints the approval fingerprint, checks the scheduled-run gates, parses the query file, connects to both models (first-touch time), compares model definitions and estimates the duration. Nothing heavy runs until `confirm_run = True` (or a scheduled run passes its gates).
4. **Warm-up + result comparison**: every query once per model (first-run latency, row count, result checksum), then rows per table.
5. **Number check**: the measures the report uses, compared in both models under a controlled filter (a few random business units, a 10-day window that ends a week before the latest date both models have loaded, so late-arriving data has settled), so refresh timing can't explain a difference. A difference is flagged as CRITICAL with its likely cause; the run continues. Also runs in the dry run.
6. **Server cost**: both models are traced at the same time while the queries alternate A,B / B,A (single user, best effort), so both are measured under the same conditions: server duration and CPU per query.
7. **Load test**: for every concurrency level and iteration, alternating rounds A,B -> B,A -> ...; each virtual user clicks through the report like a person: the queries of one click run in parallel as captured, with a think time between clicks.
8. **Report**: starts with an **executive summary** for decision makers (blockers, trust checks, bottom line, pros, cons, trade-offs, questions, next steps, copy-as-text), then one chart: how long a user waits after each click. Everything else (number check, scorecard, server CPU per query, a plain per-visual table, errors, data-quality notes) is in one collapsed **Details for analysts** section. After a dry run: the approval summary. The report is also saved to `Files/LoadTest/Reports/`.
9. **History**: a table of all saved comparisons (one row each), then the trend of the runs with the same approval fingerprint.
10. **Text summary**: the exact numbers as plain text, for sharing (saved with the result JSON; a scheduled run ends with it).

**One-time setup**
1. Create a lakehouse (default name `lh_loadtest`) in a workspace on a Fabric capacity, create folder `Files/LoadTest/Queries/` and upload your `PowerBIPerformanceData.json` there.
2. This notebook must be a **Python** notebook (not PySpark). The *Parameters* cell (Step 1) must be the parameter cell (cell menu > *Toggle parameter cell*; already set when you import the `.ipynb`).
3. You need Read + Build (XMLA read) on both models; the server-cost trace may need Contributor (or higher) on both workspaces.

**Run**: set the parameters, *Run all* with `confirm_run = False` (dry run: checks the settings, runs the number check and ends with an approval summary; nothing is saved and no load is generated), review, set `confirm_run = True`, *Run all*.
**Scheduled runs** (schedule or pipeline, created in the Fabric UI): paste the fingerprint from the dry run into `schedule_approval` and save. A scheduled run checks its gates: outside the time window, an overlapping run or a recent refresh skips the run with the reason; a missing or outdated approval, or an unreachable model, fails it (the job alerts). See README: Scheduled runs.
To open a past comparison later: set `report_comparison_id` and run Steps 1, 2 and 8 (and optionally 10).

## Step 1 - Parameters
The only cell you edit. `confirm_run = False` is a dry run (checks only); `True` runs the real test. Scheduled runs use these values as saved, and need `schedule_approval`.

In [ ]:
# Step 1 - Parameters
# ---- What to compare (A = baseline, B = candidate) ---------------------------
test_name = "SLP v2 - Prod vs 7874 SLP"
a_label = "Prod"
a_workspace = "CFF Insights"
a_model = "prod_dataset_sales_location_planning_v2"
b_label = "Branch"
b_workspace = "FCS Insights [7874 SLP]"
b_model = "prod_dataset_sales_location_planning_v2"

# ---- Where the query file and the results live -------------------------------
host_workspace = ""                 # workspace of the lakehouse; "" = workspace of this notebook
lakehouse_name = "lh_loadtest"
query_file = "Files/LoadTest/Queries/PowerBIPerformanceData.json"   # path inside the lakehouse

# ---- Which captured queries to replay ----------------------------------------
include_user_actions = ""           # "" = all, in captured order. Or labels "Refresh,ChangePage" or action numbers "1,3" (shown in preflight)

# ---- Load shape ---------------------------------------------------------------
concurrency_levels = "1,3,5"        # virtual users per step, comma separated
iterations = 3                      # rounds per model per concurrency level (each round = one full replay per virtual user)
think_time_between_actions_sec = 60 # pause between two user actions (clicks); the queries of one click run in parallel, as captured, without pauses
cooldown_between_rounds_sec = 0     # pause between rounds (lets the capacity settle)
ramp_up_sec = 0                     # delay between starting each virtual user inside a round
query_timeout_sec = 600             # per query

# ---- Which test steps run --------------------------------------------------------
run_warmup = True                   # Step 4: warm-up pass (excluded from load statistics)
compare_results = True              # Step 4: row count + result checksum per query during warm-up
checksum_float_decimals = 6         # rounding before checksum (avoids false "different" flags)
run_number_check = True             # Step 5: numbers under a controlled filter (also runs in the dry run)
run_server_trace = True             # Step 6: traced single-user pass per model (server duration + CPU)
run_load_test = True                # Step 7: load test

# ---- Number check (Step 5) -------------------------------------------------------
number_check_bu_count = 3           # random business units, chosen from those with data in the window in both models
number_check_days = 10              # window length (days)
number_check_end_offset_days = 7    # the window ends this many days before the latest date both models have loaded (the newest days may still be loading)
number_check_tolerance_pct = 0.0001 # relative difference (%) still counted as equal; text values must match exactly
number_check_seed = 0               # 0 = new random pick (the seed is printed and saved); a saved seed repeats a pick
number_check_bu_column = ""         # "" = auto-detect; or e.g. "'Business Unit'[Business Unit]"
number_check_date_column = ""       # "" = auto-detect; or e.g. "'Date'[Date]"

# ---- Safety ----------------------------------------------------------------------
confirm_run = False                 # False = preflight + number check only. Set True after approval to load the models.
max_concurrency_guard = 5           # refuse concurrency levels (virtual users) above this value
max_concurrent_queries_guard = 15   # refuse when virtual users x parallel queries of one click exceeds this value

# ---- Scheduled runs (a schedule / pipeline passes no parameters: these defaults apply) ----------
run_mode = "auto"                   # "auto" = detect (pipeline or scheduled job = scheduled); or "interactive" / "scheduled"
schedule_approval = ""              # approval fingerprint printed by an interactive dry run; a scheduled run needs a match
schedule_window = ""                # allowed start time of a scheduled run, e.g. "22:00-05:00"; "" = any time
schedule_timezone = "UTC"           # time zone of schedule_window, e.g. "Europe/Berlin"
schedule_min_minutes_since_refresh = 30  # a scheduled run is skipped when a model refreshed more recently (empty caches)
warmup_passes = 1                   # warm-up passes per model: the 1st is measured (first run + checksum), more are unmeasured
save_report_files = True            # save report (.html), text summary (.txt) and result (.json) to Files/LoadTest/Reports/

# ---- Report ----------------------------------------------------------------------
noise_band_pct = 5                  # changes within +/- this % are shown as "no meaningful change"
top_n_queries = 15                  # number of most-changed queries shown in charts
report_comparison_id = ""           # "" = the test this run makes (only with confirm_run = True); or a comparison_id from Step 9 (History) to reopen a saved test
app_name = "ModelCompare_LoadTest"  # Application Name sent to the engine (visible in traces / monitoring)

## Step 2 - Setup
Loads the helper code and finds the lakehouse. Result: one "Setup OK" line.

In [ ]:
# Step 2 - Setup and helpers
import hashlib
import html as _html
import json
import math
import os
import random
import re
import tempfile
import threading
import time
import urllib.parse
import uuid
from datetime import datetime, timedelta, timezone
from decimal import Decimal

import numpy as np
import pandas as pd
import pyarrow as pa
from deltalake import DeltaTable, write_deltalake
from IPython.display import HTML, display
import sempy.fabric as fabric

# create_tom_server loads the ADOMD.NET / TOM assemblies into this Python process (same approach as RunPerfScenario).
try:
    _tom = fabric.create_tom_server(readonly=True)
except Exception:
    _tom = fabric.create_tom_server(readonly=True, workspace=a_workspace)
import System  # noqa: E402
from System.Globalization import CultureInfo  # noqa: E402
from Microsoft.AnalysisServices.AdomdClient import AdomdConnection  # noqa: E402

PARAM_NAMES = [
    "test_name", "a_label", "a_workspace", "a_model", "b_label", "b_workspace", "b_model",
    "host_workspace", "lakehouse_name", "query_file", "include_user_actions", "concurrency_levels",
    "iterations", "think_time_between_actions_sec",
    "cooldown_between_rounds_sec", "ramp_up_sec", "query_timeout_sec", "run_warmup", "compare_results",
    "checksum_float_decimals", "run_number_check", "run_server_trace", "run_load_test",
    "number_check_bu_count", "number_check_days", "number_check_end_offset_days", "number_check_tolerance_pct",
    "number_check_seed", "number_check_bu_column", "number_check_date_column", "confirm_run", "max_concurrency_guard",
    "max_concurrent_queries_guard", "noise_band_pct", "top_n_queries", "app_name",
    "run_mode", "schedule_approval", "schedule_window", "schedule_timezone", "schedule_min_minutes_since_refresh",
    "warmup_passes", "save_report_files",
]
# Settings that change the load on the models: a scheduled run needs an approval fingerprint of exactly these
# (report-only settings, the number-check seed and end offset are left out; scheduled runs always warm up).
FINGERPRINT_PARAMS = [
    "a_workspace", "a_model", "b_workspace", "b_model", "include_user_actions", "concurrency_levels", "iterations",
    "think_time_between_actions_sec", "cooldown_between_rounds_sec", "ramp_up_sec", "query_timeout_sec", "warmup_passes",
    "compare_results", "run_number_check", "run_server_trace", "run_load_test", "number_check_bu_count",
    "number_check_days", "number_check_bu_column", "number_check_date_column", "max_concurrency_guard",
    "max_concurrent_queries_guard",
]
SCHEDULED_CONTEXT_KEYS = ("isForPipeline", "isScheduledRun", "isForScheduler")   # truthy = not started by a person
REPORTS_DIR = "Files/LoadTest/Reports"
ROUND_PAIR_AGREE = 0.8               # a change is clear only if >= 80% of the A/B round pairs agree in direction ...
ROUND_PAIR_MIN = 3                   # ... and there are at least 3 pairs
MIN_SAMPLES = 5                      # fewer measurements (or no confidence interval) = no coloured verdict
ROUND_DRIFT_PCT = 20                 # a model's round medians at one level differing by more than this = round drift
TREND_RUNS = 5                       # the latest run is compared with the range of the last 5 comparable runs
TRANSIENT_ERRORS = ("throttling", "timeout", "connection")   # drop only the click they hit, not the query
PERSISTENT_FAIL_PCT = 20             # a query failing in more than 20% of its runs in a model is left out of click metrics
XMLA_PREFIX = "powerbi://api.powerbi.com/v1.0/myorg/"
RECONNECT_AFTER_SEC = 40 * 60        # reconnect (fresh token) before access tokens expire
PROGRESS_EVERY_SEC = 60
BOOT_N = 1000                        # bootstrap resamples for every 95% confidence interval in the report
ROWS_LIST_PCT = 1                    # tables whose row count differs by more than this % are listed
ROWS_TRADEOFF_PCT = 10               # ... and by more than this % are a trade-off in the executive summary
COLD_MS = 5000                       # a first touch slower than this means the model was cold (evicted)
COLOR = {"A": "#1f77b4", "B": "#ff7f0e"}
GREEN, RED, GREY, AMBER = "#2e7d32", "#c62828", "#757575", "#b26a00"
_FONT = "font-family:Segoe UI,Arial,sans-serif;font-size:13px"
MODE_NAMES = {0: "Import", 1: "DirectQuery", 2: "Default", 3: "Push", 4: "Dual", 5: "DirectLake"}
SOURCE_NAMES = {1: "Query", 2: "Calculated", 3: "None", 4: "M", 5: "Entity", 6: "PolicyRange",
                7: "CalculationGroup", 8: "Inferred", 9: "Parquet"}
_INV = CultureInfo.InvariantCulture

MODELS = {
    "A": {"key": "A", "label": a_label, "workspace": a_workspace, "model": a_model},
    "B": {"key": "B", "label": b_label, "workspace": b_workspace, "model": b_model},
}


# ---------- small utilities ----------
def utcnow():
    return datetime.now(timezone.utc)


def _parse_list(value):
    return [t.strip() for t in re.split(r"[,;]", str(value or "")) if t.strip()]


def _parse_int_list(value):
    return [int(t) for t in _parse_list(value)]


def _slug(text):
    return (re.sub(r"[^A-Za-z0-9]+", "-", str(text)).strip("-")[:60]) or "test"


def _short_hash(text, n=16):
    return hashlib.sha1(str(text or "").encode("utf-8")).hexdigest()[:n]


def _norm_hash(text):
    return _short_hash(re.sub(r"\s+", " ", str(text or "")).strip())


def _isnan(v):
    return v is None or (isinstance(v, float) and math.isnan(v)) or v is pd.NA or v is pd.NaT


def fmt_ms(ms):
    if _isnan(ms):
        return "-"
    return f"{ms / 1000:,.2f} s" if abs(ms) >= 1000 else f"{ms:,.0f} ms"


def fmt_hms(seconds):
    seconds = int(max(seconds, 0))
    return f"{seconds // 3600}h {seconds % 3600 // 60:02d}m {seconds % 60:02d}s"


def banner(text):
    print("\n" + "=" * 100 + f"\n{text}\n" + "=" * 100)


def _fmt_short(ms):
    if _isnan(ms):
        return "-"
    if abs(ms) < 1000:
        return f"{ms:,.0f} ms"
    s = ms / 1000
    return f"{s:.2f} s" if abs(s) < 10 else (f"{s:.1f} s" if abs(s) < 100 else f"{s:,.0f} s")


def _pct0(v):
    return "-" if _isnan(v) else f"{v:+.0f}%".replace("-", "−")


def _fmt_rel(rel):
    return "-" if _isnan(rel) else f"{rel:+.1f}%"


def _fmt_ci(ci):
    return "-" if _isnan(ci[0]) else f"[{ci[0]:+.1f}%, {ci[1]:+.1f}%]"


def _join_list(items, limit=4):
    items = [str(i) for i in items]
    if len(items) <= 1:
        return "".join(items)
    if len(items) <= limit:
        return ", ".join(items[:-1]) + " and " + items[-1]
    return ", ".join(items[:limit]) + f" and {len(items) - limit} more"


def _is_true(v):
    return (not _isnan(v)) and bool(v) is True


def _is_false(v):
    return (not _isnan(v)) and bool(v) is False


def _bool_col(s):
    return s.map(lambda v: False if _isnan(v) else bool(v)).astype(bool)


def _html_table(headers, rows, colors=None):
    th = "".join(f"<th style='text-align:left;padding:6px 10px;border-bottom:2px solid #444'>{h}</th>" for h in headers)
    body = []
    for i, r in enumerate(rows):
        tds = []
        for j, v in enumerate(r):
            color = (colors or {}).get((i, j))
            style = f"padding:5px 10px;border-bottom:1px solid #ddd;{'color:' + color + ';font-weight:600' if color else ''}"
            tds.append(f"<td style='{style}'>{v}</td>")
        body.append("<tr>" + "".join(tds) + "</tr>")
    return f"<table style='border-collapse:collapse;{_FONT}'><tr>{th}</tr>{''.join(body)}</table>"


def fmt_ts(v):
    """A timestamp as 'YYYY-MM-DD HH:MM UTC' ('-' when missing)."""
    if _isnan(v) or v is None or str(v).strip() == "":
        return "-"
    t = pd.Timestamp(v)
    t = t.tz_convert("UTC") if t.tzinfo is not None else t
    return f"{t:%Y-%m-%d %H:%M} UTC"


def _plural(n, word, plural=None):
    return f"{n:,} {word if n == 1 else (plural or word + 's')}"


ERROR_CLASSES = [   # first match wins; the text of real engine errors varies, so the patterns are broad
    ("throttling", r"throttl|too many requests|\b429\b|capacity .{0,40}(limit|exceeded|overload)|exceeded .{0,20}capacity"),
    ("memory", r"out of memory|memory (limit|error|pressure)|resource governing|exceeds the (maximum|max) .{0,30}memory"),
    ("timeout", r"time ?out|timed out|timeout|cancel+ed because .{0,40}(time|duration)"),
    ("connection", r"connect|network|socket|transport|authenticat|unauthori[sz]|access token|token .{0,10}expired|"
                   r"\b40[13]\b|database .{0,80}(could not|couldn't) be found"),
]


def error_class(message):
    """timeout / throttling / connection / memory / query (a DAX or engine error of the query itself)."""
    text = str(message or "").lower()
    for name, pattern in ERROR_CLASSES:
        if re.search(pattern, text):
            return name
    return "query"


# ---------- lakehouse (OneLake) ----------
def _resolve_host():
    if str(host_workspace or "").strip():
        ws_id = fabric.resolve_workspace_id(str(host_workspace).strip())
    else:
        ctx = notebookutils.runtime.context
        ws_id = ctx.get("currentWorkspaceId") if hasattr(ctx, "get") else None
        ws_id = ws_id or fabric.get_workspace_id()
    lh_id = fabric.resolve_item_id(lakehouse_name, "Lakehouse", ws_id)
    return ws_id, lh_id


HOST_WS_ID, LAKEHOUSE_ID = _resolve_host()
LAKEHOUSE_ROOT = f"abfss://{HOST_WS_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}"


def _storage_options():
    return {"bearer_token": notebookutils.credentials.getToken("storage"), "use_fabric_endpoint": "true"}


def read_lakehouse_file(rel_path):
    rel = str(rel_path).lstrip("/")
    local = os.path.join(tempfile.gettempdir(), f"mclt_{uuid.uuid4().hex}_{os.path.basename(rel)}")
    try:
        notebookutils.fs.cp(f"{LAKEHOUSE_ROOT}/{rel}", f"file://{local}")
        with open(local, "rb") as fh:
            return fh.read()
    except Exception as exc:
        fallback = os.path.join("/lakehouse/default", rel)
        if os.path.exists(fallback):
            with open(fallback, "rb") as fh:
                return fh.read()
        raise FileNotFoundError(f"Could not read '{rel}' from lakehouse '{lakehouse_name}': {exc}") from exc
    finally:
        if os.path.exists(local):
            os.remove(local)


TS = pa.timestamp("us", tz="UTC")
SCHEMAS = {
    "lt_comparison": pa.schema([
        ("comparison_id", pa.string()), ("test_name", pa.string()), ("status", pa.string()),
        ("updated_utc", TS), ("started_utc", TS), ("finished_utc", TS),
        ("a_label", pa.string()), ("a_workspace", pa.string()), ("a_model", pa.string()),
        ("b_label", pa.string()), ("b_workspace", pa.string()), ("b_model", pa.string()),
        ("query_file", pa.string()), ("query_file_sha256", pa.string()), ("query_count", pa.int64()),
        ("parameters_json", pa.string()), ("message", pa.string()),
    ]),
    "lt_query_catalog": pa.schema([
        ("comparison_id", pa.string()), ("query_seq", pa.int64()), ("query_key", pa.string()),
        ("action_seq", pa.int64()), ("action_label", pa.string()), ("visual_title", pa.string()),
        ("visual_type", pa.string()), ("visual_id", pa.string()), ("captured_ms", pa.float64()),
        ("captured_rows", pa.int64()), ("query_text", pa.string()),
    ]),
    "lt_model_profile": pa.schema([
        ("comparison_id", pa.string()), ("model_key", pa.string()), ("model_label", pa.string()),
        ("item_type", pa.string()), ("item_name", pa.string()), ("value_num", pa.float64()),
        ("value_text", pa.string()), ("captured_utc", TS),
    ]),
    "lt_executions": pa.schema([
        ("comparison_id", pa.string()), ("phase", pa.string()), ("model_key", pa.string()),
        ("model_label", pa.string()), ("concurrency", pa.int64()), ("round_no", pa.int64()),
        ("iteration", pa.int64()), ("user_id", pa.int64()), ("action_seq", pa.int64()),
        ("query_seq", pa.int64()), ("query_key", pa.string()), ("start_utc", TS), ("end_utc", TS),
        ("duration_ms", pa.float64()), ("rows", pa.int64()), ("checksum", pa.string()),
        ("success", pa.bool_()), ("error", pa.string()),
    ]),
    "lt_server_metrics": pa.schema([
        ("comparison_id", pa.string()), ("model_key", pa.string()), ("model_label", pa.string()),
        ("query_seq", pa.int64()), ("query_key", pa.string()), ("server_duration_ms", pa.float64()),
        ("cpu_ms", pa.float64()), ("se_queries", pa.int64()), ("se_cpu_ms", pa.float64()),
        ("se_duration_ms", pa.float64()), ("se_cache_matches", pa.int64()), ("matched_by", pa.string()),
    ]),
    "lt_result_compare": pa.schema([
        ("comparison_id", pa.string()), ("query_seq", pa.int64()), ("query_key", pa.string()),
        ("rows_a", pa.int64()), ("rows_b", pa.int64()), ("checksum_a", pa.string()), ("checksum_b", pa.string()),
        ("same_rows", pa.bool_()), ("same_result", pa.bool_()), ("error_a", pa.string()), ("error_b", pa.string()),
        ("first_run_ms_a", pa.float64()), ("first_run_ms_b", pa.float64()),
    ]),
    # one row of headline numbers per finished comparison (trend in Step 9); new facts go into metrics_json
    "lt_run_summary": pa.schema([
        ("comparison_id", pa.string()), ("test_name", pa.string()), ("started_utc", TS), ("status", pa.string()),
        ("run_mode", pa.string()), ("fingerprint", pa.string()), ("comparable", pa.bool_()),
        ("p50_a", pa.float64()), ("p50_b", pa.float64()), ("p90_a", pa.float64()), ("p90_b", pa.float64()),
        ("cpu_a", pa.float64()), ("cpu_b", pa.float64()), ("metrics_json", pa.string()), ("written_utc", TS),
    ]),
}


def _conform(df, name):
    schema = SCHEMAS[name]
    src = pd.DataFrame(df).reset_index(drop=True)
    out = pd.DataFrame(index=range(len(src)))
    for f in schema:
        col = src[f.name] if f.name in src.columns else pd.Series([None] * len(src), dtype="object")
        if pa.types.is_timestamp(f.type):
            col = pd.to_datetime(col, utc=True, errors="coerce")
        elif pa.types.is_integer(f.type):
            col = pd.to_numeric(col, errors="coerce").round().astype("Int64")
        elif pa.types.is_floating(f.type):
            col = pd.to_numeric(col, errors="coerce").astype("float64")
        elif pa.types.is_boolean(f.type):
            col = col.map(lambda v: None if _isnan(v) else bool(v)).astype("boolean")
        else:
            col = col.map(lambda v: None if _isnan(v) else str(v))
        out[f.name] = col
    return pa.Table.from_pandas(out, schema=schema, preserve_index=False)


def write_table(df, name):
    if df is None or len(df) == 0:
        return 0
    tbl = _conform(df, name)
    write_deltalake(f"{LAKEHOUSE_ROOT}/Tables/{name}", tbl, mode="append", storage_options=_storage_options())
    return tbl.num_rows


def read_table(name, comparison_id=None, columns=None):
    """Read one results table, filtered to one comparison (or a list of them) and to some columns while reading
    (falls back to a full read)."""
    ids = list(comparison_id) if isinstance(comparison_id, (list, set, tuple)) else None
    try:
        dt = DeltaTable(f"{LAKEHOUSE_ROOT}/Tables/{name}", storage_options=_storage_options())
    except Exception as exc:
        if "TableNotFound" not in type(exc).__name__ and "not found" not in str(exc).lower():
            print(f"  (could not read table {name}: {str(exc)[:200]})")
        return pd.DataFrame(columns=columns or [f.name for f in SCHEMAS[name]])
    try:
        import pyarrow.dataset as ds
        flt = (ds.field("comparison_id").isin(ids) if ids is not None else
               (ds.field("comparison_id") == comparison_id) if comparison_id is not None else None)
        return dt.to_pyarrow_dataset().to_table(filter=flt, columns=columns).to_pandas()
    except Exception:
        df = dt.to_pandas()
    if comparison_id is not None and "comparison_id" in df.columns:
        keep = df["comparison_id"].isin(ids) if ids is not None else df["comparison_id"] == comparison_id
        df = df[keep].reset_index(drop=True)
    return df[[c for c in columns if c in df.columns]] if columns else df


def write_lakehouse_file(rel_path, content):
    """Write a text file to the lakehouse (Files/...); returns its path inside the lakehouse."""
    rel = str(rel_path).lstrip("/")
    local = os.path.join(tempfile.gettempdir(), f"mclt_{uuid.uuid4().hex}_{os.path.basename(rel)}")
    try:
        with open(local, "w", encoding="utf-8") as fh:
            fh.write(content)
        notebookutils.fs.cp(f"file://{local}", f"{LAKEHOUSE_ROOT}/{rel}")
    finally:
        if os.path.exists(local):
            os.remove(local)
    return rel


def read_comparisons():
    df = read_table("lt_comparison")
    if df.empty:
        return df
    df = df.sort_values("updated_utc").groupby("comparison_id", as_index=False).tail(1)
    return df.sort_values("started_utc", ascending=False).reset_index(drop=True)


def write_status(status, message=""):
    now = utcnow()
    finished = now if (status.startswith("completed") or status in ("failed", "skipped")) else None
    row = {
        "comparison_id": COMPARISON_ID, "test_name": test_name, "status": status, "updated_utc": now,
        "started_utc": RUN_STARTED_UTC, "finished_utc": finished,
        "a_label": a_label, "a_workspace": a_workspace, "a_model": a_model,
        "b_label": b_label, "b_workspace": b_workspace, "b_model": b_model,
        "query_file": query_file, "query_file_sha256": QUERY_FILE_SHA, "query_count": len(QUERIES),
        "parameters_json": json.dumps(PARAMS_SNAPSHOT, default=str), "message": str(message)[:2000],
    }
    write_table(pd.DataFrame([row]), "lt_comparison")


# ---------- scheduled runs ----------
def detect_run_mode():
    """('interactive' | 'scheduled', how it was decided, Fabric runtime context as plain values).
    run_mode overrides; 'auto' reads notebookutils.runtime.context (pipeline / scheduled job = scheduled)."""
    ctx = {}
    try:
        raw = notebookutils.runtime.context
        items = raw.items() if hasattr(raw, "items") else vars(raw).items()
        ctx = {str(k): (v if isinstance(v, (bool, int, float, str)) or v is None else str(v)) for k, v in items}
    except Exception as exc:
        ctx = {"error": _err_text(exc)[:200]}
    mode = str(run_mode or "auto").strip().lower()
    if mode in ("interactive", "scheduled"):
        return mode, "run_mode parameter", ctx
    hits = [k for k in SCHEDULED_CONTEXT_KEYS if str(ctx.get(k, "")).strip().lower() in ("true", "1")]
    return ("scheduled", "Fabric context: " + ", ".join(hits), ctx) if hits else ("interactive", "Fabric context", ctx)


def approval_fingerprint(query_file_sha):
    """12-character fingerprint of everything that changes the load on the models (FINGERPRINT_PARAMS + query file)."""
    def norm(v):
        if isinstance(v, bool) or v is None:
            return v
        if isinstance(v, (int, float)):
            return float(v)
        return re.sub(r"\s+", " ", str(v)).strip()
    settings = {k: norm(globals().get(k)) for k in FINGERPRINT_PARAMS}
    settings["concurrency_levels"] = _parse_int_list(concurrency_levels)
    settings["query_file_sha256"] = query_file_sha
    return hashlib.sha256(json.dumps(settings, sort_keys=True, default=str).encode("utf-8")).hexdigest()[:12]


def parse_window(text):
    """'22:00-05:00' -> (minutes from midnight start, end); None for ''. Raises ValueError when malformed."""
    if not str(text or "").strip():
        return None
    m = re.fullmatch(r"\s*(\d{1,2}):(\d{2})\s*-\s*(\d{1,2}):(\d{2})\s*", str(text))
    if not m or int(m.group(1)) > 23 or int(m.group(3)) > 23 or int(m.group(2)) > 59 or int(m.group(4)) > 59:
        raise ValueError(f"schedule_window {text!r} must look like '22:00-05:00'")
    return int(m.group(1)) * 60 + int(m.group(2)), int(m.group(3)) * 60 + int(m.group(4))


def in_window(window, tz_name, now=None):
    """(inside?, local time text) for a window from parse_window; a window may cross midnight."""
    from zoneinfo import ZoneInfo
    local = (now or utcnow()).astimezone(ZoneInfo(str(tz_name or "UTC")))
    if window is None:
        return True, f"{local:%H:%M} {tz_name}"
    start, end = window
    t = local.hour * 60 + local.minute
    inside = (start <= t < end) if start <= end else (t >= start or t < end)
    return inside, f"{local:%H:%M} {tz_name}"


# ---------- XMLA / ADOMD.NET ----------
def _cs(value):
    value = str(value)
    if any(ch in value for ch in ";=\"'"):
        return '"' + value.replace('"', '""') + '"'
    return value


def _data_source(workspace, with_id=False):
    name = str(workspace)
    if with_id:  # documented fallback for workspace names that clash: "<name> - <workspace guid>"
        name = f"{name} - {fabric.resolve_workspace_id(workspace)}"
    return XMLA_PREFIX + urllib.parse.quote(name, safe="")


def _err_text(exc):
    msg = getattr(exc, "Message", None) or str(exc)
    return re.sub(r"\s+", " ", str(msg)).strip()[:1000]


def open_connection(model_key):
    """New ADOMD connection with a fresh token. Every virtual user gets its own connection (= own session)."""
    m = MODELS[model_key]
    token = notebookutils.credentials.getToken("pbi")
    last_error = None
    for with_id in (False, True):
        try:
            cs = (f"Data Source={_data_source(m['workspace'], with_id)};Initial Catalog={_cs(m['model'])};"
                  f"Password={token};Application Name={_cs(app_name)};Timeout={int(query_timeout_sec)};")
            con = AdomdConnection(cs)
            con.Open()
            return con
        except Exception as exc:
            last_error = _err_text(exc)
    raise RuntimeError(f"Cannot connect to {m['label']} ({m['workspace']} / {m['model']}): {last_error}")


def _safe_close(con):
    if con is not None:
        try:
            con.Close()
        except Exception:
            pass


def _to_py(v):
    if v is None or isinstance(v, (bool, int, float, str)):
        return v
    if isinstance(v, Decimal):
        return float(v)
    if isinstance(v, datetime):
        return v.isoformat()
    tn = type(v).__name__
    if tn == "DBNull":
        return None
    if tn == "Decimal":
        return float(System.Convert.ToDouble(v))
    if tn == "DateTime":
        return str(v.ToString("o", _INV))
    try:
        return str(v.ToString(_INV))
    except Exception:
        return str(v)


def _canon(v, decimals):
    v = _to_py(v)
    if v is None:
        return "\u2205"
    if isinstance(v, bool):
        return "T" if v else "F"
    if isinstance(v, int):
        return str(v)
    if isinstance(v, float):
        if math.isnan(v):
            return "NaN"
        if math.isinf(v):
            return "inf" if v > 0 else "-inf"
        r = round(v, decimals) + 0.0
        if r.is_integer() and abs(r) < 1e15:
            return str(int(r))
        return f"{r:.{decimals}f}".rstrip("0").rstrip(".")
    return str(v)


def _result_checksum(columns, values):
    """Order-independent checksum of a result set (rows are hashed, sorted, then combined)."""
    decimals = int(checksum_float_decimals)
    row_hashes = sorted(
        hashlib.sha1("\x1f".join(_canon(v, decimals) for v in row).encode("utf-8")).hexdigest() for row in values
    )
    h = hashlib.sha1("\x1e".join(columns).encode("utf-8"))
    for rh in row_hashes:
        h.update(rh.encode("ascii"))
    return h.hexdigest()[:16]


def run_query(con, text, want_checksum=False):
    """Execute one DAX query and read all rows (like a client would). Timing excludes the checksum work."""
    started = utcnow()
    t0 = time.perf_counter()
    rows, error, columns, values = 0, None, [], []
    reader = None
    try:
        cmd = con.CreateCommand()
        cmd.CommandText = text
        cmd.CommandTimeout = int(query_timeout_sec)
        reader = cmd.ExecuteReader()
        n = reader.FieldCount
        if want_checksum:
            columns = [str(reader.GetName(i)) for i in range(n)]
            while reader.Read():
                values.append([reader.GetValue(i) for i in range(n)])
                rows += 1
        else:
            while reader.Read():
                rows += 1
    except Exception as exc:
        error = _err_text(exc)
    finally:
        if reader is not None:
            try:
                reader.Close()
            except Exception:
                pass
    elapsed = (time.perf_counter() - t0) * 1000.0
    checksum = _result_checksum(columns, values) if (want_checksum and error is None) else None
    return {"start_utc": started, "end_utc": started + timedelta(milliseconds=elapsed), "duration_ms": elapsed,
            "rows": rows if error is None else None, "checksum": checksum, "success": error is None, "error": error}


def query_rows(con, dax, timeout=None):
    """Small helper for metadata and check queries: (column names, rows as lists of Python values)."""
    cmd = con.CreateCommand()
    cmd.CommandText = dax
    cmd.CommandTimeout = int(timeout or query_timeout_sec)
    reader = cmd.ExecuteReader()
    try:
        n = reader.FieldCount
        cols = [str(reader.GetName(i)) for i in range(n)]
        data = []
        while reader.Read():
            data.append([_to_py(reader.GetValue(i)) for i in range(n)])
    finally:
        reader.Close()
    return cols, data


def query_df(con, dax, timeout=None):
    """Like query_rows, as a pandas DataFrame (column brackets removed)."""
    cols, data = query_rows(con, dax, timeout)
    return pd.DataFrame(data, columns=[c.strip("[]") for c in cols])


# ---------- DAX text ----------
_DAX_TOKENS = re.compile(r"\[(?:[^\]]|\]\])*\]|'(?:[^']|'')*'|\"(?:[^\"]|\"\")*\"|//[^\n]*|--[^\n]*|/\*.*?\*/", re.S)
_REF = re.compile(r"(?:'((?:[^']|'')+)'|\b([A-Za-z_]\w*))?\[((?:[^\]]|\]\])+)\]")


def dax_scrub(text):
    """DAX without comments and with empty string literals, so names inside strings or comments don't count."""
    def keep(m):
        s = m.group(0)
        if s[0] == '"':
            return '""'
        return " " if s[0] in "/-" else s
    return _DAX_TOKENS.sub(keep, str(text or ""))


def _refs(text):
    """(table or None, name) of every 'Table'[Name], Table[Name] and [Name] in scrubbed DAX."""
    return [((tq.replace("''", "'") if tq else tu) or None, name.replace("]]", "]")) for tq, tu, name in _REF.findall(text)]


def dax_table(table):
    return "'" + str(table).replace("'", "''") + "'"


def dax_name(name):
    return "[" + str(name).replace("]", "]]") + "]"


def dax_col(table, column):
    return dax_table(table) + dax_name(column)


def dax_str(text):
    return '"' + str(text).replace('"', '""') + '"'


def dax_date(d):
    return f"DATE({d.year}, {d.month}, {d.day})"


def dax_literal(v):
    if v is None:
        return "BLANK()"
    if isinstance(v, bool):
        return "TRUE()" if v else "FALSE()"
    if isinstance(v, int):
        return str(v)
    if isinstance(v, float):
        return f"{v:.15f}".rstrip("0").rstrip(".")
    return dax_str(v)


def parse_column_ref(text):
    """"'Table'[Column]" or "Table[Column]" -> (table, column), else None."""
    m = re.fullmatch(r"\s*(?:'((?:[^']|'')+)'|([A-Za-z_][\w ]*?))\s*\[((?:[^\]]|\]\])+)\]\s*", str(text or ""))
    if not m:
        return None
    return (m.group(1).replace("''", "'") if m.group(1) else m.group(2).strip()), m.group(3).replace("]]", "]")


def exec_record(phase, model_key, q, r, concurrency=1, round_no=0, iteration=0, user_id=0):
    return {
        "comparison_id": COMPARISON_ID, "phase": phase, "model_key": model_key,
        "model_label": MODELS[model_key]["label"], "concurrency": concurrency, "round_no": round_no,
        "iteration": iteration, "user_id": user_id,
        "action_seq": q["action_seq"] if q else -1, "query_seq": q["query_seq"] if q else -1,
        "query_key": q["query_key"] if q else "",
        "start_utc": r["start_utc"], "end_utc": r["end_utc"], "duration_ms": r["duration_ms"], "rows": r["rows"],
        "checksum": r["checksum"], "success": r["success"], "error": r["error"],
    }


# ---------- Performance Analyzer file ----------
def _ts(value):
    return datetime.fromisoformat(str(value).replace("Z", "+00:00"))


def parse_performance_analyzer(raw_bytes):
    """Returns (queries, actions). Each DAX query is linked to its visual through the parentId chain
    (Execute DAX Query -> Execute Semantic Query -> Query -> Visual Container Lifecycle) and to the
    user action (click) that preceded it."""
    doc = json.loads(raw_bytes.decode("utf-8-sig"))
    events = doc.get("events", []) if isinstance(doc, dict) else []
    by_id = {e.get("id"): e for e in events if e.get("id")}

    user_actions = sorted((e for e in events if e.get("name") == "User Action" and e.get("start")),
                          key=lambda e: _ts(e["start"]))
    actions = []
    for i, e in enumerate(user_actions, start=1):
        label = str((e.get("metrics") or {}).get("sourceLabel") or "Action")
        actions.append({"action_seq": i, "action_label": re.sub(r"^UserAction_", "", label), "start": _ts(e["start"])})

    def visual_of(event):
        seen, cur = set(), event
        while cur is not None and id(cur) not in seen:
            seen.add(id(cur))
            if cur.get("name") == "Visual Container Lifecycle":
                return cur.get("metrics") or {}
            cur = by_id.get(cur.get("parentId"))
        return {}

    def action_of(start):
        found = (0, "BeforeFirstAction")
        for a in actions:
            if a["start"] <= start:
                found = (a["action_seq"], a["action_label"])
            else:
                break
        return found

    dax = [(i, e) for i, e in enumerate(events)
           if e.get("name") == "Execute DAX Query" and (e.get("metrics") or {}).get("QueryText") and e.get("start")]
    dax.sort(key=lambda t: (_ts(t[1]["start"]), t[0]))
    queries = []
    for seq, (_, e) in enumerate(dax, start=1):
        m = e.get("metrics") or {}
        vis = visual_of(e)
        start = _ts(e["start"])
        a_seq, a_lbl = action_of(start)
        end = _ts(e["end"]) if e.get("end") else start
        captured_ms = (end - start).total_seconds() * 1000.0 if e.get("end") else None
        text = m["QueryText"]
        queries.append({
            "query_seq": seq, "query_key": _short_hash(text, 12), "action_seq": a_seq, "action_label": a_lbl,
            "visual_title": str(vis.get("visualTitle") or ""), "visual_type": str(vis.get("visualType") or ""),
            "visual_id": str(vis.get("visualId") or ""), "captured_ms": captured_ms,
            "captured_rows": m.get("RowCount"), "text": text, "captured_start": start, "captured_end": end,
        })
    return queries, actions


def select_queries(queries, include_spec):
    tokens = _parse_list(include_spec)
    if not tokens:
        return list(queries)
    numbers = {int(t) for t in tokens if t.isdigit()}
    labels = {re.sub(r"^useraction_", "", t.lower()) for t in tokens if not t.isdigit()}
    return [q for q in queries if q["action_seq"] in numbers or q["action_label"].lower() in labels]


def group_by_action(queries):
    groups = []
    for q in queries:
        if groups and groups[-1][0] == q["action_seq"]:
            groups[-1][2].append(q)
        else:
            groups.append((q["action_seq"], q["action_label"], [q]))
    return groups


def max_overlap(intervals):
    """Largest number of (start, end) intervals open at the same moment; at least 1."""
    events = []
    for s, e in intervals:
        e = e if e > s else s + timedelta(microseconds=1)
        events += [(s, 1), (e, -1)]
    best = cur = 0
    for _, step in sorted(events):  # at the same moment an end (-1) sorts before a start (+1)
        cur += step
        best = max(best, cur)
    return max(best, 1)


def build_click_plan(queries):
    """One entry per user action (click): its queries, how many of them Power BI ran at the same time in the capture
    (= parallel slots in the replay) and the captured wall time of the click."""
    plan = []
    for a_seq, label, group in group_by_action(queries):
        starts, ends = [q["captured_start"] for q in group], [q["captured_end"] for q in group]
        plan.append({"action_seq": a_seq, "label": label, "queries": group,
                     "parallel": max_overlap(list(zip(starts, ends))),
                     "captured_wall_ms": (max(ends) - min(starts)).total_seconds() * 1000.0})
    return plan


def query_label(q, width=48):
    title = q.get("visual_title") or q.get("visual_type") or "visual"
    text = f"#{int(q['query_seq'])} a{int(q['action_seq'])} {q.get('action_label', '')} | {title} ({q.get('visual_type', '')})"
    return text if len(text) <= width else text[: width - 1] + "\u2026"


# ---------- model metadata ----------
def collect_model_metadata(con, model_key):
    label, now, rows = MODELS[model_key]["label"], utcnow(), []

    def add(item_type, item_name, value_num=None, value_text=None):
        rows.append({"comparison_id": COMPARISON_ID, "model_key": model_key, "model_label": label,
                     "item_type": item_type, "item_name": item_name, "value_num": value_num,
                     "value_text": value_text, "captured_utc": now})

    counts = query_df(con, 'EVALUATE ROW("Tables", COUNTROWS(INFO.TABLES()), "Measures", COUNTROWS(INFO.MEASURES()), '
                           '"Columns", COUNTROWS(INFO.COLUMNS()), "Relationships", COUNTROWS(INFO.RELATIONSHIPS()), '
                           '"Roles", COUNTROWS(INFO.ROLES()), "CalculationGroups", COUNTROWS(INFO.CALCULATIONGROUPS()))')
    for col in counts.columns:
        add("object_count", col, float(counts.iloc[0][col] or 0))

    tables = query_df(con, 'EVALUATE SELECTCOLUMNS(INFO.TABLES(), "ID", [ID], "TName", [Name], "CG", [CalculationGroupID])')
    tname = dict(zip(tables["ID"], tables["TName"]))
    cg_table = {cg: t for cg, t in zip(tables["CG"], tables["TName"]) if not _isnan(cg)}

    parts = query_df(con, 'EVALUATE SELECTCOLUMNS(INFO.PARTITIONS(), "TableID", [TableID], "PName", [Name], '
                          '"Mode", [Mode], "SourceType", [Type], "Def", [QueryDefinition])')
    for _, p in parts.iterrows():
        t = tname.get(p["TableID"], str(p["TableID"]))
        mode = MODE_NAMES.get(int(p["Mode"]), str(p["Mode"])) if not _isnan(p["Mode"]) else "?"
        src = SOURCE_NAMES.get(int(p["SourceType"]), str(p["SourceType"])) if not _isnan(p["SourceType"]) else "?"
        add("storage_mode", f"{t}/{p['PName']}", value_text=f"{mode} ({src})")
        add("definition_hash", f"partition:{t}/{p['PName']}", value_text=_norm_hash(p["Def"]))

    meas = query_df(con, 'EVALUATE SELECTCOLUMNS(INFO.MEASURES(), "TableID", [TableID], "MName", [Name], "E", [Expression])')
    for _, r in meas.iterrows():
        add("definition_hash", f"measure:{tname.get(r['TableID'], '?')}[{r['MName']}]", value_text=_norm_hash(r["E"]))

    items = query_df(con, 'EVALUATE SELECTCOLUMNS(INFO.CALCULATIONITEMS(), "CGID", [CalculationGroupID], "IName", [Name], "E", [Expression])')
    for _, r in items.iterrows():
        add("definition_hash", f"calc_item:{cg_table.get(r['CGID'], '?')}/{r['IName']}", value_text=_norm_hash(r["E"]))

    ccols = query_df(con, 'EVALUATE SELECTCOLUMNS(FILTER(INFO.COLUMNS(), [Type] = 2), "TableID", [TableID], "CName", [ExplicitName], "E", [Expression])')
    for _, r in ccols.iterrows():
        add("definition_hash", f"calc_column:{tname.get(r['TableID'], '?')}[{r['CName']}]", value_text=_norm_hash(r["E"]))

    exprs = query_df(con, 'EVALUATE SELECTCOLUMNS(INFO.EXPRESSIONS(), "EName", [Name], "E", [Expression])')
    for _, r in exprs.iterrows():
        add("definition_hash", f"expression:{r['EName']}", value_text=_norm_hash(r["E"]))
    add("last_refresh_utc", "at preflight", value_text=last_refresh_utc(con))
    return rows


def last_refresh_utc(con):
    """Latest refresh / Direct Lake framing time over all partitions (a refresh clears the model's caches)."""
    try:
        df = query_df(con, 'EVALUATE ROW("LastRefresh", MAXX(INFO.PARTITIONS(), [RefreshedTime]))')
        return None if df.empty or _isnan(df.iloc[0, 0]) else str(df.iloc[0, 0])[:19]
    except Exception:
        return None


def collect_table_rows(con, model_key):
    label, now, rows = MODELS[model_key]["label"], utcnow(), []
    tables = query_df(con, 'EVALUATE SELECTCOLUMNS(INFO.TABLES(), "TName", [Name], "CG", [CalculationGroupID])')
    for _, t in tables.iterrows():
        if not _isnan(t["CG"]):
            continue
        name = str(t["TName"])
        escaped = name.replace("'", "''")
        try:
            n = query_df(con, f"EVALUATE ROW(\"n\", COUNTROWS('{escaped}'))").iloc[0, 0]
            rows.append({"comparison_id": COMPARISON_ID, "model_key": model_key, "model_label": label,
                         "item_type": "table_rows", "item_name": name, "value_num": float(n or 0), "captured_utc": now})
        except Exception as exc:
            rows.append({"comparison_id": COMPARISON_ID, "model_key": model_key, "model_label": label,
                         "item_type": "table_rows", "item_name": name, "value_text": f"error: {_err_text(exc)[:200]}",
                         "captured_utc": now})
    return rows


def diff_definitions(profile):
    d = profile[profile["item_type"] == "definition_hash"]
    a = dict(zip(d.loc[d["model_key"] == "A", "item_name"], d.loc[d["model_key"] == "A", "value_text"]))
    b = dict(zip(d.loc[d["model_key"] == "B", "item_name"], d.loc[d["model_key"] == "B", "value_text"]))
    out = []
    for name in sorted(set(a) | set(b)):
        if name not in a:
            status = "added in B"
        elif name not in b:
            status = "removed in B"
        elif a[name] != b[name]:
            status = "changed"
        else:
            continue
        kind, _, obj = name.partition(":")
        out.append({"kind": kind, "object": obj, "status": status})
    return pd.DataFrame(out, columns=["kind", "object", "status"])


# ---------- comparison data (shared by the steps and the report) ----------
def parse_params(meta):
    """The parameters a comparison ran with (from lt_comparison.parameters_json)."""
    if meta is None or _isnan(meta["parameters_json"]):
        return {}
    try:
        return json.loads(meta["parameters_json"])
    except Exception:
        return {}


def classify_results(rc):
    """Per query (index query_seq): identical / rows differ / values differ / same rows (values not compared) / error."""
    def one(r):
        if not _isnan(r["error_a"]) or not _isnan(r["error_b"]):
            return "error"
        if _is_true(r["same_result"]):
            return "identical"
        if _is_false(r["same_rows"]):
            return "rows differ"
        if _is_false(r["same_result"]):
            return "values differ"
        return "same rows"
    if rc is None or len(rc) == 0:
        return pd.Series(dtype=object)
    return pd.Series([one(r) for r in rc.to_dict("records")], index=pd.to_numeric(rc["query_seq"]).astype(int).to_numpy())


def result_counts(rc):
    c = classify_results(rc).value_counts()
    n = {k: int(c.get(k, 0)) for k in ("identical", "rows differ", "values differ", "same rows", "error")}
    return {"n": int(c.sum()), "identical": n["identical"], "rows_diff": n["rows differ"],
            "values_diff": n["values differ"], "same_rows": n["same rows"], "errors": n["error"],
            "n_diff": n["rows differ"] + n["values differ"]}


def click_waits(df, replay_mode):
    """How long a user waits after each click, per (model, round, user, click): from the first query start to the last
    query end (page-load replay), or the sum of the query times (sequential replay of comparisons saved before)."""
    keys = ["model_key", "round_no", "user_id", "action_seq"]
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=keys + ["wait_ms", "n"])
    g = df.groupby(keys)
    if replay_mode == "sequential":
        out = g.agg(wait_ms=("duration_ms", "sum"), n=("query_seq", "count"))
    else:
        out = g.agg(first=("start_utc", "min"), last=("end_utc", "max"), n=("query_seq", "count"))
        out["wait_ms"] = (pd.to_datetime(out["last"], utc=True) - pd.to_datetime(out["first"], utc=True)).dt.total_seconds() * 1000.0
        out = out.drop(columns=["first", "last"])
    return out.reset_index()


def pivot_by_model(df, value_col):
    """query_seq x model ('A', 'B') of one value column; empty when a model has no values."""
    if df is None or len(df) == 0 or value_col not in df.columns:
        return pd.DataFrame(columns=["A", "B"])
    p = df.pivot_table(index="query_seq", columns="model_key", values=value_col, aggfunc="first")
    return p if {"A", "B"} <= set(p.columns) else pd.DataFrame(columns=["A", "B"])


def table_row_diffs(prof):
    """Rows per table in both models: index = table, columns A, B and ratio (B / A)."""
    tr = prof[prof["item_type"] == "table_rows"] if len(prof) else prof
    empty = pd.DataFrame(columns=["A", "B", "ratio"])
    if len(tr) == 0:
        return empty
    tv = tr.pivot_table(index="item_name", columns="model_key", values="value_num", aggfunc="max")
    if not {"A", "B"} <= set(tv.columns):
        return empty
    tv = tv[["A", "B"]].copy()
    tv["ratio"] = tv["B"] / tv["A"]
    return tv


def refresh_info(prof):
    """Last refresh / Direct Lake framing per model at preflight and at the end; which models refreshed during the test."""
    lr = prof[prof["item_type"] == "last_refresh_utc"] if len(prof) else prof

    def one(k, when):
        v = lr.loc[(lr["model_key"] == k) & (lr["item_name"] == when), "value_text"] if len(lr) else pd.Series(dtype=object)
        return None if v.empty or _isnan(v.iloc[0]) else str(v.iloc[0])

    info = {k: {"before": one(k, "at preflight"), "after": one(k, "at end")} for k in "AB"}
    refreshed = [k for k in "AB" if info[k]["before"] and info[k]["after"] and info[k]["before"] != info[k]["after"]]
    return {"by_model": info, "refreshed": refreshed, "known": len(lr) > 0 and bool((lr["item_name"] == "at end").any())}


def capacity_info(prof):
    """True = both workspaces on the same capacity, False = different capacities, None = not verified."""
    cap = prof[prof["item_type"] == "capacity_id"] if len(prof) else prof
    if len(cap) == 0 or cap["model_key"].nunique() != 2:
        return None
    return cap["value_text"].astype(str).str.lower().nunique() == 1


# ---------- statistics ----------
def pctl(values, q):
    arr = pd.to_numeric(pd.Series(values), errors="coerce").dropna().to_numpy(dtype=float)
    return float(np.percentile(arr, q)) if arr.size else float("nan")


def rel_delta(a, b):
    if _isnan(a) or _isnan(b) or a == 0:
        return float("nan")
    return (b - a) / a * 100.0


def boot_ci_rel(a_vals, b_vals, q=50, n_boot=1000, seed=7):
    """95% bootstrap CI of the relative change (B vs A, %) of the q-th percentile."""
    a = pd.to_numeric(pd.Series(a_vals), errors="coerce").dropna().to_numpy(dtype=float)
    b = pd.to_numeric(pd.Series(b_vals), errors="coerce").dropna().to_numpy(dtype=float)
    if a.size < 5 or b.size < 5:
        return (float("nan"), float("nan"))
    rng = np.random.default_rng(seed)

    def boot(x):
        out = np.empty(n_boot)
        for s in range(0, n_boot, 100):
            k = min(100, n_boot - s)
            out[s:s + k] = np.percentile(x[rng.integers(0, x.size, size=(k, x.size))], q, axis=1)
        return out

    sa, sb = boot(a), boot(b)
    ok = sa > 0
    if not ok.any():
        return (float("nan"), float("nan"))
    rel = (sb[ok] - sa[ok]) / sa[ok] * 100.0
    lo, hi = np.percentile(rel, [2.5, 97.5])
    return (float(lo), float(hi))


def boot_ci_paired_sum(a_vals, b_vals, n_boot=2000, seed=11):
    """95% bootstrap CI of the relative change of a total, resampling paired items (e.g. queries)."""
    a = pd.to_numeric(pd.Series(a_vals), errors="coerce").to_numpy(dtype=float)
    b = pd.to_numeric(pd.Series(b_vals), errors="coerce").to_numpy(dtype=float)
    ok = ~(np.isnan(a) | np.isnan(b))
    a, b = a[ok], b[ok]
    if a.size < 5:
        return (float("nan"), float("nan"))
    idx = np.random.default_rng(seed).integers(0, a.size, size=(n_boot, a.size))
    sa, sb = a[idx].sum(axis=1), b[idx].sum(axis=1)
    good = sa > 0
    rel = (sb[good] - sa[good]) / sa[good] * 100.0
    if rel.size == 0:
        return (float("nan"), float("nan"))
    lo, hi = np.percentile(rel, [2.5, 97.5])
    return (float(lo), float(hi))


# ---------- number check (Step 5): stored rows -> view ----------
# Step 5 stores its result in lt_model_profile (item_type numcheck_*). The view below is built only from those rows,
# so the dry run shows exactly what a reopened report shows later.
NC_FLAGGED = ("different", "error in A", "error in B")   # statuses that count as a difference between the models


def same_value(a, b, tol_pct):
    """Number-check equality: both blank, the same text, or numbers within tol_pct % of the larger absolute value."""
    if a is None or b is None:
        return a is None and b is None
    if isinstance(a, (str, bool)) or isinstance(b, (str, bool)):
        return str(a) == str(b)
    a, b = float(a), float(b)
    if math.isnan(a) or math.isnan(b):
        return math.isnan(a) and math.isnan(b)
    return abs(a - b) <= max(1e-9, float(tol_pct) / 100.0 * max(abs(a), abs(b)))


def _nc_json(v, default=None):
    try:
        return json.loads(v) if isinstance(v, str) else default
    except Exception:
        return default


def numcheck_from_profile(prof):
    """The number check of one comparison from its numcheck_* rows; None when it has none (saved before Step 5 existed)."""
    if prof is None or len(prof) == 0 or "item_type" not in prof.columns:
        return None
    nc = prof[prof["item_type"].astype(str).str.startswith("numcheck_")]
    if not (nc["item_type"] == "numcheck_config").any():
        return None
    recs = {t: g.to_dict("records") for t, g in nc.groupby("item_type")}
    cfg = {str(r["item_name"]): (None if _isnan(r.get("value_text")) else str(r["value_text"]),
                                 None if _isnan(r.get("value_num")) else float(r["value_num"]))
           for r in recs["numcheck_config"]}
    txt, num = (lambda k: cfg.get(k, (None, None))[0]), (lambda k: cfg.get(k, (None, None))[1])

    def value(r):
        if not _isnan(r.get("value_num")):
            return float(r["value_num"])
        return None if _isnan(r.get("value_text")) else str(r["value_text"])

    def keyed(item_type):
        out = {"A": {}, "B": {}}
        for r in recs.get(item_type, []):
            k = _nc_json(r["item_name"])
            if isinstance(k, list) and r["model_key"] in out:
                out[r["model_key"]][tuple(k)] = value(r)
        return out

    values, by_date, rows = keyed("numcheck_value"), keyed("numcheck_value_by_date"), keyed("numcheck_rows")
    errors = {"A": {}, "B": {}}
    for r in recs.get("numcheck_error", []):
        if r["model_key"] in errors:
            errors[r["model_key"]][str(r["item_name"])] = value(r)
    causes = {}
    for r in recs.get("numcheck_cause", []):
        k = _nc_json(r["item_name"], [])
        if len(k) == 2:
            causes.setdefault(k[0], []).append((k[1], value(r)))
    order = {s: i for i, s in enumerate(NC_FLAGGED + ("error in both", "same"))}
    results = sorted(({"measure": str(r["item_name"]), "status": str(r["value_text"]),
                       "max_rel_pct": None if _isnan(r.get("value_num")) else float(r["value_num"])}
                      for r in recs.get("numcheck_result", [])),
                     key=lambda r: (order.get(r["status"], 9), -(r["max_rel_pct"] or 0), r["measure"]))
    data_dates = [{"model": r["model_key"], "table": str(r["item_name"]), **(_nc_json(r.get("value_text"), {}) or {})}
                  for r in recs.get("numcheck_data_date", [])]
    tol = num("tolerance_pct") or 0.0
    where = {}
    for r in results:
        if r["status"] != "different":
            continue
        m = r["measure"]
        keys = {k for mk in "AB" for k in by_date[mk] if k[0] == m}
        dates = sorted({k[2] for k in keys if not same_value(by_date["A"].get(k), by_date["B"].get(k), tol)})
        bus = sorted({str(k[1]) for mk in "AB" for k in values[mk]
                      if k[0] == m and k[1] is not None and not same_value(values["A"].get(k), values["B"].get(k), tol)})
        where[m] = {"dates": dates, "bus": bus}
    return {
        "status": txt("status") or "unknown", "reason": txt("reason"),
        "date_column": txt("date_column"), "date_source": txt("date_column_source"),
        "bu_column": txt("bu_column"), "bu_source": txt("bu_column_source"),
        "anchor": txt("anchor_date"), "anchor_rule": txt("anchor_rule"),
        "start": txt("window_start"), "end": txt("window_end"),
        "bus": _nc_json(txt("bus"), []), "n_bu_candidates": num("bu_candidates"), "seed": num("seed"), "tolerance": tol,
        "measures": _nc_json(txt("measures_compared"), []), "query_scoped": _nc_json(txt("query_scoped_skipped"), []),
        "only_in": {"A": _nc_json(txt("only_in_A"), []), "B": _nc_json(txt("only_in_B"), [])},
        "dependency_source": txt("dependency_source"), "fact_tables": _nc_json(txt("fact_tables"), []),
        "data_dates": data_dates, "values": values, "by_date": by_date, "rows": rows, "errors": errors,
        "results": results, "causes": causes, "where": where,
        "flagged": [r["measure"] for r in results if r["status"] in NC_FLAGGED],
        "unchecked": [r["measure"] for r in results if r["status"] == "error in both"],
        "n_flagged": int(num("status") or 0), "n_unchecked": int(num("measures_unchecked") or 0),
    }


def _nc_value(v):
    if v is None:
        return "(blank)"
    if isinstance(v, float):
        if v.is_integer() and abs(v) < 1e15:
            return f"{v:,.0f}"
        return f"{v:,.2f}" if abs(v) >= 1000 else (f"{v:,.4f}".rstrip("0").rstrip(".") if abs(v) >= 1 else f"{v:.6g}")
    return str(v)


def _nc_sig(v):
    """A difference with about 4 significant digits (485.9, 0.01183, 12,345)."""
    return f"{v:,.0f}" if abs(v) >= 1000 else f"{v:.4g}"


def _nc_delta(a, b):
    """(difference, relative difference %) of two stored values, or ("", nan) when they aren't both numbers."""
    if isinstance(a, float) and isinstance(b, float):
        return ("+" if b - a >= 0 else "−") + _nc_sig(abs(b - a)), rel_delta(a, b)
    return "", float("nan")


def numcheck_dependents(NC):
    """{measure: the flagged measure it depends on}: measures that differ only because a measure they use differs."""
    flagged, out = set(NC["flagged"]) if NC else set(), {}
    for m in flagged:
        for code, text in NC["causes"].get(m, []):
            if code == "dependency_changed":
                roots = [r for r in re.findall(r"\[((?:[^\]]|\]\])+)\]", str(text)) if r in flagged and r != m]
                if roots and not any(c == "logic_changed" for c, _ in NC["causes"].get(m, [])):
                    out[m] = roots[0]
    return out


def numcheck_line(NC, LBL):
    """One plain sentence with the outcome of the number check."""
    if NC is None:
        return "Number check: not available (this comparison was saved before the number check existed)."
    st, n = NC["status"], len(NC["measures"])
    scope = (f"{len(NC['bus'])} business unit{'s' if len(NC['bus']) != 1 else ''}, {NC['start']} to {NC['end']}"
             if NC["start"] else "")
    if st == "skipped":
        return f"Number check skipped: {NC['reason']}."
    if st == "error":
        return f"Number check failed: {NC['reason']}."
    if st == "differences":
        return (f"CRITICAL: {NC['n_flagged']} of {n} measures used by the report differ between {LBL['A']} and "
                f"{LBL['B']} under a controlled filter ({scope}, a window both models have loaded).")
    extra = f" {NC['n_unchecked']} could not be checked (error in both models)." if NC["n_unchecked"] else ""
    return (f"Number check: all {n - NC['n_unchecked']} measures used by the report match under a controlled filter "
            f"({scope}; tolerance {NC['tolerance']:g}%).{extra}")


def numcheck_headline(NC):
    """Short value for the History table."""
    if NC is None:
        return "-"
    if NC["status"] == "match":
        return f"match ({len(NC['measures']) - NC['n_unchecked']})"
    if NC["status"] == "differences":
        return f"{NC['n_flagged']} differ"
    return NC["status"]


def _nc_where(NC, m):
    w = NC["where"].get(m) or {}
    parts = []
    if w.get("dates"):
        ds = w["dates"]
        parts.append(f"{len(ds)} date{'s' if len(ds) != 1 else ''} ({_join_list(ds, 3)})")
    if w.get("bus"):
        parts.append(f"BU {_join_list(w['bus'], 3)}")
    return "; ".join(parts)


def numcheck_flagged_rows(NC, LBL):
    """One dict per measure that differs (or fails in one model): values, change, where, cause."""
    out = []
    for r in NC["results"] if NC else []:
        if r["status"] not in NC_FLAGGED:
            continue
        m = r["measure"]
        a, b = NC["values"]["A"].get((m, None)), NC["values"]["B"].get((m, None))
        d, rel = _nc_delta(a, b)
        if r["status"] != "different":
            k = "A" if r["status"] == "error in A" else "B"
            a_txt = f"error: {NC['errors']['A'].get(m, '')}"[:160] if k == "A" else _nc_value(a)
            b_txt = f"error: {NC['errors']['B'].get(m, '')}"[:160] if k == "B" else _nc_value(b)
        else:
            a_txt, b_txt = _nc_value(a), _nc_value(b)
        out.append({"measure": m, "status": r["status"], "a": a_txt, "b": b_txt, "delta": d, "rel": rel,
                    "where": _nc_where(NC, m), "causes": [t for _, t in NC["causes"].get(m, [])]})
    return out


def numcheck_html(NC, LBL, compact=False):
    """The number check as HTML: the outcome, the measures that differ and, unless compact, everything behind it."""
    esc = _html.escape
    A, B = LBL["A"], LBL["B"]
    color = {"match": GREEN, "differences": RED}.get(NC["status"] if NC else None, AMBER)
    out = [f"<div style='{_FONT};font-weight:600;color:{color};margin:4px 0'>{esc(numcheck_line(NC, LBL))}</div>"]
    if NC is None:
        return "".join(out)
    flagged = numcheck_flagged_rows(NC, LBL)
    if flagged:
        rows = [[f"<b>{esc(f['measure'])}</b>", esc(f["status"]), esc(f["a"]), esc(f["b"]), esc(f["delta"]),
                 esc(_fmt_rel(f["rel"]) if not _isnan(f["rel"]) else ""), esc(f["where"]),
                 "<br>".join(esc(c) for c in f["causes"])] for f in flagged]
        out.append(_html_table(["Measure", "Status", f"{esc(A)} (selected BUs)", f"{esc(B)} (selected BUs)", "Δ",
                                "Δ %", "Where", "Likely cause (facts)"], rows,
                               {(i, 1): RED for i in range(len(rows))}))
    if compact:
        return "".join(out)
    cfg = []   # only the settings the check got as far as (a skipped check stops early)
    if NC["date_column"]:
        cfg.append(["Date column", esc(f"{NC['date_column']}  ({NC['date_source'] or '-'})")])
    if NC["bu_column"]:
        cfg.append(["Business-unit column", esc(f"{NC['bu_column']}  ({NC['bu_source'] or '-'})")])
    if NC["start"]:
        cfg.append(["Window", esc(f"{NC['start']} to {NC['end']}" + (f"; {NC['anchor_rule']}" if NC["anchor_rule"] else ""))])
    if NC["bus"]:
        cfg.append(["Business units", esc(_join_list(NC["bus"], 10) + (
            f"  (random {len(NC['bus'])} of {NC['n_bu_candidates']:.0f} with data in both models; seed {NC['seed']:.0f})"
            if NC["seed"] and NC["n_bu_candidates"] else ""))])
    if NC["measures"]:
        cfg.append(["Measures compared", esc(f"{len(NC['measures'])} (tolerance {NC['tolerance']:g}%; dependencies from "
                                             f"{NC['dependency_source'] or '-'})")])
    for k in "AB":
        if NC["only_in"][k]:
            cfg.append([f"Only in {esc(LBL[k])} (not compared)", esc(_join_list(NC["only_in"][k], 12))])
    if NC["query_scoped"]:
        cfg.append(["Report-level measures (not compared)", esc(_join_list(NC["query_scoped"], 12))])
    if cfg:
        out.append("<div style='margin-top:8px'>" + _html_table(["Setting", "Value"], cfg) + "</div>")
    if NC["data_dates"]:
        dd = [[esc(LBL.get(d["model"], d["model"])), esc(d["table"]), esc(str(d.get("latest") or "-")),
               esc(str(d.get("max") or "-")), esc(str(d.get("cap") or "-")), esc(str(d.get("mode") or "")),
               esc(str(d.get("use") or ""))] for d in sorted(NC["data_dates"], key=lambda d: (d["model"], d["table"]))]
        out.append(f"<div style='{_FONT};margin-top:8px'><b>Latest date with data per table</b> (the window must lie "
                   "inside what both models have loaded)</div>"
                   + _html_table(["Model", "Table", "Latest date (up to cap)", "Latest date (any)", "Cap (last refresh)",
                                  "Storage", "Use"], dd))
    same = [r["measure"] for r in NC["results"] if r["status"] == "same"]
    if same:
        out.append(f"<div style='{_FONT};margin-top:8px'><b>Same in both models ({len(same)}):</b> "
                   f"{esc(_join_list(same, 40))}</div>")
    if NC["unchecked"]:
        out.append(f"<div style='{_FONT};margin-top:4px;color:{AMBER}'><b>Error in both models ({len(NC['unchecked'])}):</b> "
                   + esc("; ".join(f"{m}: {str(NC['errors']['A'].get(m, ''))[:120]}" for m in NC["unchecked"][:10])) + "</div>")
    drill, deps = [], numcheck_dependents(NC)
    for f in flagged:   # per date (all selected units), then per unit (whole window); dependents only named below
        m = f["measure"]
        if m in deps:
            continue

        def row(where_date, unit, a, b):
            if not same_value(a, b, NC["tolerance"]):
                d, rel = _nc_delta(a, b)
                drill.append([esc(m), esc(where_date), esc(unit), esc(_nc_value(a)), esc(_nc_value(b)), esc(d),
                              esc(_fmt_rel(rel) if not _isnan(rel) else "")])
        for k in sorted({k for mk in "AB" for k in NC["by_date"][mk] if k[0] == m and k[1] is None}, key=lambda k: k[2]):
            row(k[2], "all selected", NC["by_date"]["A"].get(k), NC["by_date"]["B"].get(k))
        for k in sorted({k for mk in "AB" for k in NC["values"][mk] if k[0] == m and k[1] is not None}, key=lambda k: str(k[1])):
            row("whole window", str(k[1]), NC["values"]["A"].get(k), NC["values"]["B"].get(k))
    if drill:
        out.append(f"<div style='{_FONT};margin-top:8px'><b>Where the numbers differ</b> (per date for all selected "
                   "business units, then per business unit for the whole window)</div>"
                   "<div style='max-height:400px;overflow:auto'>"
                   + _html_table(["Measure", "Date", "Business unit", esc(A), esc(B), "Δ", "Δ %"], drill[:300]) + "</div>")
    if deps:
        out.append(f"<div style='{_FONT};margin-top:4px;color:#555'>Same pattern, because they use a measure above: "
                   + esc("; ".join(f"[{m}] uses [{r}]" for m, r in sorted(deps.items()))) + "</div>")
    rk = sorted({k for mk in "AB" for k in NC["rows"][mk]}, key=lambda k: (k[0], k[1] is not None, str(k[1])))
    if rk:
        rr, rc_ = [], {}
        for k in rk:
            a, b = NC["rows"]["A"].get(k), NC["rows"]["B"].get(k)
            if a != b:
                rc_[(len(rr), 3)] = RED
            rr.append([esc(k[0]), esc("all selected" if k[1] is None else str(k[1])), esc(_nc_value(a)), esc(_nc_value(b))])
        out.append(f"<div style='{_FONT};margin-top:8px'><b>Rows in the window</b> (same business units and dates; "
                   "red = the row counts differ)</div>" + _html_table(["Table", "Business unit", esc(A), esc(B)], rr, rc_))
    return "".join(out)


def numcheck_text(NC, LBL):
    """The number check as plain-text lines (Step 10 and copy-as-text)."""
    lines = [numcheck_line(NC, LBL)]
    if NC is None:
        return lines
    if NC["start"]:
        seed = f"; seed {NC['seed']:.0f}" if NC["seed"] else ""
        lines.append(f"Filter: {NC['bu_column']} in {_join_list(NC['bus'], 10)}; {NC['date_column']} from {NC['start']} to "
                     f"{NC['end']}; {NC['anchor_rule']}{seed}")
    for f in numcheck_flagged_rows(NC, LBL):
        rel = f" ({_fmt_rel(f['rel'])})" if not _isnan(f["rel"]) else ""
        lines.append(f"- {f['measure']} [{f['status']}]: {LBL['A']} {f['a']} vs {LBL['B']} {f['b']}{rel}"
                     + (f"; where: {f['where']}" if f["where"] else ""))
        lines += [f"    cause: {c}" for c in f["causes"]]
    for k in "AB":
        if NC["only_in"][k]:
            lines.append(f"Used by the report but only in {LBL[k]} (not compared): {_join_list(NC['only_in'][k], 12)}")
    if NC["unchecked"]:
        lines.append(f"Error in both models (not compared): {_join_list(NC['unchecked'], 12)}")
    return [ln for ln in lines if ln]


print(f"Setup OK. Results lakehouse: {lakehouse_name} ({LAKEHOUSE_ROOT})")

## Step 3 - Preflight
Checks the settings, prints the approval fingerprint, checks the gates of a scheduled run, connects to both models, compares their definitions and estimates how long the test takes.

In [ ]:
# Step 3 - Preflight and model profile
banner("STEP 3 - PREFLIGHT")

# ---- 1. Parameters and run mode ----
CONCURRENCY = _parse_int_list(concurrency_levels)
_problems = []
if not CONCURRENCY or any(c < 1 for c in CONCURRENCY):
    _problems.append("concurrency_levels must be positive integers, e.g. '1,3,5'.")
if int(iterations) < 1:
    _problems.append("iterations must be >= 1.")
_too_high = [c for c in CONCURRENCY if c > int(max_concurrency_guard)]
if _too_high:
    _problems.append(f"concurrency {_too_high} exceeds max_concurrency_guard={max_concurrency_guard} "
                     "(raise the guard only after approval).")
for _name in ("think_time_between_actions_sec", "cooldown_between_rounds_sec", "ramp_up_sec"):
    if float(globals()[_name]) < 0:
        _problems.append(f"{_name} must be >= 0.")
if a_workspace.strip() == b_workspace.strip() and a_model.strip() == b_model.strip():
    _problems.append("Model A and model B are the same model.")
if int(number_check_bu_count) < 1 or int(number_check_days) < 1:
    _problems.append("number_check_bu_count and number_check_days must be >= 1.")
if int(number_check_end_offset_days) < 1:
    _problems.append("number_check_end_offset_days must be >= 1 (the latest loaded date itself may be incomplete).")
if float(number_check_tolerance_pct) < 0:
    _problems.append("number_check_tolerance_pct must be >= 0.")
for _name in ("number_check_bu_column", "number_check_date_column"):
    if str(globals()[_name] or "").strip() and parse_column_ref(globals()[_name]) is None:
        _problems.append(f"{_name} must look like 'Table'[Column] (or \"\" to detect it automatically).")
if int(warmup_passes) < 1:
    _problems.append("warmup_passes must be >= 1.")
if str(run_mode or "auto").strip().lower() not in ("auto", "interactive", "scheduled"):
    _problems.append("run_mode must be 'auto', 'interactive' or 'scheduled'.")
try:
    SCHEDULE_WINDOW = parse_window(schedule_window)
    in_window(SCHEDULE_WINDOW, schedule_timezone)
except Exception as _exc:
    SCHEDULE_WINDOW = None
    _problems.append(f"{_err_text(_exc)} (check schedule_window and schedule_timezone).")
if _problems:
    raise ValueError("Parameter problems:\n - " + "\n - ".join(_problems))

RUN_MODE, RUN_MODE_SOURCE, FABRIC_CONTEXT = detect_run_mode()
SCHEDULED = RUN_MODE == "scheduled"
if SCHEDULED and not run_warmup:
    run_warmup = True   # scheduled runs always warm up: off-hours, models are often evicted (cold)
RUN_ENABLED = bool(confirm_run) and not SCHEDULED   # a scheduled run is enabled by the gates below
RUN_STARTED_UTC = utcnow()
COMPARISON_ID = f"{_slug(test_name)}-{RUN_STARTED_UTC:%Y%m%d-%H%M%S}"
PARAMS_SNAPSHOT = {k: globals().get(k) for k in PARAM_NAMES}
PARAMS_SNAPSHOT.update({"run_mode_setting": run_mode, "run_mode": RUN_MODE, "run_mode_source": RUN_MODE_SOURCE,
                        "fabric_context": FABRIC_CONTEXT})
print(f"comparison_id : {COMPARISON_ID}")
print(f"A ({a_label}) : {a_workspace} / {a_model}")
print(f"B ({b_label}) : {b_workspace} / {b_model}")
print(f"run mode      : {RUN_MODE} ({RUN_MODE_SOURCE})"
      + ("; confirm_run is ignored, the approval gates decide" if SCHEDULED else ""))

# ---- 2. Query file ----
_raw = read_lakehouse_file(query_file)
QUERY_FILE_SHA = hashlib.sha256(_raw).hexdigest()
ALL_QUERIES, ACTIONS = parse_performance_analyzer(_raw)
if not ALL_QUERIES:
    raise ValueError(f"No 'Execute DAX Query' events found in {query_file}.")
QUERIES = select_queries(ALL_QUERIES, include_user_actions)
ACTION_GROUPS = group_by_action(QUERIES)
CLICK_PLAN = build_click_plan(QUERIES)   # replay like a report page: the queries of one click run in parallel as captured
MAX_PARALLEL = max((c["parallel"] for c in CLICK_PLAN), default=1)
APPROVAL_FINGERPRINT = approval_fingerprint(QUERY_FILE_SHA)
PARAMS_SNAPSHOT.update({"replay_mode": "page_load", "approval_fingerprint": APPROVAL_FINGERPRINT,
                        "click_parallelism": {str(c["action_seq"]): c["parallel"] for c in CLICK_PLAN}})

_cat = pd.DataFrame(ALL_QUERIES)
_sel = {q["action_seq"] for q in QUERIES}
_agg = (_cat.groupby(["action_seq", "action_label"])
        .agg(queries=("query_seq", "count"), unique_queries=("query_key", "nunique"),
             visuals=("visual_id", "nunique"), captured_total_s=("captured_ms", lambda s: round(s.sum() / 1000, 1)))
        .reset_index())
if ACTIONS:
    ACTION_SUMMARY = pd.DataFrame(ACTIONS)[["action_seq", "action_label"]].merge(
        _agg, on=["action_seq", "action_label"], how="outer")
else:
    ACTION_SUMMARY = _agg
ACTION_SUMMARY = ACTION_SUMMARY.fillna(
    {"queries": 0, "unique_queries": 0, "visuals": 0, "captured_total_s": 0}).sort_values("action_seq")
ACTION_SUMMARY = ACTION_SUMMARY.astype({"queries": int, "unique_queries": int, "visuals": int})
ACTION_SUMMARY["replayed"] = ACTION_SUMMARY["action_seq"].isin(_sel)
_par = {c["action_seq"]: c["parallel"] for c in CLICK_PLAN}
ACTION_SUMMARY["parallel (as captured)"] = ACTION_SUMMARY["action_seq"].map(_par).astype("Int64")
ACTION_SUMMARY["captured wall s"] = ACTION_SUMMARY["action_seq"].map(
    {c["action_seq"]: round(c["captured_wall_ms"] / 1000, 1) for c in CLICK_PLAN})
print(f"\nQuery file: {query_file}  (sha256 {QUERY_FILE_SHA[:12]}...)")
print(f"{_plural(len(ALL_QUERIES), 'DAX query', 'DAX queries')} ({_cat['query_key'].nunique()} unique) in "
      f"{_plural(len(ACTIONS), 'user action')}; replaying {_plural(len(QUERIES), 'query', 'queries')} in "
      f"{_plural(len(ACTION_GROUPS), 'action group')}.")
display(ACTION_SUMMARY.reset_index(drop=True))
if not QUERIES:
    raise ValueError(f"include_user_actions='{include_user_actions}' selects no queries. See the action list above.")
PEAK_QUERIES = max(CONCURRENCY) * MAX_PARALLEL
if run_load_test and PEAK_QUERIES > int(max_concurrent_queries_guard):
    raise ValueError(f"Peak concurrent queries per model = {max(CONCURRENCY)} virtual users x {MAX_PARALLEL} parallel queries "
                     f"of one click = {PEAK_QUERIES}, above max_concurrent_queries_guard={max_concurrent_queries_guard}. "
                     "Lower concurrency_levels, or raise the guard only after approval.")

# plan and estimated duration (printed in section 5; the scheduled-run lock below needs it now)
_q_s = sum((q["captured_ms"] or 0) for q in QUERIES) / 1000.0
_click_s = sum(c["captured_wall_ms"] for c in CLICK_PLAN) / 1000.0   # a click's queries overlap, as in the capture
_think_one_replay = max(len(CLICK_PLAN) - 1, 0) * float(think_time_between_actions_sec)
ROUNDS_PLANNED = len(CONCURRENCY) * int(iterations) * 2 if run_load_test else 0
_est = {
    f"warm-up ({2 * int(warmup_passes)} single-user passes)": 2 * int(warmup_passes) * _q_s if run_warmup else 0,
    "number check (small single-user queries)": 60 if run_number_check else 0,
    "server-cost trace (2 single-user passes)": 2 * (_q_s + 15) if run_server_trace else 0,
    "load test": ROUNDS_PLANNED * (_click_s + _think_one_replay) + max(ROUNDS_PLANNED - 1, 0) * float(cooldown_between_rounds_sec)
                 + ROUNDS_PLANNED * max(max(CONCURRENCY) - 1, 0) * float(ramp_up_sec),
}
EST_SECONDS = sum(_est.values())


def finish_skipped(reason):
    """A scheduled run that must not load the models: one 'skipped' row (History shows why), then the notebook ends
    as succeeded, so the schedule doesn't alert. Real errors raise instead, so the job fails and alerts work."""
    global RUN_ENABLED
    RUN_ENABLED = False
    write_status("skipped", reason)
    result = {"status": "skipped", "comparison_id": COMPARISON_ID, "test_name": test_name, "run_mode": RUN_MODE,
              "fingerprint": APPROVAL_FINGERPRINT, "reason": reason, "gates": GATES}
    if save_report_files:
        try:
            result["result_path"] = write_lakehouse_file(f"{REPORTS_DIR}/{COMPARISON_ID}.json", json.dumps(result, indent=1))
        except Exception as exc:
            print(f"  (could not save the result file: {_err_text(exc)[:200]})")
    print(f"\nSKIPPED: {reason}\nNothing was loaded. History (Step 9) lists this run as 'skipped' with the reason.")
    notebookutils.notebook.exit(json.dumps(result, default=str))


def _gate(name, ok, detail):
    GATES.append({"gate": name, "ok": bool(ok), "detail": detail})
    print(f"  {'ok  ' if ok else 'STOP'} {name}: {detail}")


# ---- 3. Scheduled run: approval, time window, lock (before anything touches the models) ----
GATES = []
print(f"\napproval fingerprint: {APPROVAL_FINGERPRINT}")
if SCHEDULED:
    print("Scheduled run: checking the gates ...")
    _approved = str(schedule_approval or "").strip().lower()
    _gate("approval", _approved == APPROVAL_FINGERPRINT,
          f"settings and query file match the approved fingerprint {APPROVAL_FINGERPRINT}" if _approved == APPROVAL_FINGERPRINT
          else ("schedule_approval is empty" if not _approved else
                f"settings or query file changed since the approval (approved {_approved}, now {APPROVAL_FINGERPRINT})"))
    if _approved != APPROVAL_FINGERPRINT:   # never heals by itself: fail (alert), unlike window / lock / refresh skips
        _msg = (f"Scheduled run not approved: {GATES[-1]['detail']}. To approve: open the notebook, run it interactively "
                f"with confirm_run = False (dry run), check the approval summary at the end, paste "
                f'schedule_approval = "{APPROVAL_FINGERPRINT}" into Step 1 and save the notebook.')
        PARAMS_SNAPSHOT["gates"] = GATES
        write_status("failed", _msg)
        if save_report_files:
            try:
                write_lakehouse_file(f"{REPORTS_DIR}/{COMPARISON_ID}.json", json.dumps(
                    {"status": "failed", "comparison_id": COMPARISON_ID, "test_name": test_name, "run_mode": RUN_MODE,
                     "fingerprint": APPROVAL_FINGERPRINT, "reason": _msg, "gates": GATES}, indent=1))
            except Exception as _exc:
                print(f"  (could not save the result file: {_err_text(_exc)[:200]})")
        raise RuntimeError(_msg)
    _inside, _local = in_window(SCHEDULE_WINDOW, schedule_timezone)
    _gate("time window", _inside, f"started {_local}" + (f", inside {schedule_window}" if SCHEDULE_WINDOW and _inside else
                                                          f", outside {schedule_window}" if SCHEDULE_WINDOW else
                                                          " (no window set)"))
    _stale_s = max(2 * EST_SECONDS, 1800)   # a 'running' row older than this is a crashed run, not a lock
    _comps = read_comparisons()
    _busy = _comps[(_comps["test_name"] == test_name) & (_comps["status"] == "running")
                   & (pd.to_datetime(_comps["updated_utc"], utc=True) >= utcnow() - timedelta(seconds=_stale_s))
                   ] if len(_comps) else _comps
    _gate("no overlapping run", _busy.empty,
          "no other run of this test is running" if _busy.empty else
          f"{_busy.iloc[0]['comparison_id']} of the same test is still running (last update "
          f"{fmt_ts(_busy.iloc[0]['updated_utc'])}; counted as crashed after {fmt_hms(_stale_s)})")
    PARAMS_SNAPSHOT["gates"] = GATES
    if not all(g["ok"] for g in GATES):
        finish_skipped("; ".join(f"{g['gate']}: {g['detail']}" for g in GATES if not g["ok"]))

# ---- 4. Connect: first-touch time + model metadata ----
print("\nConnecting to both models (the first connection time shows whether a model was cold / evicted) ...")
_profile_rows = []
FIRST_TOUCH_MS = {}
try:
    for _mk in ("A", "B"):
        _t0 = time.perf_counter()
        _con = open_connection(_mk)
        _touch = run_query(_con, "EVALUATE {1}")
        FIRST_TOUCH_MS[_mk] = (time.perf_counter() - _t0) * 1000.0
        if not _touch["success"]:
            _safe_close(_con)
            raise RuntimeError(f"{MODELS[_mk]['label']}: test query failed: {_touch['error']}")
        _profile_rows.append({"comparison_id": COMPARISON_ID, "model_key": _mk, "model_label": MODELS[_mk]["label"],
                              "item_type": "first_touch_ms", "item_name": "connect + EVALUATE {1}",
                              "value_num": FIRST_TOUCH_MS[_mk], "captured_utc": utcnow()})
        _profile_rows += collect_model_metadata(_con, _mk)
        _safe_close(_con)
        _cold = "  <- slow: the model was probably cold (evicted) and had to load" if FIRST_TOUCH_MS[_mk] > COLD_MS else ""
        print(f"  {MODELS[_mk]['label']:<10} first touch: {fmt_ms(FIRST_TOUCH_MS[_mk])}{_cold}")
except Exception as _exc:
    if SCHEDULED:   # e.g. the branch workspace was deleted after a merge: fail (alert), so someone removes the schedule
        _hint = ("If the branch workspace or model was removed, disable this schedule; otherwise check the permissions "
                 "of the schedule owner.")
        write_status("failed", f"preflight: {_err_text(_exc)} {_hint}")
        print(f"\nFAILED: {_err_text(_exc)}\n{_hint}")
    raise
PROFILE_DF = pd.DataFrame(_profile_rows)

# Fair comparison needs both workspaces on the same capacity (different capacities = not comparable, not a gate).
CAPACITY = {}
try:
    _ws = fabric.list_workspaces()
    for _mk in ("A", "B"):
        _hit = _ws[_ws["Name"] == MODELS[_mk]["workspace"]] if "Name" in _ws.columns else _ws.iloc[0:0]
        if len(_hit) and "Capacity Id" in _hit.columns and not _isnan(_hit.iloc[0]["Capacity Id"]):
            CAPACITY[_mk] = str(_hit.iloc[0]["Capacity Id"])
except Exception as _exc:
    print(f"  (capacity check skipped: {_err_text(_exc)[:150]})")
if len(CAPACITY) == 2:
    _same = CAPACITY["A"].lower() == CAPACITY["B"].lower()
    print("  Capacity: both workspaces are on the SAME capacity (fair comparison)." if _same else
          f"  WARNING: the workspaces are on DIFFERENT capacities ({CAPACITY['A']} vs {CAPACITY['B']}); "
          "speed differences may come from the capacity, not the model. The run is marked not comparable.")
    for _mk in ("A", "B"):
        PROFILE_DF = pd.concat([PROFILE_DF, pd.DataFrame([{
            "comparison_id": COMPARISON_ID, "model_key": _mk, "model_label": MODELS[_mk]["label"],
            "item_type": "capacity_id", "item_name": MODELS[_mk]["workspace"], "value_text": CAPACITY[_mk],
            "captured_utc": utcnow()}])], ignore_index=True)
_counts = PROFILE_DF[PROFILE_DF["item_type"] == "object_count"].pivot(index="item_name", columns="model_key", values="value_num")
_counts.columns = [MODELS[c]["label"] for c in _counts.columns]
print("\nObject counts:")
display(_counts.fillna(0).astype(int))
_modes = (PROFILE_DF[PROFILE_DF["item_type"] == "storage_mode"].groupby(["model_key", "value_text"]).size()
          .unstack("model_key", fill_value=0))
_modes.columns = [MODELS[c]["label"] for c in _modes.columns]
print("Partitions by storage mode (source type):")
display(_modes)
DEF_DIFF = diff_definitions(PROFILE_DF)
if DEF_DIFF.empty:
    print("Model definitions: no differences in measures, calculation items, calculated columns, partitions or expressions.")
else:
    print(f"Model definition differences (B vs A): {len(DEF_DIFF)}")
    display(DEF_DIFF)

if SCHEDULED:   # a fresh refresh empties the caches: the run would measure a cold model, not the change
    _lr = PROFILE_DF[PROFILE_DF["item_type"] == "last_refresh_utc"]
    for _mk in ("A", "B"):
        _v = _lr.loc[_lr["model_key"] == _mk, "value_text"]
        _t = pd.to_datetime(_v.iloc[0], utc=True, errors="coerce") if len(_v) and not _isnan(_v.iloc[0]) else pd.NaT
        _mins = (pd.Timestamp(utcnow()) - _t).total_seconds() / 60 if not pd.isna(_t) else float("nan")
        _gate(f"no recent refresh of {MODELS[_mk]['label']}",
              _isnan(_mins) or _mins >= float(schedule_min_minutes_since_refresh),
              "last refresh unknown" if _isnan(_mins) else
              f"last refresh {fmt_ts(_t)}, {_mins:,.0f} min ago (minimum {schedule_min_minutes_since_refresh} min)")
    PARAMS_SNAPSHOT["gates"] = GATES
    if not all(g["ok"] for g in GATES):
        finish_skipped("; ".join(f"{g['gate']}: {g['detail']}" for g in GATES if not g["ok"]))

# ---- 5. Plan and estimated duration ----
print("\nPlan:")
print(f"  concurrency levels {CONCURRENCY} x {iterations} iteration(s) x 2 models = {ROUNDS_PLANNED} rounds, alternating A,B / B,A")
print(f"  each virtual user clicks through {_plural(len(CLICK_PLAN), 'click')}; the queries of a click run in parallel as "
      f"captured ({min(_par.values(), default=1)}-{MAX_PARALLEL} at a time, one connection per parallel query); think time "
      f"{think_time_between_actions_sec}s between clicks")
print(f"  peak concurrent queries per model: {max(CONCURRENCY)} virtual users x {MAX_PARALLEL} = {PEAK_QUERIES} "
      f"(max_concurrent_queries_guard = {max_concurrent_queries_guard})")
if run_load_test and PEAK_QUERIES > 20:
    print("  Note: that many parallel queries need client threads too; for reliable timings add a %%configure cell with "
          '{"vCores": 8} at the top of the notebook.')
print(f"  query executions in the load test: {sum(CONCURRENCY) * int(iterations) * 2 * len(QUERIES) if run_load_test else 0:,}")
print("  estimated duration (captured Desktop timings + think times; queries get slower under load):")
for _k, _v in _est.items():
    print(f"    {_k:<42} ~ {fmt_hms(_v)}")
print(f"    {'TOTAL':<42} ~ {fmt_hms(EST_SECONDS)}")
if EST_SECONDS > 4 * 3600:
    print("  Tip: long run. Consider running this notebook from a pipeline / schedule, or lower the think times.")

# ---- 6. Persist (only when the run is confirmed, or a scheduled run passed its gates) ----
if SCHEDULED:
    RUN_ENABLED = True
if RUN_ENABLED:
    write_status("running", "preflight done")
    write_table(pd.DataFrame([{**{k: v for k, v in q.items() if k != "text"}, "comparison_id": COMPARISON_ID,
                               "query_text": q["text"]} for q in QUERIES]), "lt_query_catalog")
    write_table(PROFILE_DF, "lt_model_profile")
    print(f"\n{'All gates passed' if SCHEDULED else 'confirm_run = True'} -> running Steps 4-7. Results go to "
          f"{lakehouse_name}/Tables (comparison_id {COMPARISON_ID}).")
else:
    print("\nconfirm_run = False -> DRY RUN: nothing is saved and no load is generated (Step 5 runs a few small check "
          "queries). The approval summary is at the end of the notebook (Step 8).")

## Step 4 - Warm-up and result comparison
Runs every query once on each model (single user), checks whether both models return the same results, and counts the rows per table.

In [ ]:
# Step 4 - Warm-up and result comparison
banner("STEP 4 - WARM-UP AND RESULT COMPARISON")
if not RUN_ENABLED:
    print("Skipped: confirm_run = False (preflight only).")
else:
    try:
        WARMUP = {}
        if run_warmup:
            for _mk in ("A", "B"):
                print(f"\n{MODELS[_mk]['label']}: every query once, single user (first-run latency"
                      f"{' + result checksum' if compare_results else ''}) ...")
                _t0 = time.perf_counter()
                _con, _recs = open_connection(_mk), []
                for _q in QUERIES:
                    _r = run_query(_con, _q["text"], want_checksum=bool(compare_results))
                    _recs.append(exec_record("warmup", _mk, _q, _r))
                    if not _r["success"]:
                        _safe_close(_con)
                        _con = open_connection(_mk)
                _safe_close(_con)
                WARMUP[_mk] = pd.DataFrame(_recs)
                write_table(WARMUP[_mk], "lt_executions")
                print(f"  first query {fmt_ms(_recs[0]['duration_ms'])} | total query time "
                      f"{fmt_ms(WARMUP[_mk]['duration_ms'].sum())} | errors {(~WARMUP[_mk]['success']).sum()} | "
                      f"took {fmt_hms(time.perf_counter() - _t0)}")

            _wa, _wb = WARMUP["A"].set_index("query_seq"), WARMUP["B"].set_index("query_seq")
            _rc = []
            for _q in QUERIES:
                _ra, _rb = _wa.loc[_q["query_seq"]], _wb.loc[_q["query_seq"]]
                _both_ok = bool(_ra["success"]) and bool(_rb["success"])
                _rc.append({
                    "comparison_id": COMPARISON_ID, "query_seq": _q["query_seq"], "query_key": _q["query_key"],
                    "rows_a": _ra["rows"], "rows_b": _rb["rows"], "checksum_a": _ra["checksum"], "checksum_b": _rb["checksum"],
                    "same_rows": (_ra["rows"] == _rb["rows"]) if _both_ok else None,
                    "same_result": (_ra["checksum"] == _rb["checksum"]) if (_both_ok and compare_results) else None,
                    "error_a": _ra["error"], "error_b": _rb["error"],
                    "first_run_ms_a": _ra["duration_ms"], "first_run_ms_b": _rb["duration_ms"],
                })
            RESULT_COMPARE = pd.DataFrame(_rc)
            write_table(RESULT_COMPARE, "lt_result_compare")
            _n = result_counts(RESULT_COMPARE)
            _same = _n["identical"] if compare_results else _n["identical"] + _n["same_rows"]
            print(f"\nResult comparison ({len(QUERIES)} queries): identical {_same} | different row count "
                  f"{_n['rows_diff']} | same row count but different values {_n['values_diff']} | "
                  f"errors {_n['errors']}  (informational: B may legitimately return different numbers)")
            _cls = classify_results(RESULT_COMPARE)
            _diff = RESULT_COMPARE[_cls.isin(["rows differ", "values differ", "error"]).to_numpy()]
            if len(_diff):
                _qmap = {q["query_seq"]: q for q in QUERIES}
                display(pd.DataFrame([{
                    "query": query_label(_qmap[r.query_seq], 70), f"rows {a_label}": r.rows_a, f"rows {b_label}": r.rows_b,
                    "same values": r.same_result, f"error {a_label}": r.error_a, f"error {b_label}": r.error_b,
                } for r in _diff.itertuples()]).astype({f"rows {a_label}": "Int64", f"rows {b_label}": "Int64"}))
                print("These whole-query results also cover dates the Branch snapshot may not have loaded yet. "
                      "Step 5 compares the numbers under a controlled filter, where refresh timing can't explain a difference.")
            for _p in range(2, int(warmup_passes) + 1):   # extra passes let slow caches settle; not measured
                _t0 = time.perf_counter()
                for _mk in ("A", "B"):
                    _con = open_connection(_mk)
                    for _q in QUERIES:
                        if not run_query(_con, _q["text"])["success"]:
                            _safe_close(_con)
                            _con = open_connection(_mk)
                    _safe_close(_con)
                print(f"Extra warm-up pass {_p} of {warmup_passes} (both models, not measured): took "
                      f"{fmt_hms(time.perf_counter() - _t0)}")
        else:
            print("Warm-up disabled (run_warmup = False). Note: cold-start effects will then be part of the load statistics.")

        print("\nData volume: rows per table in both models ...")
        _rows = []
        for _mk in ("A", "B"):
            _con = open_connection(_mk)
            _rows += collect_table_rows(_con, _mk)
            _safe_close(_con)
        TABLE_ROWS = pd.DataFrame(_rows)
        write_table(TABLE_ROWS, "lt_model_profile")
        _tv = table_row_diffs(TABLE_ROWS)
        _tv = _tv[(_tv["ratio"] - 1).abs() > ROWS_LIST_PCT / 100].sort_values("A", ascending=False)
        print(f"Tables whose row count differs by more than {ROWS_LIST_PCT}%:")
        display(_tv.assign(ratio=_tv["ratio"].round(2)).astype({"A": "Int64", "B": "Int64"})
                .rename(columns={"A": f"rows {a_label}", "B": f"rows {b_label}", "ratio": "ratio B/A"}))
        write_status("running", "warm-up done")
    except BaseException as _exc:
        write_status("failed", f"warm-up: {_err_text(_exc)}")
        raise

## Step 5 - Number check
Compares the measures the report uses in both models under a controlled filter: a few random business units and a 10-day window that ends a week before the latest date both models have loaded (`number_check_end_offset_days`), so refresh timing and late-arriving data can't explain a difference. A difference is flagged as CRITICAL with its likely cause; the run continues. Also runs in the dry run (shown, not saved).

In [ ]:
# Step 5 - Number check (controlled filter)
banner("STEP 5 - NUMBER CHECK (CONTROLLED FILTER)")
NUMCHECK_BATCH = 20          # measures per check query; a failing query is split until the failing measure is found
NUMCHECK_TIMEOUT_SEC = 120   # per check query (or query_timeout_sec when that is lower)
NUMCHECK_MAX_DRILL = 20      # differing measures that get the per-date breakdown
NUMCHECK_MAX_TABLES = 30     # tables whose rows in the window are counted
BU_MAX_DISTINCT = 1000       # a business-unit column must have 2..1000 values
STALE_FACT_DAYS = 31         # a table this much older than the newest table doesn't set the window
_TIME_FUNCS = re.compile(r"\b(TODAY|NOW|UTCNOW|UTCTODAY)\s*\(", re.I)
_LOCAL_MEASURE = re.compile(r"\bMEASURE\s+(?:'(?:[^']|'')+'|[A-Za-z_]\w*)\s*\[((?:[^\]]|\]\])+)\]\s*=", re.I)


def _nc_timeout():
    return min(int(query_timeout_sec), NUMCHECK_TIMEOUT_SEC)


def _as_datetime(v):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return None
    if isinstance(v, datetime):
        return v.replace(tzinfo=None)
    try:
        return datetime.fromisoformat(str(v)[:19])
    except ValueError:
        return None


def _as_date(v):
    d = _as_datetime(v)
    return d.date() if d else None


def _first_df(con, queries):
    """The first of several equivalent metadata queries that works (older engines lack some INFO columns)."""
    for i, dax in enumerate(queries):
        try:
            return query_df(con, dax, _nc_timeout())
        except Exception:
            if i == len(queries) - 1:
                raise


def model_catalog(con):
    """What the number check needs to know about one model: tables, columns, measures, relationships, partitions and,
    when available, the calculation dependencies of the measures."""
    t = query_df(con, 'EVALUATE SELECTCOLUMNS(INFO.TABLES(), "ID", [ID], "Name", [Name], "Cat", [DataCategory], '
                      '"Hidden", [IsHidden], "CG", [CalculationGroupID])', _nc_timeout())
    c = _first_df(con, [
        'EVALUATE SELECTCOLUMNS(FILTER(INFO.COLUMNS(), [Type] <> 3), "ID", [ID], "TableID", [TableID], '
        '"Name", IF(LEN([ExplicitName]) > 0, [ExplicitName], [InferredName]), '
        '"DT", IF([ExplicitDataType] = 1, [InferredDataType], [ExplicitDataType]), "Key", [IsKey], "Unique", [IsUnique], '
        '"Hidden", [IsHidden], "Type", [Type], "E", [Expression])',
        'EVALUATE SELECTCOLUMNS(FILTER(INFO.COLUMNS(), [Type] <> 3), "ID", [ID], "TableID", [TableID], '
        '"Name", [ExplicitName], "DT", [ExplicitDataType], "Key", [IsKey], "Unique", FALSE(), "Hidden", [IsHidden], '
        '"Type", [Type], "E", [Expression])'])
    m = query_df(con, 'EVALUATE SELECTCOLUMNS(INFO.MEASURES(), "TableID", [TableID], "Name", [Name], "E", [Expression])',
                 _nc_timeout())
    r = query_df(con, 'EVALUATE SELECTCOLUMNS(INFO.RELATIONSHIPS(), "FromCol", [FromColumnID], "ToCol", [ToColumnID], '
                      '"Active", [IsActive])', _nc_timeout())
    p = query_df(con, 'EVALUATE SELECTCOLUMNS(INFO.PARTITIONS(), "TableID", [TableID], "Mode", [Mode], '
                      '"Refreshed", [RefreshedTime])', _nc_timeout())
    tname = dict(zip(t["ID"], t["Name"].astype(str)))
    tables = {str(x["Name"]): {"cat": str(x["Cat"] or ""), "hidden": _is_true(x["Hidden"]), "cg": not _isnan(x["CG"])}
              for x in t.to_dict("records")}
    columns, col_by_id = {}, {}
    for x in c.to_dict("records"):
        tn = tname.get(x["TableID"])
        if tn is None or _isnan(x["Name"]):
            continue
        columns[(tn, str(x["Name"]))] = {"dt": 0 if _isnan(x["DT"]) else int(x["DT"]), "key": _is_true(x["Key"]),
                                          "unique": _is_true(x["Unique"]), "hidden": _is_true(x["Hidden"]),
                                          "type": 0 if _isnan(x["Type"]) else int(x["Type"]),
                                          "expr": x["E"] if isinstance(x["E"], str) else ""}
        col_by_id[x["ID"]] = (tn, str(x["Name"]))
    measures = {str(x["Name"]): {"table": tname.get(x["TableID"], "?"), "expr": x["E"] if isinstance(x["E"], str) else ""}
                for x in m.to_dict("records")}
    rels = [{"from": col_by_id[x["FromCol"]], "to": col_by_id[x["ToCol"]], "active": _is_true(x["Active"])}
            for x in r.to_dict("records") if x["FromCol"] in col_by_id and x["ToCol"] in col_by_id]
    refreshed, modes = {}, {}
    for x in p.to_dict("records"):
        tn = tname.get(x["TableID"])
        if tn is None:
            continue
        modes.setdefault(tn, set()).add(-1 if _isnan(x["Mode"]) else int(x["Mode"]))
        ts = _as_datetime(x["Refreshed"])
        if ts is not None and ts.year >= 2000:
            refreshed[tn] = max(refreshed.get(tn, ts), ts)
    dep, dep_error = None, None
    try:
        d = query_df(con, 'EVALUATE FILTER(INFO.CALCDEPENDENCY(), [OBJECT_TYPE] = "MEASURE")', _nc_timeout())
        d.columns = [str(x).upper() for x in d.columns]
        if {"OBJECT", "REFERENCED_OBJECT_TYPE", "REFERENCED_TABLE", "REFERENCED_OBJECT"} <= set(d.columns):
            dep = {}
            for x in d.to_dict("records"):
                dep.setdefault(str(x["OBJECT"]).lower(), []).append(
                    (str(x["REFERENCED_OBJECT_TYPE"]).upper(), x["REFERENCED_TABLE"], x["REFERENCED_OBJECT"]))
        else:
            dep_error = "unexpected columns " + ", ".join(d.columns)
    except Exception as exc:
        dep_error = _err_text(exc)[:200]
    return {"tables": tables, "tables_lc": {n.lower(): n for n in tables}, "columns": columns,
            "cols_lc": {(tn.lower(), cn.lower()): (tn, cn) for tn, cn in columns}, "measures": measures,
            "measures_lc": {n.lower(): n for n in measures}, "rels": rels, "refreshed": refreshed, "modes": modes,
            "dep": dep, "dep_error": dep_error}


def referenced_measures(queries, cats):
    """Model measures the replayed queries use, by model. Report-level measures (DEFINE MEASURE) are skipped."""
    texts = [dax_scrub(q["text"]) for q in queries]
    local = {n.replace("]]", "]") for t in texts for n in _LOCAL_MEASURE.findall(t)}
    local_lc = {n.lower() for n in local}
    used = {"A": {}, "B": {}}
    for text in texts:
        for table, name in _refs(text):
            low = name.lower()
            if low in local_lc:
                continue
            for mk in "AB":
                cat = cats[mk]
                if table and (table.lower(), low) in cat["cols_lc"]:
                    continue
                if low in cat["measures_lc"]:
                    used[mk][low] = cat["measures_lc"][low]
    both = sorted(set(used["A"]) & set(used["B"]))
    return {"names": {low: {"A": used["A"][low], "B": used["B"][low]} for low in both},
            "compared": [used["A"][low] for low in both],
            "only_A": sorted(used["A"][low] for low in set(used["A"]) - set(used["B"])),
            "only_B": sorted(used["B"][low] for low in set(used["B"]) - set(used["A"])),
            "local": sorted(local)}


def _query_col_refs(queries):
    refs = {}
    for q in queries:
        for table, name in _refs(dax_scrub(q["text"])):
            if table:
                k = (table.lower(), name.lower())
                refs[k] = refs.get(k, 0) + 1
    return refs


def _in_both(cats, ref):
    """(table, column) as spelled in model A when the column exists in both models, else None."""
    if not ref:
        return None
    k = (ref[0].lower(), ref[1].lower())
    return cats["A"]["cols_lc"][k] if k in cats["A"]["cols_lc"] and k in cats["B"]["cols_lc"] else None


def _col_info(cat, ref):
    return cat["columns"][cat["cols_lc"][(ref[0].lower(), ref[1].lower())]]


def detect_date_column(cats, refs, override):
    """The date column of the window: the key of a table marked as date table, else a Date/Calendar table's date
    column, preferring columns with relationships and columns the report uses. Must exist in both models."""
    if str(override or "").strip():
        ref = _in_both(cats, parse_column_ref(override))
        if ref is None:
            return {"skip": f"number_check_date_column {override} is not a column in both models"}
        if any(_col_info(cats[mk], ref)["dt"] != 9 for mk in "AB"):
            return {"skip": f"number_check_date_column {override} is not a date column"}
        return {"ref": ref, "source": "override"}
    total = {}
    for mk in "AB":
        cat = cats[mk]
        to_count = {}
        for r in cat["rels"]:
            if r["active"]:
                to_count[r["to"]] = to_count.get(r["to"], 0) + 1
        date_cols = {}
        for (tn, cn), info in cat["columns"].items():
            if info["dt"] == 9:
                date_cols.setdefault(tn, []).append(cn)
        for tn, cols in date_cols.items():
            is_time = cat["tables"].get(tn, {}).get("cat") == "Time"
            for cn in cols:
                info, score, why = cat["columns"][(tn, cn)], 0, []
                if is_time and info["key"]:
                    score += 100
                    why.append("marked as date table")
                elif is_time and (info["unique"] or len(cols) == 1):
                    score += 60
                    why.append("date table")
                if re.fullmatch(r"(dim[\s_-]*)?(date|dates|calendar|day)s?", tn.strip(), re.I) and (
                        cn.lower() == "date" or info["unique"]):
                    score += 40
                    why.append("table name")
                n_rel = to_count.get((tn, cn), 0)
                if n_rel:
                    score += 20 + 5 * n_rel
                    why.append(f"{n_rel} relationship{'s' if n_rel > 1 else ''}")
                if refs.get((tn.lower(), cn.lower())):
                    score += 10
                    why.append("used by the report")
                if score >= 25:
                    k = (tn.lower(), cn.lower())
                    prev = total.get(k, (0, why, 0))
                    total[k] = (prev[0] + score, prev[1], prev[2] + 1)
    cands = sorted(((s, k, why) for k, (s, why, n) in total.items() if n == 2), key=lambda x: (-x[0], x[1]))
    if not cands:
        return {"skip": "no date column found in both models (no table marked as date table and no Date/Calendar table "
                        "with relationships); set number_check_date_column"}
    others = [dax_col(*cats["A"]["cols_lc"][k]) for _, k, _ in cands[1:4]]
    return {"ref": cats["A"]["cols_lc"][cands[0][1]],
            "source": "auto: " + ", ".join(cands[0][2]) + (f"; other candidates {', '.join(others)}" if others else "")}


_BU_PREFIXES = ("businessunits", "businessunit", "bu")
_BU_REST = {"", "name", "nm", "code", "cd", "id", "key", "desc", "description", "label"}
_BU_TABLES = {"bu", "businessunit", "businessunits", "dimbu", "dimbusinessunit", "dimbusinessunits"}


def _norm(text):
    return re.sub(r"[^a-z0-9]", "", str(text).lower())


def _bu_rest(column, table):
    """What follows 'business unit' / 'BU' in a column name ('' or 'name', 'code', ...), or None if it isn't one."""
    n = _norm(column)
    for p in _BU_PREFIXES:
        if n.startswith(p) and n[len(p):] in _BU_REST:
            return n[len(p):]
    if _norm(table) in _BU_TABLES and n in {"name", "code", "desc", "description", "label"}:
        return n
    return None


def detect_bu_column(cons, cats, refs, override):
    """The business-unit column: a text column named like 'Business Unit' / 'BU' (Name preferred over Code), visible,
    preferably used by the report, with 2..1000 values in both models."""
    if str(override or "").strip():
        ref = _in_both(cats, parse_column_ref(override))
        if ref is None:
            return {"skip": f"number_check_bu_column {override} is not a column in both models"}
        return {"ref": ref, "source": "override"}
    total = {}
    for mk in "AB":
        for (tn, cn), info in cats[mk]["columns"].items():
            rest = _bu_rest(cn, tn) if info["dt"] == 2 else None
            if rest is None:
                continue
            score = (10 + (50 if refs.get((tn.lower(), cn.lower())) else 0) + (0 if info["hidden"] else 20)
                     + (15 if rest in ("", "name") else 0) + (5 if _norm(tn) in _BU_TABLES else 0)
                     - (10 if rest in ("code", "cd", "id", "key") else 0))
            k = (tn.lower(), cn.lower())
            prev = total.get(k, (0, 0))
            total[k] = (prev[0] + score, prev[1] + 1)
    cands = [cats["A"]["cols_lc"][k] for _, k in
             sorted(((s, k) for k, (s, n) in total.items() if n == 2), key=lambda x: (-x[0], x[1]))[:3]]
    if not cands:
        return {"skip": "no business-unit column found in both models (a text column named like 'Business Unit' or "
                        "'BU'); set number_check_bu_column"}
    dax = "EVALUATE ROW(" + ", ".join(f'"n{i}", DISTINCTCOUNT({dax_col(*r)})' for i, r in enumerate(cands)) + ")"
    counts = {mk: query_rows(cons[mk], dax, _nc_timeout())[1][0] for mk in "AB"}
    for i, ref in enumerate(cands):
        if all(isinstance(counts[mk][i], (int, float)) and 2 <= counts[mk][i] <= BU_MAX_DISTINCT for mk in "AB"):
            others = [dax_col(*r) for r in cands if r != ref]
            return {"ref": ref, "source": f"auto: name match, {counts['A'][i]:.0f} values"
                                          + (f"; other candidates {', '.join(others)}" if others else "")}
    return {"skip": f"the business-unit candidates {', '.join(dax_col(*r) for r in cands)} don't have 2 to "
                    f"{BU_MAX_DISTINCT} values; set number_check_bu_column"}


def measure_closure(cat, name):
    """Measures (lower case), tables and columns one measure depends on, directly or through other measures, and
    whether any of them uses TODAY()/NOW()."""
    seen, tables, columns, todo = set(), set(), set(), [name]
    while todo:
        m = cat["measures_lc"].get(str(todo.pop()).lower())
        if m is None or m.lower() in seen:
            continue
        seen.add(m.lower())
        if cat["dep"] is not None:
            for kind, rt, ro in cat["dep"].get(m.lower(), []):
                if kind == "MEASURE":
                    todo.append(ro)
                elif kind in ("COLUMN", "CALC_COLUMN") and not _isnan(rt):
                    tables.add(str(rt))
                    columns.add((str(rt), str(ro)))
                elif kind in ("TABLE", "CALC_TABLE"):
                    tables.add(str(ro if _isnan(rt) else rt))
            continue
        text = dax_scrub(cat["measures"][m]["expr"])     # no dependency info: read the expression
        for table, nm in _refs(text):
            if table and (table.lower(), nm.lower()) in cat["cols_lc"]:
                tn, cn = cat["cols_lc"][(table.lower(), nm.lower())]
                tables.add(tn)
                columns.add((tn, cn))
            elif nm.lower() in cat["measures_lc"]:
                todo.append(nm)
        for tq in re.findall(r"'((?:[^']|'')+)'(?!\[)", text):
            if tq.replace("''", "'").lower() in cat["tables_lc"]:
                tables.add(cat["tables_lc"][tq.replace("''", "'").lower()])
        for tu in re.findall(r"\b([A-Za-z_]\w*)\b(?![\[(\w])", text):
            if tu.lower() in cat["tables_lc"]:
                tables.add(cat["tables_lc"][tu.lower()])
    exprs = [cat["measures"][cat["measures_lc"][x]]["expr"] for x in seen]
    exprs += [cat["columns"][cat["cols_lc"][(t.lower(), c.lower())]]["expr"] for t, c in columns
              if (t.lower(), c.lower()) in cat["cols_lc"]]
    return {"measures": seen, "tables": {t.lower() for t in tables}, "columns": {(t.lower(), c.lower()) for t, c in columns},
            "time_relative": any(_TIME_FUNCS.search(dax_scrub(e)) for e in exprs if e)}


def relevant_fact_tables(cats, closures, date_ref, bu_ref):
    """Tables related to the date table (many side, active relationship) that the compared measures use, in both models."""
    rel, used = {}, {}
    for mk in "AB":
        cat = cats[mk]
        skip = {date_ref[0].lower(), bu_ref[0].lower()} | {t.lower() for t, i in cat["tables"].items() if i["cg"]}
        rel[mk] = {r["from"][0].lower() for r in cat["rels"] if r["active"] and r["to"][0].lower() == date_ref[0].lower()} - skip
        used[mk] = set().union(*[c["tables"] for c in closures[mk].values()]) if closures[mk] else set()
    facts, note = (rel["A"] & used["A"]) & (rel["B"] & used["B"]), ""
    if not facts:
        facts, note = rel["A"] & rel["B"], "the measures' tables aren't related to the date table directly; all related tables are used"
    return sorted(cats["A"]["tables_lc"].get(t, t) for t in facts), note


def latest_data_dates(con, cat, facts, date_ref, run_date):
    """Per fact table: the latest date with rows (any) and up to the cap = min(run date, the table's last refresh)."""
    dc, out, parts = dax_col(*date_ref), [], []
    for i, t in enumerate(facts):
        tn = cat["tables_lc"].get(t.lower(), t)
        modes, ref = cat["modes"].get(tn, set()), cat["refreshed"].get(tn)
        cap = min(run_date, ref.date()) if (ref is not None and modes & {0, 2, 3, 5}) else run_date
        out.append({"table": t, "cap": cap, "refreshed_utc": ref.strftime("%Y-%m-%d %H:%M") if ref else None,
                    "mode": "/".join(MODE_NAMES.get(x, str(x)) for x in sorted(modes))})
        has_rows = f"NOT ISEMPTY(CALCULATETABLE({dax_table(t)}))"
        parts += [f'"x{i}", MAXX(FILTER(VALUES({dc}), {has_rows}), {dc})',
                  f'"c{i}", MAXX(FILTER(VALUES({dc}), {dc} < {dax_date(cap + timedelta(days=1))} && {has_rows}), {dc})']
    _, rows = query_rows(con, "EVALUATE ROW(" + ", ".join(parts) + ")", _nc_timeout())
    for i, d in enumerate(out):
        d["max"], d["latest"] = _as_date(rows[0][2 * i]), _as_date(rows[0][2 * i + 1])
        d["future"] = d["max"] is not None and d["max"] > d["cap"]
    return out


def model_latest(dd):
    """The latest date one model has fully loaded: the earliest latest date over its current fact tables. Tables with
    rows after their last refresh (plan / forecast) and tables much older than the newest one are ignored."""
    for d in dd:
        d["use"] = ("ignored: rows after the last refresh (plan / forecast)" if d["future"]
                    else ("ignored: no rows" if d["latest"] is None else None))
    actual = [d for d in dd if d["latest"] is not None and not d["future"]]
    if actual:
        newest = max(d["latest"] for d in actual)
        for d in actual:
            stale = (newest - d["latest"]).days > STALE_FACT_DAYS
            d["use"] = f"ignored: more than {STALE_FACT_DAYS} days older than the newest table" if stale else "sets the window"
        return min(d["latest"] for d in actual if d["use"] == "sets the window"), "the earliest latest date of its fact tables"
    capped = [d for d in dd if d["latest"] is not None]
    for d in capped:
        d["use"] = "sets the window"
    if capped:
        return min(d["latest"] for d in capped), "all its fact tables have rows after the last refresh; latest date up to it"
    return None, "no fact table has rows"


def numcheck_define(ctx, with_bu=True):
    """The controlled filter: the selected business units and the window (same literal dates in both models)."""
    dc = dax_col(*ctx["date_col"])
    lines = ["DEFINE"]
    if with_bu:
        lines.append(f"    VAR __BU = TREATAS({{{', '.join(dax_literal(b) for b in ctx['bus'])}}}, {dax_col(*ctx['bu_col'])})")
    lines.append(f"    VAR __Dates = FILTER(KEEPFILTERS(VALUES({dc})), AND({dc} >= {dax_date(ctx['start'])}, "
                 f"{dc} < {dax_date(ctx['end'] + timedelta(days=1))}))")
    return "\n".join(lines)


def numcheck_query(ctx, pairs, by_date=False, with_bu=True):
    """Per business unit (plus the total of the selected ones, computed by the measure itself) of (alias, expression)
    pairs under the controlled filter; by_date adds the date as the first column."""
    group = ([dax_col(*ctx["date_col"])] if by_date else []) + [f'ROLLUPADDISSUBTOTAL({dax_col(*ctx["bu_col"])}, "__IsTotal")']
    filters = "__BU, __Dates" if with_bu else "__Dates"
    return (numcheck_define(ctx, with_bu) + "\nEVALUATE\nSUMMARIZECOLUMNS(\n    " + ", ".join(group) + f",\n    {filters},\n    "
            + ",\n    ".join(f'"{alias}", {expr}' for alias, expr in pairs) + ")")


def bu_rows(con, ctx, tables, with_bu=True):
    """Rows of some tables in the window: {business unit (None = total): [count per table]}."""
    pairs = [(f"r{i}", f"COUNTROWS({dax_table(t)})") for i, t in enumerate(tables)]
    _, rows = query_rows(con, numcheck_query(ctx, pairs, with_bu=with_bu), _nc_timeout())
    return {(None if r[1] else r[0]): [0 if v is None else v for v in r[2:]] for r in rows}


def pick_bus(rows_a, rows_b, k, seed):
    """k random business units with rows in the window in both models (reproducible with the seed)."""
    def with_rows(rows):
        return {bu for bu, r in rows.items() if bu is not None and str(bu).strip() and any(v > 0 for v in r)}
    cands = sorted(with_rows(rows_a) & with_rows(rows_b), key=str)
    seed = int(seed or 0) or random.SystemRandom().randint(1, 999_999)
    return random.Random(seed).sample(cands, min(int(k), len(cands))), seed, len(cands)


def eval_measures(con, items, ctx, by_date=False):
    """{key: {"values": {(bu,) or (bu, date): value}, "error": text or None}} for (key, measure name) items. Runs in
    batches; a failing batch is split until each failing measure is isolated (not after a timeout)."""
    out = {}

    def run(batch):
        t0 = time.perf_counter()
        try:
            _, rows = query_rows(con, numcheck_query(ctx, [(f"m{i}", dax_name(n)) for i, (_, n) in enumerate(batch)],
                                                     by_date=by_date), _nc_timeout())
        except Exception as exc:
            err = _err_text(exc)
            timed_out = time.perf_counter() - t0 >= 0.9 * _nc_timeout() or "timeout" in err.lower()
            if len(batch) > 1 and not timed_out:
                run(batch[:len(batch) // 2])
                run(batch[len(batch) // 2:])
            else:
                for key, _ in batch:
                    out[key] = {"values": {}, "error": err[:500]}
            return
        k = 3 if by_date else 2          # columns: [date,] business unit, __IsTotal, m0, m1, ...
        for i, (key, _) in enumerate(batch):
            vals = {}
            for r in rows:
                bu = None if r[k - 1] else r[k - 2]
                v = r[k + i]
                if v is not None:
                    vals[(bu, str(_as_date(r[0]))) if by_date else (bu,)] = v
            out[key] = {"values": vals, "error": None}

    for s in range(0, len(items), NUMCHECK_BATCH):
        run(items[s:s + NUMCHECK_BATCH])
    return out


def compare_measures(res, measures, tol):
    """One result per measure: same / different / error in A / error in B / error in both, and the largest |change %|."""
    out = []
    for m in measures:
        ra, rb = res["A"].get(m, {"values": {}, "error": "not run"}), res["B"].get(m, {"values": {}, "error": "not run"})
        if ra["error"] or rb["error"]:
            status = "error in both" if ra["error"] and rb["error"] else ("error in A" if ra["error"] else "error in B")
            out.append({"measure": m, "status": status, "max_rel_pct": None})
            continue
        va, vb = ra["values"], rb["values"]
        diff = [k for k in set(va) | set(vb) if not same_value(va.get(k), vb.get(k), tol)]
        rels = [abs(rel_delta(float(va[k]), float(vb[k]))) for k in diff
                if all(isinstance(x.get(k), (int, float)) and not isinstance(x.get(k), bool) for x in (va, vb))]
        rels = [x for x in rels if not _isnan(x)]
        out.append({"measure": m, "status": "different" if diff else "same", "max_rel_pct": max(rels) if rels else None})
    return out


def _changed_objects(profile):
    """Objects whose definition differs between the models (from the preflight definition hashes)."""
    ch = {"measure": {}, "calc_column": {}, "partition": {}, "expression": []}
    dd = diff_definitions(profile) if profile is not None and len(profile) else pd.DataFrame(columns=["kind", "object"])
    for r in dd.to_dict("records"):
        obj = str(r["object"])
        if r["kind"] in ("measure", "calc_column") and "[" in obj:
            tn, cn = obj.rsplit("[", 1)
            cn = cn[:-1]
            if r["kind"] == "measure":
                ch["measure"][cn.lower()] = cn
            else:
                ch["calc_column"][(tn.lower(), cn.lower())] = dax_col(tn, cn)
        elif r["kind"] == "partition":
            ch["partition"][obj.rsplit("/", 1)[0].lower()] = obj.rsplit("/", 1)[0]
        elif r["kind"] == "expression":
            ch["expression"].append(obj)
    return ch


def diagnose(m, status, cl, changed, rows_differ, errors):
    """Facts that can explain why one measure differs: changed logic, changed dependencies, different rows in the
    window, run-date functions, an error in one model; otherwise 'unexplained'."""
    A, B, out = a_label, b_label, []
    if m.lower() in changed["measure"]:
        out.append(("logic_changed", f"The definition of [{m}] differs in {B}."))
    deps = sorted({x for mk in "AB" for x in cl[mk]["measures"]} - {m.lower()})
    tbls = {t for mk in "AB" for t in cl[mk]["tables"]}
    cols = {c for mk in "AB" for c in cl[mk]["columns"]}
    dep_changed = ([f"[{changed['measure'][x]}]" for x in deps if x in changed["measure"]]
                   + [f"{changed['calc_column'][c]} (calculated column)" for c in sorted(cols) if c in changed["calc_column"]]
                   + [f"the partitions of '{changed['partition'][t]}'" for t in sorted(tbls) if t in changed["partition"]])
    if dep_changed:
        out.append(("dependency_changed", f"It depends on objects that differ in {B}: {_join_list(dep_changed, 5)}."))
    hits = [txt for t, txt in rows_differ.items() if t.lower() in tbls]
    if hits:
        txt = f"Rows in the window differ: {_join_list(hits, 3)}."
        if changed["expression"]:
            txt += f" {B} also has different data-source settings or parameters: {_join_list(changed['expression'], 3)}."
        out.append(("rows_differ", txt))
    if any(cl[mk]["time_relative"] for mk in "AB"):
        out.append(("time_relative", "It uses TODAY() / NOW(), so its value depends on the run date and the latest data, "
                                     "not only on the window."))
    if status in ("error in A", "error in B"):
        k = status[-1]
        out.append(("error_one_model", f"It fails in {MODELS[k]['label']} only: {str(errors[k].get(m, ''))[:200]}"))
    if not out:
        out.append(("unexplained", f"Same definition, same dependencies and the same rows in the window: investigate (for "
                                   f"example data corrected in place in {A} after {B} was refreshed)."))
    return out


def _nc_row(item_type, model_key, item_name, value_num=None, value_text=None):
    label = MODELS[model_key]["label"] if model_key in MODELS else f"{a_label} vs {b_label}"
    return {"comparison_id": COMPARISON_ID, "model_key": model_key, "model_label": label, "item_type": item_type,
            "item_name": item_name, "value_num": value_num, "value_text": value_text, "captured_utc": utcnow()}


def _nc_key(*parts):
    return json.dumps(list(parts), ensure_ascii=False, default=str)


def _nc_store(v):
    """(value_num, value_text) of one measure value: numbers as numbers, everything else as text."""
    if v is None:
        return None, None
    if isinstance(v, bool) or not isinstance(v, (int, float)):
        return None, str(v)
    return float(v), None


def run_number_check_pass():
    """The whole number check. Returns its lt_model_profile rows (numcheck_*); a failure becomes status 'error'."""
    rows, cons, A, B = [], {}, a_label, b_label

    def cfg(name, text=None, num=None):
        rows.append(_nc_row("numcheck_config", "AB", name, num, text))

    def finish(status, reason=None, n_flagged=0):
        cfg("status", status, n_flagged)
        if reason:
            cfg("reason", reason)
        return rows

    cfg("version", num=1)
    cfg("tolerance_pct", num=float(number_check_tolerance_pct))
    try:
        for mk in "AB":
            cons[mk] = open_connection(mk)
        cats = {mk: model_catalog(cons[mk]) for mk in "AB"}
        dep_err = cats["A"]["dep_error"] or cats["B"]["dep_error"]
        cfg("dependency_source", "INFO.CALCDEPENDENCY" if not dep_err
            else f"measure expressions (INFO.CALCDEPENDENCY not available: {dep_err[:120]})")

        # 1. the measures the report uses
        used = referenced_measures(QUERIES, cats)
        names = used["names"]
        cfg("measures_compared", json.dumps(used["compared"], ensure_ascii=False), len(used["compared"]))
        for mk in "AB":
            if used[f"only_{mk}"]:
                cfg(f"only_in_{mk}", json.dumps(used[f"only_{mk}"], ensure_ascii=False), len(used[f"only_{mk}"]))
        if used["local"]:
            cfg("query_scoped_skipped", json.dumps(used["local"], ensure_ascii=False), len(used["local"]))
        print(f"Measures used by the replayed queries: {len(used['compared'])} in both models"
              + "".join(f"; only in {MODELS[mk]['label']}: {_join_list(used[f'only_{mk}'], 5)}" for mk in "AB" if used[f"only_{mk}"])
              + (f"; {len(used['local'])} report-level measures skipped" if used["local"] else ""))
        if not used["compared"]:
            return finish("skipped", "no model measure used by the replayed queries exists in both models")

        # 2. the business-unit and date columns
        refs = _query_col_refs(QUERIES)
        date = detect_date_column(cats, refs, number_check_date_column)
        if "skip" in date:
            return finish("skipped", date["skip"])
        cfg("date_column", dax_col(*date["ref"]))
        cfg("date_column_source", date["source"])
        bu = detect_bu_column(cons, cats, refs, number_check_bu_column)
        if "skip" in bu:
            return finish("skipped", bu["skip"])
        cfg("bu_column", dax_col(*bu["ref"]))
        cfg("bu_column_source", bu["source"])
        print(f"Date column: {dax_col(*date['ref'])} ({date['source']})\n"
              f"Business-unit column: {dax_col(*bu['ref'])} ({bu['source']})")

        # 3. the window: inside what both models have loaded
        closures = {mk: {low: measure_closure(cats[mk], n[mk]) for low, n in names.items()} for mk in "AB"}
        facts, note = relevant_fact_tables(cats, closures, date["ref"], bu["ref"])
        if not facts:
            return finish("skipped", f"no table is related to the date table {dax_col(*date['ref'])}")
        cfg("fact_tables", json.dumps(facts, ensure_ascii=False), len(facts))
        if note:
            cfg("fact_tables_note", note)
        latest, rule = {}, {}
        for mk in "AB":
            dd = latest_data_dates(cons[mk], cats[mk], facts, date["ref"], utcnow().date())
            latest[mk], rule[mk] = model_latest(dd)
            for d in dd:
                rows.append(_nc_row("numcheck_data_date", mk, d["table"], 1.0 if d["use"] == "sets the window" else 0.0,
                                    json.dumps({k: d[k] for k in ("latest", "max", "cap", "refreshed_utc", "mode", "use")},
                                               default=str)))
                print(f"  {MODELS[mk]['label']:<10} {d['table']}: latest date {d['latest'] or '-'} (any {d['max'] or '-'}; "
                      f"cap {d['cap']}) -> {d['use']}")
        missing = [MODELS[mk]["label"] for mk in "AB" if latest[mk] is None]
        if missing:
            return finish("skipped", f"{_join_list(missing)} has no rows in {_join_list(facts)}")
        anchor = min(latest["A"], latest["B"])
        offset = int(number_check_end_offset_days)
        gap = "one day" if offset == 1 else f"{offset} days"
        end = anchor - timedelta(days=offset)
        start = end - timedelta(days=int(number_check_days) - 1)
        cfg("anchor_date", anchor.isoformat())
        cfg("end_offset_days", num=offset)
        cfg("anchor_rule", f"ends {gap} before {anchor}, the earlier of {A} {latest['A']} and {B} {latest['B']} "
                           f"(number_check_end_offset_days: the newest days may still be loading); "
                           f"per model: {rule['A'] if rule['A'] == rule['B'] else rule['A'] + ' / ' + rule['B']}")
        cfg("window_start", start.isoformat())
        cfg("window_end", end.isoformat(), int(number_check_days))
        print(f"Window: {start} to {end} (ends {gap} before {anchor}, the latest date both models have loaded; "
              f"number_check_end_offset_days = {offset})")
        ctx = {"date_col": date["ref"], "bu_col": bu["ref"], "start": start, "end": end, "bus": []}

        # 4. a few random business units with data in the window
        per_bu = {mk: bu_rows(cons[mk], ctx, facts, with_bu=False) for mk in "AB"}
        for mk in "AB":
            only = {k: v for k, v in per_bu[mk].items() if k is not None}
            if len(only) >= 2 and None in per_bu[mk] and all(v == per_bu[mk][None] for v in only.values()):
                return finish("skipped", f"{dax_col(*bu['ref'])} doesn't filter {_join_list(facts)} in "
                                         f"{MODELS[mk]['label']}; set number_check_bu_column")
        bus, seed, n_cand = pick_bus(per_bu["A"], per_bu["B"], number_check_bu_count, number_check_seed)
        cfg("seed", num=seed)
        cfg("bu_candidates", num=n_cand)
        cfg("bus", json.dumps(bus, ensure_ascii=False, default=str), len(bus))
        if not bus:
            return finish("skipped", f"no business unit has rows from {start} to {end} in both models "
                                     "(lower number_check_end_offset_days or number_check_days)")
        ctx["bus"] = bus
        print(f"Business units: {_join_list(bus, 10)} (random {len(bus)} of {n_cand} with rows in both models; seed {seed})")

        # 5. the numbers
        tol = float(number_check_tolerance_pct)
        print(f"Comparing {len(names)} measures (batches of {NUMCHECK_BATCH}) ...")
        res = {mk: eval_measures(cons[mk], [(n["A"], n[mk]) for n in names.values()], ctx) for mk in "AB"}
        results = compare_measures(res, used["compared"], tol)
        flagged = [r["measure"] for r in results if r["status"] in NC_FLAGGED]
        differ = [r["measure"] for r in sorted(results, key=lambda r: -(r["max_rel_pct"] or 0))
                  if r["status"] == "different"][:NUMCHECK_MAX_DRILL]
        drill = {mk: eval_measures(cons[mk], [(m, names[m.lower()][mk]) for m in differ], ctx, by_date=True)
                 if differ else {} for mk in "AB"}
        tables = sorted({cats["A"]["tables_lc"].get(t, t) for m in flagged for mk in "AB"
                         for t in closures[mk][m.lower()]["tables"]} - set(facts))
        tables = (facts + [t for t in tables if t.lower() in cats["A"]["tables_lc"]])[:NUMCHECK_MAX_TABLES]
        win = {mk: bu_rows(cons[mk], ctx, tables) for mk in "AB"}

        # 6. store
        for mk in "AB":
            for key, r in res[mk].items():
                if r["error"]:
                    rows.append(_nc_row("numcheck_error", mk, key, None, r["error"]))
                for (b_,), v in r["values"].items():
                    rows.append(_nc_row("numcheck_value", mk, _nc_key(key, b_), *_nc_store(v)))
            for key, r in drill[mk].items():
                for (b_, d_), v in r["values"].items():
                    rows.append(_nc_row("numcheck_value_by_date", mk, _nc_key(key, b_, d_), *_nc_store(v)))
            for b_, counts in win[mk].items():
                for t, n in zip(tables, counts):
                    rows.append(_nc_row("numcheck_rows", mk, _nc_key(t, b_), float(n)))
        rows_differ = {}
        for i, t in enumerate(tables):
            for b_ in sorted(set(win["A"]) | set(win["B"]), key=lambda x: (x is not None, str(x))):
                a_n, b_n = (win[mk].get(b_, [0] * len(tables))[i] for mk in "AB")
                if a_n != b_n and t not in rows_differ:
                    rows_differ[t] = (f"'{t}' {a_n:,.0f} vs {b_n:,.0f} rows "
                                      f"({'all selected business units' if b_ is None else 'business unit ' + str(b_)})")
        changed = _changed_objects(globals().get("PROFILE_DF"))
        cfg("measures_unchecked", num=sum(r["status"] == "error in both" for r in results))
        for r in results:
            rows.append(_nc_row("numcheck_result", "AB", r["measure"], r["max_rel_pct"], r["status"]))
            if r["status"] in NC_FLAGGED:
                errs = {mk: {r["measure"]: res[mk][r["measure"]]["error"]} for mk in "AB"}
                cl = {mk: closures[mk][r["measure"].lower()] for mk in "AB"}
                for code, text in diagnose(r["measure"], r["status"], cl, changed, rows_differ, errs):
                    rows.append(_nc_row("numcheck_cause", "AB", _nc_key(r["measure"], code), None, text))
        if flagged:
            return finish("differences", n_flagged=len(flagged))
        if all(r["status"] == "error in both" for r in results):
            return finish("error", "every check query failed in both models: "
                                   + str(res["A"][results[0]["measure"]]["error"])[:300])
        return finish("match")
    except Exception as exc:
        return finish("error", _err_text(exc)[:500])
    finally:
        for con in cons.values():
            _safe_close(con)


NUMCHECK = None
if not run_number_check:
    print("Skipped: run_number_check = False.")
    if RUN_ENABLED:
        write_table(pd.DataFrame([_nc_row("numcheck_config", "AB", "status", value_text="skipped"),
                                  _nc_row("numcheck_config", "AB", "reason", value_text="run_number_check = False")]),
                    "lt_model_profile")
else:
    try:
        NUMCHECK_ROWS = run_number_check_pass()
        NUMCHECK = numcheck_from_profile(pd.DataFrame(NUMCHECK_ROWS))
        display(HTML(f"<div style='{_FONT};max-width:1150px'>" + numcheck_html(NUMCHECK, {"A": a_label, "B": b_label})
                     + "</div>"))
        print("\n" + numcheck_line(NUMCHECK, {"A": a_label, "B": b_label})
              + (" The run continues: the performance comparison is still valid." if NUMCHECK["status"] == "differences" else ""))
        if RUN_ENABLED:
            write_table(pd.DataFrame(NUMCHECK_ROWS), "lt_model_profile")
            write_status("running", f"number check: {NUMCHECK['status']}")
            print(f"Saved {len(NUMCHECK_ROWS)} rows to lt_model_profile.")
        else:
            print("(dry run: shown, not saved)")
    except Exception as _exc:   # the number check never stops the run
        print(f"The number check could not complete: {_err_text(_exc)[:500]}. The run continues.")
    except BaseException:
        if RUN_ENABLED:
            write_status("failed", "number check cancelled")
        raise

## Step 6 - Server cost
Runs every query once more on each model (single user) while a trace measures the server time and CPU of each query.

In [ ]:
# Step 6 - Server cost (traced single-user pass)
banner("STEP 6 - SERVER COST (TRACED SINGLE-USER PASS)")


def _trace_schema():
    wanted = {"queryend", "vertipaqsequeryend", "vertipaqsequerycachematch"}
    try:
        base = fabric.Trace.get_default_query_trace_schema()
        schema = {k: list(v) for k, v in base.items() if re.sub(r"\s+", "", k).lower() in wanted}
    except Exception:
        schema = {}
    if not schema:
        cols = ["EventClass", "EventSubclass", "CurrentTime", "TextData", "SessionID", "StartTime", "EndTime",
                "Duration", "CpuTime", "Success"]
        schema = {"QueryEnd": cols, "VertiPaqSEQueryEnd": cols, "VertiPaqSEQueryCacheMatch": cols[:5]}
    return schema


def _match_server_metrics(logs, session_ids, executed, model_key):
    """Map QueryEnd (+ storage-engine events) of OUR sessions back to the executed queries.
    Other users' events on the same model are ignored (they are never stored)."""
    base = [{"comparison_id": COMPARISON_ID, "model_key": model_key, "model_label": MODELS[model_key]["label"],
             "query_seq": q["query_seq"], "query_key": q["query_key"], "matched_by": None} for q, _ in executed]
    if logs is None or len(logs) == 0:
        return base
    L = logs.copy()
    L.columns = [re.sub(r"\s+", "", str(c)).lower() for c in L.columns]
    if "eventclass" not in L.columns:
        return base
    L["ev"] = L["eventclass"].astype(str).str.replace(r"\s+", "", regex=True).str.lower()
    for c in ("starttime", "endtime", "currenttime"):
        if c in L.columns:
            L[c] = pd.to_datetime(L[c], errors="coerce")
    for c in ("duration", "cputime"):
        L[c] = pd.to_numeric(L[c], errors="coerce") if c in L.columns else np.nan
    sid = (L["sessionid"].astype(str).str.upper() if "sessionid" in L.columns
           else pd.Series([""] * len(L), index=L.index))
    mine, how = L[sid.isin(session_ids)], "session"
    if mine[mine["ev"] == "queryend"].empty and "applicationname" in L.columns:
        apps = L[(L["ev"] == "queryend") & (L["applicationname"].astype(str) == app_name)]
        mine, how = L[sid.isin(set(sid[apps.index]))], "application"
    qe = mine[mine["ev"] == "queryend"]
    order_col = next((c for c in ("starttime", "currenttime") if c in qe.columns), None)
    qe = (qe.sort_values(order_col) if order_col else qe).to_dict("records")
    for r in qe:
        r["norm"] = _norm_hash(r.get("textdata"))
    se = mine[mine["ev"] == "vertipaqsequeryend"]
    cm = mine[mine["ev"] == "vertipaqsequerycachematch"]
    has_window = {"starttime", "endtime"} <= set(mine.columns)
    used = set()
    for pos, (q, _) in enumerate(executed):
        key = _norm_hash(q["text"])
        hit = next((i for i, r in enumerate(qe) if i not in used and r["norm"] == key), None)
        method = how
        if hit is None and len(qe) == len(executed) and pos not in used:
            hit, method = pos, how + "+order"
        if hit is None:
            continue
        used.add(hit)
        r = qe[hit]
        rec = base[pos]
        rec.update({"server_duration_ms": r.get("duration"), "cpu_ms": r.get("cputime"), "matched_by": method})
        if has_window and not pd.isna(r.get("starttime")) and not pd.isna(r.get("endtime")):
            w = se[(se["starttime"] >= r["starttime"]) & (se["endtime"] <= r["endtime"])]
            sub = w["eventsubclass"].astype(str) if "eventsubclass" in w.columns else pd.Series([""] * len(w))
            scans = w[~(sub.str.contains("internal", case=False) | (sub == "10"))]
            t_col = "currenttime" if "currenttime" in cm.columns else "starttime"
            hits = cm[(cm[t_col] >= r["starttime"]) & (cm[t_col] <= r["endtime"])] if t_col in cm.columns else cm.iloc[0:0]
            rec.update({"se_queries": len(scans), "se_cpu_ms": scans["cputime"].sum(),
                        "se_duration_ms": scans["duration"].sum(), "se_cache_matches": len(hits)})
    return base


if not RUN_ENABLED:
    print("Skipped: confirm_run = False (preflight only).")
elif not run_server_trace:
    print("Skipped: run_server_trace = False.")
else:
    try:
        SERVER_ROWS = []
        _schema = _trace_schema()
        # Both traces run at the same time and the queries alternate A,B / B,A, so both models are measured
        # under the same conditions (same moment, same cache distance from the warm-up).
        _st = {k: {"trace": None, "con": None, "recs": [], "executed": [], "sids": set(), "logs": None} for k in ("A", "B")}
        _active = []
        for _mk in ("A", "B"):
            _m, _s = MODELS[_mk], _st[_mk]
            try:
                _tc = fabric.create_trace_connection(_m["model"], _m["workspace"])
                _s["trace"] = _tc.create_trace(_schema, f"{app_name}_{uuid.uuid4().hex[:8]}")
                _s["trace"].start(delay=5)
                _s["con"] = open_connection(_mk)
                _s["sids"].add(str(_s["con"].SessionID).upper())
                _active.append(_mk)
            except Exception as _exc:
                print(f"  Server trace not available for {_m['label']}: {_err_text(_exc)[:300]}\n"
                      "  (tracing may need Contributor or higher on the workspace). Continuing without server metrics.")
        print(f"Traced single-user pass of {len(QUERIES)} queries, interleaved query by query: "
              + " / ".join(MODELS[k]["label"] for k in _active))
        try:
            for _i, _q in enumerate(QUERIES):
                for _mk in (_active if _i % 2 == 0 else _active[::-1]):
                    _s = _st[_mk]
                    _r = run_query(_s["con"], _q["text"])
                    _s["executed"].append((_q, _r))
                    _s["recs"].append(exec_record("trace", _mk, _q, _r))
                    if not _r["success"]:
                        _safe_close(_s["con"])
                        _s["con"] = open_connection(_mk)
                        _s["sids"].add(str(_s["con"].SessionID).upper())
            time.sleep(5)  # trace events arrive asynchronously
            for _mk in _active:
                try:
                    _st[_mk]["logs"] = _st[_mk]["trace"].stop(timeout=10)
                except Exception as _exc:
                    print(f"  Could not read the trace for {MODELS[_mk]['label']}: {_err_text(_exc)[:300]}")
        finally:
            for _mk in ("A", "B"):
                _safe_close(_st[_mk]["con"])
                if _st[_mk]["trace"] is not None:
                    try:
                        _st[_mk]["trace"].drop()  # drops only this notebook's trace
                    except Exception:
                        pass
        for _mk in ("A", "B"):
            _s = _st[_mk]
            write_table(pd.DataFrame(_s["recs"]), "lt_executions")
            if _s["executed"]:
                _rows = _match_server_metrics(_s["logs"], _s["sids"], _s["executed"], _mk)
                SERVER_ROWS += _rows
                _sm = pd.DataFrame(_rows)
                _matched = int(_sm["matched_by"].notna().sum())
                _cpu = pd.to_numeric(_sm.get("cpu_ms"), errors="coerce").sum() if "cpu_ms" in _sm else float("nan")
                _dur = pd.to_numeric(_sm.get("server_duration_ms"), errors="coerce").sum() if "server_duration_ms" in _sm else float("nan")
                print(f"  {MODELS[_mk]['label']:<10} matched server events for {_matched}/{len(_s['executed'])} queries | "
                      f"server duration {fmt_ms(_dur)} | server CPU {fmt_ms(_cpu)}")
        write_table(pd.DataFrame(SERVER_ROWS), "lt_server_metrics")
        write_status("running", "server-cost pass done")
    except BaseException as _exc:
        write_status("failed", f"server trace: {_err_text(_exc)}")
        raise

## Step 7 - Load test
Virtual users click through the report at the same time, in alternating rounds on each model: the queries of one click run in parallel as captured, with a think time between clicks. The longest step; progress is printed while it runs.

In [ ]:
# Step 7 - Load test
banner("STEP 7 - LOAD TEST")
STOP_EVENT = threading.Event()


def _run_click(model_key, queries, slots, record):
    """One click, like a report page: its queries run up to len(slots) at a time, each slot on its own connection.
    Returns the connection errors that stop the virtual user (query errors are recorded and don't stop it)."""
    queue, qlock, broken = list(queries), threading.Lock(), []

    def worker(slot):
        while not STOP_EVENT.is_set() and not broken:
            with qlock:
                if not queue:
                    return
                q = queue.pop(0)
            if slot["con"] is None:   # only after a failed query in this click; otherwise connected between clicks
                try:
                    slot["con"], slot["opened"] = open_connection(model_key), time.time()
                except Exception as exc:
                    broken.append(exc)
                    return
            r = run_query(slot["con"], q["text"])
            record(q, r)
            if not r["success"]:
                _safe_close(slot["con"])
                slot["con"] = None

    n = min(len(slots), len(queries))
    if n <= 1:
        worker(slots[0])
    else:
        workers = [threading.Thread(target=worker, args=(slots[i],), daemon=True) for i in range(n)]
        for t in workers:
            t.start()
        for t in workers:
            t.join()
    return broken


def _virtual_user(model_key, user_id, concurrency, round_no, iteration, start_at, sink, progress, lock):
    """One virtual user clicks through the report: the queries of one click run in parallel as captured (one connection
    per parallel slot), with think time only between clicks."""
    think_a = float(think_time_between_actions_sec)
    if STOP_EVENT.wait(max(start_at - time.time(), 0)):
        return
    slots = [{"con": None, "opened": 0.0} for _ in range(MAX_PARALLEL)]

    def record(q, r):
        with lock:
            sink.append(exec_record("load", model_key, q, r, concurrency, round_no, iteration, user_id))
            progress["done"] += 1
            progress["errors"] += 0 if r["success"] else 1

    try:
        for ci, click in enumerate(CLICK_PLAN):
            if STOP_EVENT.is_set():
                return
            use = slots[:click["parallel"]]
            for s in use:   # (re)connect between clicks, so connecting never counts as waiting time
                if s["con"] is None or time.time() - s["opened"] > RECONNECT_AFTER_SEC:
                    _safe_close(s["con"])
                    s["con"], s["opened"] = open_connection(model_key), time.time()
            broken = _run_click(model_key, click["queries"], use, record)
            if broken:
                raise broken[0]
            if ci < len(CLICK_PLAN) - 1 and think_a > 0 and STOP_EVENT.wait(think_a):
                return
    except Exception as exc:
        now = utcnow()
        failure = {"start_utc": now, "end_utc": now, "duration_ms": None, "rows": None, "checksum": None,
                   "success": False, "error": f"virtual user stopped: {_err_text(exc)}"}
        with lock:
            sink.append(exec_record("load", model_key, None, failure, concurrency, round_no, iteration, user_id))
            progress["errors"] += 1
    finally:
        for s in slots:
            _safe_close(s["con"])


def _record_refresh_at_end():
    """Store each model's last refresh time again; the report flags models refreshed during the test."""
    rows = []
    for mk in ("A", "B"):
        con = None
        try:
            con = open_connection(mk)
            rows.append({"comparison_id": COMPARISON_ID, "model_key": mk, "model_label": MODELS[mk]["label"],
                         "item_type": "last_refresh_utc", "item_name": "at end", "value_text": last_refresh_utc(con),
                         "captured_utc": utcnow()})
        except Exception as exc:
            print(f"  (could not read the last refresh time of {MODELS[mk]['label']}: {_err_text(exc)[:150]})")
        finally:
            _safe_close(con)
    write_table(pd.DataFrame(rows), "lt_model_profile")


if not RUN_ENABLED:
    print("Skipped: confirm_run = False (preflight only).")
elif not run_load_test:
    print("Skipped: run_load_test = False.")
    _record_refresh_at_end()
    write_status("completed", "no load test (run_load_test = False)")
else:
    ROUND_PLAN, _k = [], 0
    for _c in CONCURRENCY:
        for _it in range(1, int(iterations) + 1):
            _k += 1
            for _mk in (("A", "B") if _k % 2 == 1 else ("B", "A")):   # A,B / B,A / A,B ... cancels drift
                ROUND_PLAN.append({"round_no": len(ROUND_PLAN) + 1, "concurrency": _c, "iteration": _it, "model_key": _mk})
    print(f"{len(ROUND_PLAN)} rounds (A = {a_label}, B = {b_label}; number = virtual users): "
          + " ".join(f"{r['model_key']}{r['concurrency']}" for r in ROUND_PLAN))
    _t_test, _total_err, _sink, _threads, _sink_saved = time.time(), 0, [], [], True
    try:
        for _rp in ROUND_PLAN:
            _mk, _c = _rp["model_key"], _rp["concurrency"]
            _sink, _progress, _lock, _sink_saved = [], {"done": 0, "errors": 0}, threading.Lock(), False
            _expected = _c * len(QUERIES)
            print(f"\nRound {_rp['round_no']}/{len(ROUND_PLAN)} | {MODELS[_mk]['label']} | {_c} virtual user(s) | "
                  f"iteration {_rp['iteration']}/{iterations} | started {utcnow():%H:%M:%S} UTC")
            _start_at = time.time() + 2
            _threads = [threading.Thread(target=_virtual_user, daemon=True,
                                         args=(_mk, _u, _c, _rp["round_no"], _rp["iteration"],
                                               _start_at + (_u - 1) * float(ramp_up_sec), _sink, _progress, _lock))
                        for _u in range(1, _c + 1)]
            _t0 = _last = time.time()
            for _t in _threads:
                _t.start()
            while any(_t.is_alive() for _t in _threads):
                time.sleep(2)
                if time.time() - _last >= PROGRESS_EVERY_SEC:
                    _last = time.time()
                    print(f"    ... {_progress['done']}/{_expected} queries, errors {_progress['errors']}, "
                          f"elapsed {fmt_hms(_last - _t0)}")
            _df = pd.DataFrame(_sink)
            write_table(_df, "lt_executions")
            _sink_saved = True
            _ok = _df[_df["success"] == True] if len(_df) else _df  # noqa: E712
            _err = len(_df) - len(_ok)
            _total_err += _err
            _cw = click_waits(_ok[_ok["query_seq"] > 0], "page_load") if len(_ok) else None
            print(f"  done: {len(_ok)}/{_expected} ok, errors {_err} | query median {fmt_ms(pctl(_ok.get('duration_ms', []), 50))} | "
                  f"p90 {fmt_ms(pctl(_ok.get('duration_ms', []), 90))} | median wait per click "
                  f"{fmt_ms(pctl(_cw['wait_ms'], 50) if _cw is not None else float('nan'))} | took {fmt_hms(time.time() - _t0)}")
            if _rp["round_no"] < len(ROUND_PLAN) and float(cooldown_between_rounds_sec) > 0:
                time.sleep(float(cooldown_between_rounds_sec))
        _record_refresh_at_end()
        write_status("completed", f"load test done in {fmt_hms(time.time() - _t_test)}; errors {_total_err}")
        print(f"\nLoad test completed in {fmt_hms(time.time() - _t_test)} (errors: {_total_err}).")
    except BaseException as _exc:   # includes cancelling the cell
        STOP_EVENT.set()            # stops the background virtual users
        for _t in _threads:
            _t.join(timeout=10)
        if _sink and not _sink_saved and not any(_t.is_alive() for _t in _threads):
            write_table(pd.DataFrame(_sink), "lt_executions")   # keep the partial round
        write_status("failed", f"load test interrupted: {_err_text(_exc) or type(_exc).__name__}")
        print(f"Load test stopped. The rounds so far are saved: run Steps 8, 9 and 10 to see the partial report "
              f"(comparison_id {COMPARISON_ID}).")
        raise

## Step 8 - Report
The executive summary comes first (blockers, trust checks, bottom line, pros, cons, trade-offs). The details for analysts are in one collapsed section at the end. After a dry run: the approval summary.

In [ ]:
# Step 8 - Report (dashboard)
banner("STEP 8 - REPORT")
import base64  # noqa: E402
import io  # noqa: E402
import matplotlib.ticker  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402

NAN2 = (float("nan"), float("nan"))
NOTICEABLE_MS = 100     # latency changes smaller than this are not noticeable to report users
CPU_CHANGE_MIN_MS = 50  # per-query server CPU changes smaller than this are not named as the largest increase / decrease
SAMPLES_MIN_LEVEL = 30  # fewer successful samples per model and level = percentiles are indicative
CLASS_COLOR = {"better": GREEN, "worse": RED, "same": GREY, "unclear": AMBER, "few": GREY, None: GREY}
REPORT_HTML = []        # everything the report shows, in order; saved as the report file
REPORT_FILE = None


def emit(html_):
    """Show one part of the report and keep it for the saved report file."""
    REPORT_HTML.append(html_)
    display(HTML(html_))


def _unit(values_ms):
    """Chart unit: milliseconds when everything is sub-second, otherwise seconds."""
    mx = pd.to_numeric(pd.Series(values_ms), errors="coerce").max()
    return ("ms", 1.0) if (not _isnan(mx) and mx < 1000) else ("s", 1000.0)


def _pick_comparison_id():
    """report_comparison_id if set, else the test this notebook run just made; None after a dry run (no fallback)."""
    cid = str(report_comparison_id or "").strip()
    if cid:
        return cid
    return COMPARISON_ID if globals().get("RUN_ENABLED") else None


def load_comparison(cid):
    comps = read_comparisons()
    meta = comps[comps["comparison_id"] == cid] if len(comps) else comps
    data = {n: read_table(n, cid) for n in ("lt_query_catalog", "lt_model_profile", "lt_executions",
                                             "lt_server_metrics", "lt_result_compare")}
    ex = data["lt_executions"].copy()
    if len(ex):
        ex["duration_ms"] = pd.to_numeric(ex["duration_ms"], errors="coerce")
        ex["success"] = _bool_col(ex["success"])
        for c in ("query_seq", "round_no", "user_id", "concurrency", "action_seq", "iteration"):
            ex[c] = pd.to_numeric(ex[c], errors="coerce").fillna(-1).astype(int)
        for c in ("start_utc", "end_utc"):
            ex[c] = pd.to_datetime(ex[c], utc=True, errors="coerce")
    data["lt_executions"] = ex
    sm = data["lt_server_metrics"].copy()
    for c in ("query_seq", "cpu_ms", "server_duration_ms", "se_queries", "se_cpu_ms", "se_cache_matches"):
        if c in sm.columns:
            sm[c] = pd.to_numeric(sm[c], errors="coerce")
    data["lt_server_metrics"] = sm
    for n in ("lt_query_catalog", "lt_result_compare"):
        if len(data[n]):
            data[n]["query_seq"] = pd.to_numeric(data[n]["query_seq"], errors="coerce").fillna(-1).astype(int)
    return (meta.iloc[0] if len(meta) else None), data


def paired_totals(df, value_col):
    """Totals over the queries that have a value for BOTH models, plus a paired bootstrap CI of the change."""
    nan = float("nan")
    p = pivot_by_model(df, value_col).dropna(subset=["A", "B"])
    if p.empty:
        return nan, nan, (nan, nan), 0
    return float(p["A"].sum()), float(p["B"].sum()), boot_ci_paired_sum(p["A"], p["B"]), len(p)


def headline(ex, sm):
    load = ex[(ex["phase"] == "load") & ex["success"] & (ex["query_seq"] > 0)] if len(ex) else ex
    a = load.loc[load["model_key"] == "A", "duration_ms"] if len(load) else pd.Series(dtype=float)
    b = load.loc[load["model_key"] == "B", "duration_ms"] if len(load) else pd.Series(dtype=float)
    cpu_a, cpu_b, _, n_cpu = paired_totals(sm, "cpu_ms")
    return {"n_a": len(a), "n_b": len(b), "median_a": pctl(a, 50), "median_b": pctl(b, 50),
            "p90_a": pctl(a, 90), "p90_b": pctl(b, 90), "cpu_a": cpu_a, "cpu_b": cpu_b, "n_cpu": n_cpu}


# ---------- when is a change "clear"? ----------
def round_values(df, col, fn):
    """{(model, level, iteration): fn(values of col)} per load round (each A round has a B round at the same level and
    iteration; they ran one after the other)."""
    if df is None or not len(df) or "concurrency" not in df.columns:
        return {}
    out = {}
    for (k, c, it), g in df.groupby(["model_key", "concurrency", "iteration"]):
        v = fn(g[col])
        if not _isnan(v):
            out[(k, int(c), int(it))] = float(v)
    return out


def pair_consistency(vals, level=None):
    """The A/B round pairs of one metric: how many show B slower / faster, the range of their changes, and A's own
    round-to-round drift (median over levels of (max - min) / median of A's rounds at that level)."""
    nan = float("nan")
    rels = [rel_delta(vals[("A", c, it)], vals[("B", c, it)]) for (k, c, it) in vals
            if k == "A" and ("B", c, it) in vals and (level is None or c == level)]
    rels = [r for r in rels if not _isnan(r)]
    spreads = []
    for c in sorted({c for k, c, _ in vals if k == "A" and (level is None or c == level)}):
        av = [v for (k, c2, _), v in vals.items() if k == "A" and c2 == c]
        if len(av) >= 2 and np.median(av) > 0:
            spreads.append((max(av) - min(av)) / float(np.median(av)) * 100.0)
    return {"n": len(rels), "slower": sum(r > 0 for r in rels), "faster": sum(r < 0 for r in rels),
            "lo": min(rels) if rels else nan, "hi": max(rels) if rels else nan,
            "drift": float(np.median(spreads)) if spreads else nan}


def judge(rel, ci=NAN2, cons=None, n=None):
    """(class, why). Lower is better. 'same' = inside the noise band; 'few' = fewer than MIN_SAMPLES measurements or no
    confidence interval (no verdict); 'unclear' = the 95% CI includes 0, or fewer than ROUND_PAIR_MIN A/B round pairs,
    or fewer than ROUND_PAIR_AGREE of them agree in direction, or the change is smaller than A's own round-to-round
    drift; otherwise 'better' / 'worse'."""
    if _isnan(rel):
        return None, ""
    if abs(rel) < float(noise_band_pct):
        return "same", ""
    if n is not None and n < MIN_SAMPLES:
        return "few", ("single measurement, no verdict" if n == 1 else f"only {n} measurements, no verdict")
    if _isnan(ci[0]):
        return "few", "no confidence interval, no verdict"
    if ci[0] <= 0 <= ci[1]:
        return "unclear", "the 95% range includes no change"
    if cons is not None:
        if cons["n"] < ROUND_PAIR_MIN:
            return "unclear", f"only {_plural(cons['n'], 'A/B round pair')} (at least {ROUND_PAIR_MIN} needed)"
        agree = cons["slower"] if rel > 0 else cons["faster"]
        if agree < ROUND_PAIR_AGREE * cons["n"]:
            return "unclear", f"only {agree} of {cons['n']} A/B round pairs agree"
        if not _isnan(cons["drift"]) and abs(rel) < cons["drift"]:
            return "unclear", f"smaller than A's own round-to-round spread ({cons['drift']:.0f}%)"
    return ("better" if rel < 0 else "worse"), ""


def classify_change(rel, ci=NAN2, nan_ci="few", cons=None, n=None):
    """judge()'s class. nan_ci='direction' only for chart marks (dot colours), never for a verdict."""
    if nan_ci == "direction" and _isnan(ci[0]) and not _isnan(rel) and abs(rel) >= float(noise_band_pct):
        return "better" if rel < 0 else "worse"
    return judge(rel, ci, cons, n)[0]


def _rel_color(rel, ci=NAN2):
    """Chart marks: grey = inside the noise band, green / red = direction (a single measurement, not a verdict)."""
    return CLASS_COLOR[classify_change(rel, ci, nan_ci="direction")]


def _pairs_short(k, words=("faster", "slower")):
    c = k.get("cons")
    if not c or not c["n"] or _isnan(k["rel"]):
        return ""
    agree = c["slower"] if k["rel"] > 0 else c["faster"]
    return f"{agree} of {c['n']} round pairs {words[1] if k['rel'] > 0 else words[0]}"


def _reading(k, b_label_, words=("faster", "slower")):
    cls = k.get("cls")
    if cls is None:
        return "", GREY
    if cls == "same":
        return "≈ no meaningful change (inside noise band)", GREY
    arrow, word = ("▼", words[0]) if k["rel"] < 0 else ("▲", words[1])
    if cls == "few":
        return f"{b_label_} {word}: {k['why']}", GREY
    if cls == "unclear":
        return f"{arrow} {b_label_} {word}, not clear: {k['why']}", AMBER
    return f"{arrow} {b_label_} {word}", CLASS_COLOR[cls]


def _change_text(k, words, latency=False):
    """Plain-language change of B vs A, e.g. 'about 2.1x faster (-52%)'. Returns (text, class)."""
    a, b, rel, v = k["a"], k["b"], k["rel"], k["cls"]
    if v is None:
        return "not measured", None
    if v == "same":
        return f"no meaningful change ({_pct0(rel)})", v
    if v == "few":
        return f"{_pct0(rel)} ({k['why']})", v
    if v == "unclear":
        return f"{abs(rel):.0f}% {words[0] if rel < 0 else words[1]}, not clear: {k['why']}", v
    word = words[0] if v == "better" else words[1]
    ratio = (a / b if v == "better" else b / a) if (a > 0 and b > 0) else 0
    txt = f"about {ratio:.1f}× {word} ({_pct0(rel)})" if ratio >= 1.5 else f"{word} ({_pct0(rel)})"
    if latency and abs(b - a) < NOTICEABLE_MS:
        txt += ", not noticeable to users"
    pairs = _pairs_short(k, ("faster", "slower"))
    return txt + (f"; {pairs}" if pairs else ""), v


def kp(a, b, ci=NAN2, cons=None, n=None, **extra):
    rel = rel_delta(a, b)
    cls, why = judge(rel, ci, cons, n)
    return {"a": a, "b": b, "rel": rel, "ci": ci, "cls": cls, "why": why, "cons": cons, "n": n, **extra}


def _visual_change(a_vals, b_vals, cons):
    """Plain words for the typical-wait change of one visual, and its class."""
    k = kp(pctl(a_vals, 50), pctl(b_vals, 50), boot_ci_rel(a_vals, b_vals, 50, BOOT_N), cons,
           n=min(len(a_vals), len(b_vals)))
    if k["cls"] is None:
        return "-", None
    if k["cls"] == "same":
        return f"about the same ({_pct0(k['rel'])})", "same"
    if k["cls"] == "unclear":
        return f"no clear difference ({_pct0(k['rel'])}; {k['why']})", "unclear"
    return _change_text(k, ("faster", "slower"), latency=True)


def _chart_html(title, fn):
    """Draw a chart into an <img> tag (shown inline and kept in the saved report)."""
    try:
        before = set(plt.get_fignums())
        msg = fn()
        out = ""
        for n in sorted(set(plt.get_fignums()) - before):
            fig = plt.figure(n)
            buf = io.BytesIO()
            fig.savefig(buf, format="png", dpi=100, bbox_inches="tight")
            plt.close(fig)
            out += (f"<img style='max-width:100%;margin:6px 0' "
                    f"src='data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}'>")
        if msg:
            out += f"<div style='{_FONT};color:#555;margin-bottom:8px'>{_html.escape(str(msg))}</div>"
        return out
    except Exception as exc:
        plt.close("all")
        return f"<div style='{_FONT};color:#999'>(chart '{_html.escape(title)}' skipped: {_html.escape(str(exc)[:200])})</div>"


def _visual_name(q, seq):
    title = str(q.get("visual_title") or q.get("visual_type") or "visual") if q else ""
    return f"#{int(seq)} {title if len(title) <= 40 else title[:39] + chr(0x2026)}".strip()


def cpu_drivers(sm, cat):
    """Server CPU of the traced pass (one run per query and model): net change, largest increase, largest decrease."""
    p = pivot_by_model(sm, "cpu_ms").dropna(subset=["A", "B"])
    if p.empty:
        return None
    d = p["B"] - p["A"]
    info = {int(r["query_seq"]): r for r in cat.to_dict("records")} if len(cat) else {}

    def one(seq):
        return {"seq": int(seq), "name": _visual_name(info.get(int(seq)), seq), "a": float(p.at[seq, "A"]),
                "b": float(p.at[seq, "B"])}
    return {"n": len(d), "net": float(d.sum()), "rel": rel_delta(float(p["A"].sum()), float(p["B"].sum())),
            "inc": one(d.idxmax()) if d.max() >= CPU_CHANGE_MIN_MS else None,
            "dec": one(d.idxmin()) if d.min() <= -CPU_CHANGE_MIN_MS else None}


def _cpu_driver_text(dr):
    if not dr:
        return ""
    net = ("−" if dr["net"] < 0 else "+") + _fmt_short(abs(dr["net"]))
    parts = [f"{x} {dr[k]['name']}: {_fmt_short(dr[k]['a'])} → {_fmt_short(dr[k]['b'])}"
             for x, k in (("largest increase", "inc"), ("largest decrease", "dec")) if dr[k]]
    return (f"Server CPU, one traced run per query: net {net} ({_pct0(dr['rel'])}) over "
            f"{_plural(dr['n'], 'query', 'queries')}" + (f"; {'; '.join(parts)}." if parts else "."))


def _levels_text(levels):
    return _join_list([_plural(int(lv), "user") for lv in levels])


# ---------- all numbers, computed once ----------
def compute_metrics(cid, meta, D):
    """Every number the report, the executive summary, the text summary and History show, computed once."""
    nan = float("nan")
    LBL = {k: (str(meta[f"{k.lower()}_label"]) if meta is not None else k) for k in "AB"}
    cat, prof, ex, sm, rc = (D[n] for n in ("lt_query_catalog", "lt_model_profile", "lt_executions",
                                             "lt_server_metrics", "lt_result_compare"))
    params = parse_params(meta)
    mode = params.get("replay_mode", "sequential")
    qinfo = {int(r["query_seq"]): r for r in cat.to_dict("records")} if len(cat) else {}
    n_q = len(qinfo) or int(ex.loc[ex["query_seq"] > 0, "query_seq"].nunique())
    load_all = ex[ex["phase"] == "load"]
    load_q = load_all[load_all["query_seq"] > 0]
    load_ok = load_q[load_q["success"]]
    dur = {k: load_ok.loc[load_ok["model_key"] == k, "duration_ms"] for k in "AB"}
    # Click-based metrics compare like with like. A query that fails persistently in a model (a query error, or more
    # than PERSISTENT_FAIL_PCT of its runs) is left out of every click; a transient failure (throttling, timeout,
    # connection) only drops the click it happened in (and so that user's click-through).
    fails = load_q[~load_q["success"]]
    fails = fails.assign(cls=fails["error"].map(error_class)) if len(fails) else fails.assign(cls=pd.Series(dtype=str))
    persistent = {"A": set(), "B": set()}
    for (s, k), g in fails.groupby(["query_seq", "model_key"]) if len(fails) else []:
        runs_n = int(((load_q["query_seq"] == s) & (load_q["model_key"] == k)).sum())
        if (~g["cls"].isin(TRANSIENT_ERRORS)).any() or len(g) > PERSISTENT_FAIL_PCT / 100 * runs_n:
            persistent[k].add(int(s))
    failing = persistent["A"] | persistent["B"]
    common = load_ok[~load_ok["query_seq"].isin(failing)]
    n_common = int(common["query_seq"].nunique())
    hit = fails[~fails["query_seq"].isin(failing)]
    hit_keys = set(zip(hit["model_key"], hit["round_no"], hit["user_id"], hit["action_seq"])) if len(hit) else set()
    rinfo = (load_all.groupby("round_no").agg(concurrency=("concurrency", "first"), iteration=("iteration", "first"))
             if len(load_all) else pd.DataFrame(columns=["concurrency", "iteration"]))

    def with_round(df):   # adds the level (concurrency) and iteration of each row's round
        if not len(df):
            return df.assign(concurrency=pd.Series(dtype=int), iteration=pd.Series(dtype=int))
        return df.assign(concurrency=df["round_no"].map(rinfo["concurrency"]), iteration=df["round_no"].map(rinfo["iteration"]))
    clicks = click_waits(common, mode)
    if hit_keys and len(clicks):   # a click with a transient failure is incomplete: leave that click instance out
        keep = [k not in hit_keys for k in zip(clicks["model_key"], clicks["round_no"], clicks["user_id"], clicks["action_seq"])]
        clicks = clicks[keep].reset_index(drop=True)
    clicks = with_round(clicks)
    j = (clicks.groupby(["model_key", "round_no", "user_id"]).agg(total=("wait_ms", "sum"), n=("n", "sum")).reset_index()
         if len(clicks) else pd.DataFrame(columns=["model_key", "round_no", "user_id", "total", "n"]))
    j = with_round(j[j["n"] == n_common])
    journeys = {k: j.loc[j["model_key"] == k, "total"] for k in "AB"}
    med = lambda s: pctl(s, 50)   # noqa: E731

    K = {}
    if len(load_ok):
        n_min = min(len(dur["A"]), len(dur["B"]))
        for q, name in ((50, "p50"), (90, "p90"), (95, "p95")):
            cons = pair_consistency(round_values(load_ok, "duration_ms", lambda s, q=q: pctl(s, q)))
            K[name] = kp(pctl(dur["A"], q), pctl(dur["B"], q), boot_ci_rel(dur["A"], dur["B"], q, BOOT_N), cons, n_min,
                         n_a=len(dur["A"]), n_b=len(dur["B"]))
        K["max"] = kp(pctl(dur["A"], 100), pctl(dur["B"], 100), n=1)
        K["journey"] = kp(pctl(journeys["A"], 50), pctl(journeys["B"], 50),
                          boot_ci_rel(journeys["A"], journeys["B"], 50, BOOT_N), pair_consistency(round_values(j, "total", med)),
                          min(len(journeys["A"]), len(journeys["B"])), n_a=len(journeys["A"]), n_b=len(journeys["B"]))
    err_n = {k: int((~load_all.loc[load_all["model_key"] == k, "success"]).sum()) for k in "AB"}
    exec_n = {k: int((load_all["model_key"] == k).sum()) for k in "AB"}
    if len(load_all):
        K["err_pct"] = kp(*(100.0 * err_n[k] / exec_n[k] if exec_n[k] else nan for k in "AB"))
    ft = prof[prof["item_type"] == "first_touch_ms"] if len(prof) else prof
    first_touch = {k: pd.to_numeric(ft.loc[ft["model_key"] == k, "value_num"], errors="coerce").max() for k in "AB"} if len(ft) else {}
    if first_touch:
        K["first_touch"] = kp(first_touch["A"], first_touch["B"], n=1)
    wa, wb, wci, wn = paired_totals(ex[(ex["phase"] == "warmup") & ex["success"]], "duration_ms")
    if wn:
        K["warmup"] = kp(wa, wb, wci, n=wn)
    for key, col in (("cpu", "cpu_ms"), ("server_dur", "server_duration_ms"), ("se_queries", "se_queries")):
        a, b, ci, n = paired_totals(sm, col)
        if n:
            K[key] = kp(a, b, ci, n=n)

    by_level = []
    p50_rounds = round_values(load_ok, "duration_ms", med)
    p90_rounds = round_values(load_ok, "duration_ms", lambda s: pctl(s, 90))
    for c in sorted(load_ok["concurrency"].unique()):
        sel = load_ok[load_ok["concurrency"] == c]
        a, b = sel.loc[sel["model_key"] == "A", "duration_ms"], sel.loc[sel["model_key"] == "B", "duration_ms"]
        lv = load_all[load_all["concurrency"] == c]
        by_level.append({"level": int(c), "n_a": len(a), "n_b": len(b),
                         "p50": kp(pctl(a, 50), pctl(b, 50), boot_ci_rel(a, b, 50, BOOT_N),
                                   pair_consistency(p50_rounds, int(c)), min(len(a), len(b))),
                         "p90": kp(pctl(a, 90), pctl(b, 90), boot_ci_rel(a, b, 90, BOOT_N),
                                   pair_consistency(p90_rounds, int(c)), min(len(a), len(b))),
                         "max": (pctl(a, 100), pctl(b, 100)),
                         "errors": {k: int((~lv.loc[lv["model_key"] == k, "success"]).sum()) for k in "AB"}})

    par = params.get("click_parallelism", {})
    actions = []
    for a_seq in sorted(clicks["action_seq"].unique()) if len(clicks) else []:
        qs = [q for q in qinfo.values() if int(q["action_seq"]) == a_seq]
        sel = clicks[clicks["action_seq"] == a_seq]
        w = {k: sel.loc[sel["model_key"] == k, "wait_ms"] for k in "AB"}
        actions.append({"action_seq": int(a_seq), "label": qs[0]["action_label"] if qs else "",
                        "n_queries": int(common.loc[common["action_seq"] == a_seq, "query_seq"].nunique()),
                        "parallel": par.get(str(int(a_seq))), "waits": w,
                        **kp(pctl(w["A"], 50), pctl(w["B"], 50), boot_ci_rel(w["A"], w["B"], 50, BOOT_N),
                             pair_consistency(round_values(sel, "wait_ms", med)), min(len(w["A"]), len(w["B"])))})

    # per query (visual): load latency, server CPU, result check
    med_q = load_ok.groupby(["query_seq", "model_key"])["duration_ms"].median().unstack("model_key") if len(load_ok) else pd.DataFrame()
    p90 = load_ok.groupby(["query_seq", "model_key"])["duration_ms"].quantile(0.9).unstack("model_key") if len(load_ok) else pd.DataFrame()
    errs = load_q[~load_q["success"]].groupby(["query_seq", "model_key"]).size().unstack("model_key") if len(load_q) else pd.DataFrame()
    cpu = pivot_by_model(sm, "cpu_ms")
    rcls = classify_results(rc)
    rcm = rc.set_index("query_seq") if len(rc) else pd.DataFrame()
    runs = {(int(s), k): g["duration_ms"].to_numpy(dtype=float) for (s, k), g in load_ok.groupby(["query_seq", "model_key"])}
    q_rounds = {}
    for (s, k, c, it), v in (load_ok.groupby(["query_seq", "model_key", "concurrency", "iteration"])["duration_ms"].median().items()
                             if len(load_ok) else []):
        q_rounds.setdefault(int(s), {})[(k, int(c), int(it))] = float(v)

    def _get(df, seq, k):
        try:
            return df.loc[seq, k]
        except Exception:
            return nan

    per_query = []
    for seq in sorted(qinfo) if qinfo else sorted(med_q.index):
        q = qinfo.get(seq, {})
        ma, mb, pa_, pb = _get(med_q, seq, "A"), _get(med_q, seq, "B"), _get(p90, seq, "A"), _get(p90, seq, "B")
        cA, cB = _get(cpu, seq, "A"), _get(cpu, seq, "B")
        change, change_cls = _visual_change(runs.get((seq, "A"), np.array([])), runs.get((seq, "B"), np.array([])),
                                            pair_consistency(q_rounds.get(seq, {})))
        per_query.append({
            "#": seq, "action": f"{q.get('action_seq', '')}. {q.get('action_label', '')}",
            "visual": q.get("visual_title") or q.get("visual_type", ""), "type": q.get("visual_type", ""),
            f"rows {LBL['A']}": _get(rcm, seq, "rows_a"), f"rows {LBL['B']}": _get(rcm, seq, "rows_b"),
            "result": rcls.get(seq, "n/a") if len(rcls) else "n/a",
            f"median {LBL['A']}": ma, f"median {LBL['B']}": mb, "Δ% median": rel_delta(ma, mb),
            f"p90 {LBL['A']}": pa_, f"p90 {LBL['B']}": pb, "Δ% p90": rel_delta(pa_, pb),
            f"CPU {LBL['A']}": cA, f"CPU {LBL['B']}": cB, "Δ% CPU": rel_delta(cA, cB),
            f"errors {LBL['A']}": _get(errs, seq, "A"), f"errors {LBL['B']}": _get(errs, seq, "B"),
            "change": change, "change_cls": change_cls, "runs": len(runs.get((seq, "A"), [])),
        })

    # "fails only in one model" = persistent failures: in the load test (above) or a query error in another phase
    other_q = ex[~ex["success"] & (ex["query_seq"] > 0) & (ex["phase"] != "load")]
    other_q = other_q[~other_q["error"].map(error_class).isin(TRANSIENT_ERRORS)] if len(other_q) else other_q
    f_a, f_b = (persistent[k] | set(other_q.loc[other_q["model_key"] == k, "query_seq"]) for k in "AB")
    other_err = ex[(ex["phase"] != "load") & ~ex["success"]]
    all_err = pd.concat([load_all[~load_all["success"]], other_err])
    if len(all_err):
        all_err = all_err.assign(error=all_err["error"].astype(str), cls=all_err["error"].map(error_class))
        err_table = (all_err.assign(error=lambda d: d["error"].str.slice(0, 160))
                     .groupby(["phase", "model_label", "cls", "error"]).size().reset_index(name="count")
                     .rename(columns={"cls": "class"}).sort_values("count", ascending=False))
    else:
        err_table = pd.DataFrame()
    load_err = all_err[all_err["phase"] == "load"] if len(all_err) else all_err
    err_class = {k: (load_err.loc[load_err["model_key"] == k, "cls"].value_counts().to_dict() if len(load_err) else {})
                 for k in "AB"}

    dd = diff_definitions(prof) if len(prof) else pd.DataFrame(columns=["kind", "object", "status"])
    tv = table_row_diffs(prof)
    big_tv = tv.dropna(subset=["A", "B"])
    big_tv = big_tv[big_tv["A"] > 0]
    big_tv = big_tv[(big_tv["ratio"] - 1).abs() > ROWS_TRADEOFF_PCT / 100].sort_values("A", ascending=False)
    refresh = refresh_info(prof)
    t_ref = {k: pd.to_datetime(refresh["by_model"][k]["before"], errors="coerce") for k in "AB"}
    gap_days = ((t_ref["A"] - t_ref["B"]).total_seconds() / 86400
                if not (pd.isna(t_ref["A"]) or pd.isna(t_ref["B"])) else nan)

    # planned vs run (rounds per model, levels): partial runs must say what is missing
    lv_plan = _parse_int_list(params.get("concurrency_levels", "")) if params.get("concurrency_levels") else []
    its = int(params.get("iterations") or 0)
    planned_on = bool(lv_plan and its and params.get("run_load_test", True))
    rounds_run = {k: int(load_all.loc[load_all["model_key"] == k, "round_no"].nunique()) for k in "AB"}
    lv_ok = {k: set(load_ok.loc[load_ok["model_key"] == k, "concurrency"]) for k in "AB"}
    drift = {}
    for k in "AB":   # the largest spread of one model's round medians at one level
        for c in sorted({c for kk, c, _ in p50_rounds if kk == k}):
            v = [x for (kk, c2, _), x in p50_rounds.items() if kk == k and c2 == c]
            if len(v) >= 2 and np.median(v) > 0:
                s = (max(v) - min(v)) / float(np.median(v)) * 100
                if s > drift.get(k, (-1,))[0]:
                    drift[k] = (s, c, len(v))

    NC = numcheck_from_profile(prof)
    attrib = None   # whole-query differences: do they use a measure that differs in the number check?
    if NC and NC["status"] in ("match", "differences") and len(rc) and qinfo and "query_text" in cat.columns:
        flagged = {m.lower(): m for m in NC["flagged"]}
        diff_seqs = [int(s) for s, v in classify_results(rc).items() if v in ("rows differ", "values differ")]
        uses = {s: sorted({flagged[n.lower()] for _, n in _refs(dax_scrub(qinfo[s].get("query_text")))
                           if n.lower() in flagged}) for s in diff_seqs if s in qinfo}
        attrib = {"n_diff": len(diff_seqs), "with": [s for s in uses if uses[s]], "without": [s for s in uses if not uses[s]],
                  "measures": sorted({m for v in uses.values() for m in v})}

    trace = {k: (int(sm.loc[sm["model_key"] == k, "matched_by"].notna().sum()), int((sm["model_key"] == k).sum()))
             for k in "AB"} if len(sm) and "matched_by" in sm.columns else {}
    started = meta["started_utc"] if meta is not None else None
    finished = meta["finished_utc"] if meta is not None else None
    M = {
        "cid": cid, "labels": LBL, "params": params, "replay_mode": mode, "test_name": meta["test_name"] if meta is not None else cid,
        "status": str(meta["status"]) if meta is not None else "?", "message": str(meta["message"]) if meta is not None else "",
        "meta": meta, "t0": pd.Timestamp(started) if not _isnan(started) else None,
        "t1": pd.Timestamp(finished) if not _isnan(finished) else None,
        "run_mode": params.get("run_mode"), "fingerprint": params.get("approval_fingerprint"),
        "qinfo": qinfo, "n_q": n_q, "ex": ex, "sm": sm, "prof": prof, "rc": rc, "cat": cat,
        "load_all": load_all, "load_ok": load_ok, "dur": dur, "failing": failing, "common": common, "n_common": n_common,
        "levels": sorted(int(c) for c in load_all["concurrency"].unique()), "clicks": clicks, "journeys": journeys,
        "kpis": K, "err_n": err_n, "exec_n": exec_n, "err_class": err_class, "by_level": by_level, "actions": actions,
        "per_query": pd.DataFrame(per_query), "results": result_counts(rc), "err_table": err_table,
        "fail_only": {"A": sorted(f_a - f_b), "B": sorted(f_b - f_a)}, "first_touch": first_touch,
        "defs": dd, "logic": dd.loc[dd["kind"].isin(["measure", "calc_item", "calc_column"]), "object"].tolist(),
        "source": dd.loc[dd["kind"].isin(["expression", "partition"]), "object"].tolist(),
        "tables": tv, "tables_listed": tv[(tv["ratio"] - 1).abs() > ROWS_LIST_PCT / 100],
        "big": [(str(n), r["A"], r["B"], r["ratio"]) for n, r in big_tv.iterrows()],
        "capacity": capacity_info(prof), "refresh": refresh, "refresh_gap_days": gap_days,
        "cpu_drivers": cpu_drivers(sm, cat), "trace": trace, "number_check": NC, "attrib": attrib,
        "levels_planned": lv_plan, "levels_both": sorted(lv_ok["A"] & lv_ok["B"]),
        "rounds_run": rounds_run, "rounds_planned_per_model": len(lv_plan) * its if planned_on else None,
        "planned_q": {k: sum(lv_plan) * its * n_q for k in "AB"} if planned_on else None,
        "executed_q": {k: int((load_q["model_key"] == k).sum()) for k in "AB"}, "round_drift": drift, "trend": None,
    }
    M["trust"] = trust_checks(M)
    return M


# ---------- can I trust this run? ----------
def trust_checks(M):
    """[{name, ok, text, strong}]: the facts that decide how far the numbers can be trusted. The report shows
    'N of M checks passed' and only the failing ones (strong = red)."""
    LBL, P, out = M["labels"], M["params"], []

    def check(name, ok, text, strong=False):
        out.append({"name": name, "ok": bool(ok), "text": text, "strong": strong})
    NC = M["number_check"]
    check("number check", NC is not None and NC["status"] in ("match", "differences"),
          "Number check: not available (saved before it existed); the numbers were not compared under a controlled filter."
          if NC is None else f"Number check {NC['status']}: {NC['reason'] or '-'}. The numbers were not compared under a "
          "controlled filter, so silence here does not mean they match.", strong=True)
    if M["planned_q"]:
        pq, eq = M["planned_q"], M["executed_q"]
        rr, rp = M["rounds_run"], M["rounds_planned_per_model"]
        text = (f"Coverage: {sum(eq.values()):,} of {sum(pq.values()):,} planned load queries ran ({LBL['A']} "
                f"{rr['A']} of {rp} rounds, {LBL['B']} {rr['B']} of {rp})")
        if rr["A"] != rr["B"]:
            text += "; the A,B / B,A balance is broken, so drift is not cancelled"
        missing = [lv for lv in M["levels_planned"] if lv not in M["levels_both"]]
        if missing:
            text += f"; not measured: the {_join_list([f'{lv}-user' for lv in missing])} level{'s' if len(missing) > 1 else ''}"
        check("coverage", all(eq[k] >= pq[k] for k in "AB"), text + ".", strong=True)
    lo = M["load_ok"]
    if len(lo):
        runs_ = lo.groupby(["concurrency", "iteration"])["model_key"].nunique()
        pairs = {int(c): int((runs_.loc[c] == 2).sum()) for c in sorted(lo["concurrency"].unique())}
        few = {c: n for c, n in pairs.items() if n < ROUND_PAIR_MIN}
        check("round pairs", not few, "A/B round pairs: " + "; ".join(f"{n} at {_plural(c, 'user')}" for c, n in few.items())
              + f" (at least {ROUND_PAIR_MIN} needed): no result at that level can be clear. Set iterations >= {ROUND_PAIR_MIN}.")
    if M["trace"]:
        bad = {k: (m, t) for k, (m, t) in M["trace"].items() if t and m < 0.9 * t}
        check("trace", not bad, "Server trace matched only " + "; ".join(f"{LBL[k]} {m} of {t} queries" for k, (m, t) in bad.items())
              + ": server CPU is incomplete.")
    elif P.get("run_server_trace"):
        check("trace", False, "Server trace not available (permissions?): server CPU was not measured.")
    if M["refresh"]["known"]:
        rf = M["refresh"]["refreshed"]
        check("refresh", not rf, f"{_join_list([LBL[k] for k in rf])} refreshed during the test: part of its queries ran "
              "with cleared caches.", strong=True)
    cap = M["capacity"]
    check("capacity", cap is True, "The models ran on DIFFERENT capacities: not comparable (speed and cost may come from "
          "the capacity)." if cap is False else "Capacity not verified: a fair comparison needs both on the same capacity.",
          strong=cap is False)
    if len(M["tables"]):   # the data age is in the Data line; a different data volume can bias speed and cost
        big = M["big"]
        check("data volume", not big, "Data volume: " + "; ".join(f"'{n}' {ra:,.0f} → {rb:,.0f} rows (×{ratio:.2f})"
                                                                for n, ra, rb, ratio in big[:3])
              + f" (more than {ROWS_TRADEOFF_PCT}% different): speed and cost partly reflect the data, not only the model.")
    cold = [k for k, v in M["first_touch"].items() if not _isnan(v) and v > COLD_MS]
    check("cold start", not cold, "; ".join(f"{LBL[k]} was cold at the start (first touch {fmt_ms(M['first_touch'][k])})"
                                            for k in cold) + "; the warm-up absorbed it.")
    ec = M["err_class"]
    thr = {k: ec[k].get("throttling", 0) for k in "AB"}
    check("throttling", not any(thr.values()), "Throttling: " + "; ".join(f"{LBL[k]} {_plural(n, 'query', 'queries')}"
                                                                         for k, n in thr.items() if n)
          + " rejected because the capacity was overloaded; latencies under load are affected.", strong=True)
    other = {k: {c: n for c, n in ec[k].items() if c != "throttling"} for k in "AB"}
    check("errors", not any(other.values()), "Errors under load: " + "; ".join(
        f"{LBL[k]} " + ", ".join(f"{n} {c}" for c, n in sorted(v.items())) for k, v in other.items() if v) + ".")
    dr = {k: v for k, v in M["round_drift"].items() if v[0] > ROUND_DRIFT_PCT}
    if M["round_drift"]:
        check("round drift", not dr, "Round drift: " + "; ".join(
            f"{LBL[k]}'s {v[2]} rounds at {_plural(v[1], 'user')} differ by {v[0]:.0f}% in median latency" for k, v in dr.items())
            + f" (limit {ROUND_DRIFT_PCT}%).")
    mode = M["run_mode"]
    if mode:
        gates = P.get("gates") or []
        check("run mode", mode != "scheduled" or all(g.get("ok") for g in gates),
              f"Scheduled run: gates {', '.join(g['gate'] for g in gates if not g.get('ok'))} did not pass.")
    return out


def incomplete_text(M):
    """Plain words for a run that did not finish (None when it finished)."""
    if M["status"] == "completed":
        return None
    msg, P = str(M["message"] or ""), M["params"]
    phase = msg.split(":", 1)[0].replace("interrupted", "").strip() or "the test"
    if "interrupt" in msg.lower() or "cancel" in msg.lower():
        head = f"The test did not finish: it was cancelled during the {phase}"
    elif M["status"] == "failed":
        head = f"The test did not finish: it stopped with an error during the {phase} ({msg.split(':', 1)[-1].strip()[:150]})"
    else:
        head = f"The test did not finish (status '{M['status']}': the notebook stopped before it saved a final status)"
    rp, rr = M["rounds_planned_per_model"], M["rounds_run"]
    done = len(set(M["load_all"]["round_no"])) if len(M["load_all"]) else 0
    rounds = f"{done} of {2 * rp} load rounds have results" if rp else f"{done} load rounds have results"
    missing = [lv for lv in M["levels_planned"] if lv not in M["levels_both"]]
    miss = f"; not measured: the {_join_list([f'{lv}-user' for lv in missing])} level{'s' if len(missing) > 1 else ''}" if missing else ""
    return f"{head}; {rounds}{miss}. The results below are partial."


# ---------- trend: runs with the same approval fingerprint ----------
def trend_key(row, p):
    """The approval fingerprint; for comparisons saved before it existed: query file + models + load shape."""
    fp = p.get("approval_fingerprint")
    if fp:
        return str(fp)
    parts = [row.get("query_file_sha256"), row.get("a_workspace"), row.get("a_model"), row.get("b_workspace"),
             row.get("b_model")] + [p.get(k) for k in ("concurrency_levels", "iterations", "think_time_between_actions_sec",
                                                       "replay_mode")]
    return "legacy-" + _short_hash(json.dumps([None if _isnan(x) else str(x) for x in parts]), 8)


def summary_row(M):
    """lt_run_summary row of one comparison (written once, by the run that made it)."""
    K, NC = M["kpis"], M["number_check"]
    g = lambda k, s: K[k][s] if k in K else float("nan")   # noqa: E731
    failed = [t for t in M["trust"] if not t["ok"]]
    not_comp = ([f"status {M['status']}"] if M["status"] != "completed" else []) + \
               [t["name"] for t in failed if t["name"] in ("coverage", "refresh", "capacity", "throttling", "round drift")]
    metrics = {
        "version": 1, "levels": M["levels"], "journey_a": g("journey", "a"), "journey_b": g("journey", "b"),
        "first_touch_a": M["first_touch"].get("A"), "first_touch_b": M["first_touch"].get("B"),
        "classes": {k: K[k]["cls"] for k in ("p50", "p90", "journey", "cpu") if k in K},
        "by_level_p90": {str(lv["level"]): [lv["p90"]["a"], lv["p90"]["b"]] for lv in M["by_level"]},
        "numbers": numcheck_headline(NC), "number_check": NC["status"] if NC else None,
        "errors": M["err_class"], "trust_failed": [t["name"] for t in failed], "not_comparable": not_comp,
        "report_file": REPORT_FILE,
    }
    clean = lambda v: None if _isnan(v) else v   # noqa: E731
    return {"comparison_id": M["cid"], "test_name": M["test_name"], "started_utc": M["t0"], "status": M["status"],
            "run_mode": M["run_mode"] or "interactive", "fingerprint": M["fingerprint"], "comparable": not not_comp,
            "p50_a": g("p50", "a"), "p50_b": g("p50", "b"), "p90_a": g("p90", "a"), "p90_b": g("p90", "b"),
            "cpu_a": g("cpu", "a"), "cpu_b": g("cpu", "b"),
            "metrics_json": json.dumps(metrics, default=lambda v: None if _isnan(v) else str(v)), "written_utc": utcnow()}


def history_frame():
    """One row per saved comparison (newest first) with its headline numbers: from lt_run_summary, or computed on read
    for comparisons without a summary row (saved before it existed, or stopped early). Nothing is written back."""
    comps = read_comparisons()
    if comps.empty:
        return comps
    summ = read_table("lt_run_summary")
    if len(summ):
        summ = summ.sort_values("written_utc").groupby("comparison_id").tail(1).set_index("comparison_id")
    todo = [c for c, s in zip(comps["comparison_id"], comps["status"]) if c not in summ.index and s != "skipped"]
    ex = read_table("lt_executions", todo, ["comparison_id", "phase", "model_key", "query_seq", "duration_ms", "success"]) \
        if todo else pd.DataFrame(columns=["comparison_id"])
    sm = read_table("lt_server_metrics", todo, ["comparison_id", "model_key", "query_seq", "cpu_ms"]) \
        if todo else pd.DataFrame(columns=["comparison_id"])
    pr = read_table("lt_model_profile", todo, ["comparison_id", "model_key", "item_type", "item_name", "value_num",
                                               "value_text"]) if todo else pd.DataFrame(columns=["comparison_id", "item_type"])
    if len(ex):
        ex["duration_ms"] = pd.to_numeric(ex["duration_ms"], errors="coerce")
        ex["success"] = _bool_col(ex["success"])
        ex["query_seq"] = pd.to_numeric(ex["query_seq"], errors="coerce").fillna(-1).astype(int)
    for c in ("cpu_ms", "query_seq"):
        if c in sm.columns:
            sm[c] = pd.to_numeric(sm[c], errors="coerce")
    pr = pr[pr["item_type"].astype(str).str.startswith("numcheck_")] if len(pr) else pr
    by = {n: dict(tuple(df.groupby("comparison_id"))) if len(df) else {} for n, df in (("ex", ex), ("sm", sm), ("pr", pr))}
    rows = []
    for _, r in comps.iterrows():
        cid, p = r["comparison_id"], parse_params(r)
        base = {"comparison_id": cid, "started": r["started_utc"], "status": r["status"], "message": r["message"],
                "query file": os.path.basename(str(r["query_file"])) if not _isnan(r["query_file"]) else "",
                "A": f"{r['a_label']}: {r['a_workspace']} / {r['a_model']}",
                "B": f"{r['b_label']}: {r['b_workspace']} / {r['b_model']}",
                "replay": "page load" if p.get("replay_mode") == "page_load" else "sequential",
                "levels": p.get("concurrency_levels", ""), "iterations": p.get("iterations", ""),
                "run": p.get("run_mode") or "interactive", "fingerprint": p.get("approval_fingerprint") or "-",
                "key": trend_key(r, p)}
        if cid in summ.index:
            s, m = summ.loc[cid], _nc_json(summ.loc[cid, "metrics_json"], {}) or {}
            base.update({k: s[k] for k in ("p50_a", "p50_b", "p90_a", "p90_b", "cpu_a", "cpu_b")})
            base.update(comparable=bool(s["comparable"]), not_comparable=m.get("not_comparable", []),
                        numbers=m.get("numbers", "-"), source="summary")
        elif r["status"] == "skipped":
            base.update(comparable=False, not_comparable=["skipped"], numbers="-", source="skipped")
        else:
            h = headline(by["ex"].get(cid, ex.iloc[0:0]), by["sm"].get(cid, sm.iloc[0:0]))
            base.update(p50_a=h["median_a"], p50_b=h["median_b"], p90_a=h["p90_a"], p90_b=h["p90_b"], cpu_a=h["cpu_a"],
                        cpu_b=h["cpu_b"], comparable=r["status"] == "completed",
                        not_comparable=[] if r["status"] == "completed" else [f"status {r['status']}"],
                        numbers=numcheck_headline(numcheck_from_profile(by["pr"].get(cid))), source="computed on read")
        rows.append(base)
    return pd.DataFrame(rows)


def trend_view(H, key):
    """The runs with one fingerprint (oldest first) and how the latest one compares with the range of the last
    TREND_RUNS comparable runs before it (no statistical claim: a min-max range)."""
    T = H[(H["key"] == key) & (H["status"] != "skipped")].sort_values("started").reset_index(drop=True)
    if T.empty:
        return None
    latest, prev = T.iloc[-1], T.iloc[:-1]
    ref = prev[prev["comparable"].astype(bool)].tail(TREND_RUNS)
    flags = {}
    for col in ("p90_a", "p90_b"):
        vals = pd.to_numeric(ref[col], errors="coerce").dropna()
        v = latest.get(col)
        if len(vals) >= 3 and not _isnan(v):
            lo, hi = float(vals.min()), float(vals.max())
            flags[col] = ("above" if v > hi else "below" if v < lo else "inside", lo, hi, float(v))
    return {"rows": T, "latest": latest, "n_ref": len(ref), "flags": flags, "key": key}


def trend_line(TV, LBL):
    """One sentence for the main view of a scheduled run."""
    if TV is None:
        return ""
    n, nc = len(TV["rows"]), int(TV["rows"]["comparable"].astype(bool).sum())
    head = f"same approval fingerprint {TV['key']} ({_plural(n, 'run')}, {nc} comparable)"
    if not TV["flags"]:
        return f"{head}: a range needs at least 3 earlier comparable runs; there {'is' if TV['n_ref'] == 1 else 'are'} {TV['n_ref']}."
    parts = []
    for col, k in (("p90_b", "B"), ("p90_a", "A")):
        if col in TV["flags"]:
            state, lo, hi, v = TV["flags"][col]
            rng = f"{_fmt_short(lo).replace(' ms', '')}–{_fmt_short(hi)}" if hi < 1000 else f"{_fmt_short(lo)}–{_fmt_short(hi)}"
            parts.append(f"{LBL[k]} p90 {_fmt_short(v)} is {state} " + (
                f"the range of the last {TV['n_ref']} comparable runs ({rng})" if not parts else f"it ({rng})"))
    return f"{head}: " + "; ".join(parts) + "."


def _freshness(M):
    """'Prod refreshed 2026-10-23 05:01; Branch refreshed 2026-10-13 04:00 (10 days older)', or '' when unknown."""
    A, B = M["labels"]["A"], M["labels"]["B"]
    rb = M["refresh"]["by_model"]
    if not (rb["A"]["before"] or rb["B"]["before"]):
        return ""
    when = {k: (rb[k]["before"] or "unknown")[:16].replace("T", " ") for k in "AB"}
    txt = f"{A} refreshed {when['A']}; {B} refreshed {when['B']}"
    gap = M["refresh_gap_days"]
    if not _isnan(gap) and abs(gap) >= 1:
        txt += f" ({abs(gap):.0f} day{'s' if abs(gap) >= 1.5 else ''} {'older' if gap > 0 else 'newer'} in {B})"
    return txt


def _attrib_text(M):
    """Whole-query differences split by whether the query uses a measure that differs in the number check, e.g.
    '2 use a measure that differs in the number check and 6 use only measures that match; Branch's data is 9 days older'."""
    at, B = M["attrib"], M["labels"]["B"]
    if not at or not at["n_diff"]:
        return ""
    parts = ([f"{len(at['with'])} use a measure that differs in the number check"] if at["with"] else []) + \
            ([f"{len(at['without'])} use only measures that match"] if at["without"] else [])
    gap = M["refresh_gap_days"]
    age = (f"; {B}'s data is {abs(gap):.0f} day{'s' if abs(gap) >= 1.5 else ''} {'older' if gap > 0 else 'newer'}"
           if not _isnan(gap) and abs(gap) >= 1 else "")
    return f"{' and '.join(parts)}{age}"


# ---------- executive summary: facts only, no merge verdict ----------
def summary_sections(M):
    """The executive summary once, as plain-text parts; _summary_html and _summary_text render it."""
    A, B, LBL = M["labels"]["A"], M["labels"]["B"], M["labels"]
    K, NC, R = M["kpis"], M["number_check"], M["results"]
    page = M["replay_mode"] != "sequential"
    done = M["status"] == "completed"
    kpis, pros, cons, unclear, same = [], [], [], [], []

    def kpi(label, key, words, latency, short=None, fmt=_fmt_short):
        k = K.get(key)
        if not k:
            return None, float("nan")
        txt, v = _change_text(k, words, latency)
        kpis.append((label, fmt(k["a"]), fmt(k["b"]), txt, v))
        name = label.rstrip(" ¹²")
        if short and v in ("better", "worse"):
            lo, hi = sorted(k["ci"])
            (pros if v == "better" else cons).append(
                f"{name}: {A} {fmt(k['a'])} → {B} {fmt(k['b'])}, {txt} (95% range {_pct0(lo)} to {_pct0(hi)})")
        elif short and v in ("unclear", "few"):
            unclear.append(f"{short} {_pct0(k['rel'])} ({k['why']})")
        elif short and v == "same":
            same.append(short)
        return v, k["rel"]

    speed, speed_rel = kpi("Slowest 10% of visuals (all levels pooled)", "p90", ("faster", "slower"), True,
                           "slowest 10% of visuals")
    kpi("Slowest single visual (single run)", "max", ("faster", "slower"), True)
    kpi("Typical visual (median, all levels pooled)", "p50", ("faster", "slower"), True, "typical visual")
    if K.get("journey") and K["journey"]["n_a"] and K["journey"]["n_b"]:
        kpi(("Waiting time for one full click-through of the report ¹" if page
             else "Query time for one full click-through of the report ¹"), "journey", ("faster", "slower"), True,
            "full click-through")
    cap, cap_rel = kpi("Server CPU for one full click-through (drives CU usage) ²", "cpu", ("less", "more"), False,
                       "server CPU")
    e, n_e, ec = M["err_n"], M["exec_n"], M["err_class"]
    cls_txt = lambda k: (" (" + ", ".join(f"{n} {c}" for c, n in sorted(ec[k].items())) + ")") if ec[k] else ""  # noqa: E731
    if len(M["load_all"]):
        ev = "same" if e["A"] == e["B"] else ("worse" if e["B"] > e["A"] else "better")
        etxt = ("none in either model" if e["A"] == e["B"] == 0 else "same number" if ev == "same"
                else f"{abs(e['B'] - e['A'])} {'more' if ev == 'worse' else 'fewer'} in {B}")
        kpis.append(("Failed queries under load", f"{e['A']} of {n_e['A']:,}{cls_txt('A')}", f"{e['B']} of {n_e['B']:,}{cls_txt('B')}",
                     etxt, ev))   # failures are named here (and in the trust line), not again in Pros / Cons
    clear = {kind: sorted([a for a in M["actions"] if a["cls"] == kind], key=lambda a: a["rel"] if kind == "better" else -a["rel"])
             for kind in ("better", "worse")}
    for kind, bucket in (("better", pros), ("worse", cons)):
        for a in clear[kind][:3]:
            bucket.append(f"Click {a['action_seq']}. {a['label']}: a user waits {A} {_fmt_short(a['a'])} → "
                          f"{B} {_fmt_short(a['b'])} ({_pct0(a['rel'])}; {_pairs_short(a)})")
    if M["fail_only"]["A"]:
        pros.append(f"{_plural(len(M['fail_only']['A']), 'report query', 'report queries')} "
                    f"fail{'s' if len(M['fail_only']['A']) == 1 else ''} only in {A} and work in {B}")

    # ---- blockers (red): what must be resolved or explained first; an incomplete run comes first ----
    blockers = []   # (headline, sub-lines, hide the sub-lines in HTML because a table below shows them)
    inc = incomplete_text(M)
    if inc:
        blockers.append((inc, [], False))
    if NC and NC["status"] == "differences":
        flagged = numcheck_flagged_rows(NC, LBL)
        lines = [f"[{f['measure']}]: {A} {f['a']} vs {B} {f['b']}" + (f" ({_fmt_rel(f['rel'])})" if not _isnan(f["rel"]) else "")
                 + (f", {f['where']}" if f["where"] else "") + (f"; cause: {f['causes'][0]}" if f["causes"] else "")
                 for f in flagged[:5]]
        if len(flagged) > 5:
            lines.append(f"+{len(flagged) - 5} more in Details for analysts > Number check")
        blockers.append((f"Numbers differ under a controlled filter: {NC['n_flagged']} of {len(NC['measures'])} measures used "
                         f"by the report ({_join_list(['[' + f['measure'] + ']' for f in flagged], 3)}; "
                         f"{_plural(len(NC['bus']), 'business unit')}, {NC['start']} to {NC['end']}, tolerance "
                         f"{NC['tolerance']:g}%).", lines, True))
    if M["fail_only"]["B"]:
        names = [query_label(M["qinfo"][s], 60) if s in M["qinfo"] else f"#{s}" for s in M["fail_only"]["B"]]
        blockers.append((f"{_plural(len(names), 'report query', 'report queries')} fail{'s' if len(names) == 1 else ''} "
                         f"only in {B}: {_join_list(names, 3)}.", [], False))

    T = M["trust"]
    trust = {"passed": sum(t["ok"] for t in T), "total": len(T), "failing": [(t["text"], t["strong"]) for t in T if not t["ok"]]}

    # ---- context ----
    t0, t1, levels = M["t0"], M["t1"], M["levels"]
    rr, rp = M["rounds_run"], M["rounds_planned_per_model"]
    if levels and rp and rr["A"] == rr["B"] == rp:
        rounds = (f"{_join_list(levels)} concurrent user{'' if levels == [1] else 's'} × "
                  f"{M['params'].get('iterations', '?')} round(s) per model")
    elif levels:
        rounds = f"rounds run: {A} {rr['A']}, {B} {rr['B']}" + (f" of {rp} planned per model" if rp else "")
    else:
        rounds = "no load test"
    context = [f"Test {fmt_ts(t0)}" if t0 is not None else "Test date unknown",
               {True: "same capacity", False: "DIFFERENT capacities", None: "capacity not verified"}[M["capacity"]],
               rounds, f"{_plural(M['n_q'], 'recorded report query', 'recorded report queries')}",
               "page-load replay" if page else "sequential replay"]
    if M["run_mode"]:
        context.append(f"{M['run_mode']} run")
    fresh = _freshness(M)

    # ---- bottom line: clear facts first; an unclear result is said to be unclear, never "looks" ----
    sentences = []
    n_clear = len(clear["better"]) + len(clear["worse"])
    if n_clear:
        parts = [f"{w} on " + _join_list([f"click {a['action_seq']}. {a['label']} ({_pct0(a['rel'])})" for a in clear[kind][:3]])
                 for kind, w in (("worse", "slower"), ("better", "faster")) if clear[kind]]
        sentences.append(f"By click, {B} is " + " and ".join(parts)
                         + f"; {'both are' if n_clear == 2 else 'it is' if n_clear == 1 else 'all are'} clear.")
    p90 = K.get("p90")
    if speed in ("better", "worse"):
        sp = f"{B} is {'faster' if speed == 'better' else 'slower'} for users (slowest 10% of visuals, all levels pooled: {_pct0(speed_rel)})"
    elif speed == "same":
        sp = f"{B} is about as fast as {A} for users (slowest 10% of visuals, all levels pooled: {_pct0(speed_rel)})"
    elif speed:
        sp = ((f"Overall (all levels pooled) there is no clear difference: slowest 10% of visuals {_pct0(speed_rel)}, "
               if n_clear else f"No clear speed difference for users (slowest 10% of visuals {_pct0(speed_rel)}, "
                               "all levels pooled; ") + f"{p90['why']}" + ("" if n_clear else ")"))
    else:
        sp = None
    if sp:
        sentences.append(sp + ".")
    cp = None
    if cap:
        ca, cb = K["cpu"]["a"], K["cpu"]["b"]
        ratio = (ca / cb if cap == "better" else cb / ca) if (ca > 0 and cb > 0) else 0
        amount = f"about {ratio:.1f}× " if ratio >= 1.5 else ""
        cp = {"better": f"{B} uses {amount}less capacity (server CPU {_pct0(cap_rel)})",
              "worse": f"{B} uses {amount}more capacity (server CPU {_pct0(cap_rel)})",
              "same": f"{B} uses about the same capacity (server CPU {_pct0(cap_rel)})"}.get(
            cap, f"No clear capacity difference (server CPU {_pct0(cap_rel)}; {K['cpu']['why']})")
        sentences.append(cp + ".")
    opposite = speed in ("better", "worse") and cap in ("better", "worse") and speed != cap   # both clear, opposite
    n_rc, n_diff = R["n"], R["n_diff"]
    if NC and NC["status"] == "match":
        sentences.append(f"All {len(NC['measures']) - NC['n_unchecked']} measures used by the report match under a controlled "
                         f"filter ({_plural(len(NC['bus']), 'business unit')}, {NC['start']} to {NC['end']}).")
    elif NC and NC["status"] == "differences":
        sentences.append(f"{NC['n_flagged']} of {len(NC['measures'])} measures used by the report differ under a controlled "
                         "filter (see Blockers).")
    else:
        if n_rc:
            if n_diff:
                sentences.append(f"{n_diff} of {n_rc} report queries return different numbers — review before merging.")
            elif not R["errors"]:
                sentences.append(f"All {n_rc} report queries return identical results.")
        if NC:
            sentences.append(numcheck_line(NC, LBL))
    if not sentences:
        sentences.append("Not enough measurements for a summary (see the run status).")

    neither = []
    if same:
        neither.append(f"No meaningful change (within ±{float(noise_band_pct):g}%): " + _join_list(same) + ".")
    if unclear:
        neither.append("Not clear (a repeat test could show the opposite): " + "; ".join(unclear) + ".")
    if len(M["by_level"]) >= 2:
        neither.append(f"By load level, {B} vs {A} p90: " + ", ".join(
            f"{_pct0(lv['p90']['rel'])} at {_plural(lv['level'], 'user')}" for lv in M["by_level"]) + ".")
    if M["cpu_drivers"]:
        neither.append(_cpu_driver_text(M["cpu_drivers"]))

    # ---- trade-offs ----
    trade, logic, source, big = [], M["logic"], M["source"], M["big"]
    if opposite:
        trade.append(("Speed vs capacity", f"{sp}, but {cp}."))
    if logic:
        d = f"{B} changes {_plural(len(logic), 'definition')}: {_join_list(logic)}."
        if NC and NC["status"] in ("match", "differences"):
            changed = {o.rsplit("[", 1)[-1].rstrip("]").lower() for o in logic}
            used = [m for m in NC["measures"] if m.lower() in changed]
            if used:
                diff_used = [m for m in used if m in NC["flagged"]]
                d += (f" Of these, the report uses {_join_list(used)}; under the controlled filter "
                      + (f"{_join_list(diff_used)} return{'s' if len(diff_used) == 1 else ''} different numbers."
                         if diff_used else "they return the same numbers."))
            else:
                d += " The report's measures return the same numbers under the controlled filter." if NC["status"] == "match" else ""
        else:
            d += " Confirm the new numbers are intended."
        trade.append(("Changed logic", d))
    if n_diff:
        parts = ([f"{R['values_diff']} with the same rows but different values"] if R["values_diff"] else []) + \
                ([f"{R['rows_diff']} with a different row count"] if R["rows_diff"] else [])
        d = f"{n_diff} of {n_rc} recorded report queries return different results ({', '.join(parts)})."
        at = _attrib_text(M)
        if at:
            d = d[:-1] + f": {at}."
        elif big:
            d += f" Part of this is expected: {B} has different data ({big[0][0]}: ×{big[0][3]:.1f} rows)."
        trade.append(("Some report numbers change", d))
    if big:
        n0, ra, rb, ratio = big[0]
        more = ratio > 1
        d = f"{n0}: {ra:,.0f} → {rb:,.0f} rows (×{ratio:.1f})"
        d += f"; {len(big) - 1} more table(s) differ." if len(big) > 1 else "."
        d += (" More data uses more memory and can make the first view after each refresh slower and costlier."
              if more else f" Less data can make {B} look faster or cheaper than it would be with the full data.")
        trade.append((f"{'More' if more else 'Less'} data in {B}", d))
    if source:
        trade.append(("Different data source or parameters",
                      f"{B} differs in {_join_list(source)}. These results reflect {B}'s data; after a merge the model may "
                      "point to other data."))
    if not trade:
        checked = ["no definition changes", f"no table more than {ROWS_TRADEOFF_PCT}% larger or smaller"]
        if n_rc and not n_diff and not R["errors"]:
            checked.append("identical query results")
        trade.append(("No trade-offs detected", _join_list(checked).capitalize() + "."))

    # ---- questions and next steps ----
    questions = []
    flagged_rows = numcheck_flagged_rows(NC, LBL) if NC else []
    by_cause = {}
    for f in flagged_rows:
        for code, _ in NC["causes"].get(f["measure"], []):
            by_cause.setdefault(code, []).append(f"[{f['measure']}]")
    if by_cause.get("logic_changed") or by_cause.get("dependency_changed"):
        ms = by_cause.get("logic_changed", []) + by_cause.get("dependency_changed", [])
        questions.append(f"Are the new numbers of {_join_list(sorted(set(ms)), 3)} from the changed logic intended? "
                         "This needs business sign-off.")
    other = sorted({f"[{f['measure']}]" for f in flagged_rows} - set(by_cause.get("logic_changed", []))
                   - set(by_cause.get("dependency_changed", [])))
    if other:
        questions.append(f"Why do {_join_list(other, 3)} differ under the controlled filter? See the cause in Details.")
    if not NC or NC["status"] in ("skipped", "error"):
        if n_diff and logic:
            questions.append(f"Are the new numbers from the changed logic intended ({_join_list(logic, 3)})? "
                             "This needs business sign-off.")
        elif n_diff:
            questions.append(f"Are the different numbers in {B} expected (refresh timing, data volume or logic)?")
    if NC:
        for k in "AB":
            if NC["only_in"][k]:
                questions.append(f"The report uses {_join_list(NC['only_in'][k], 3)}, which exist{'s' if len(NC['only_in'][k]) == 1 else ''} "
                                 f"only in {LBL[k]}: is that intended?")
    if big or source:
        questions.append(f"Is {B}'s data source and volume part of the release, or only test data?")
    worse = [w for w, v in (("slower response", speed), ("higher capacity use", cap)) if v == "worse"]
    if worse:
        questions.append(f"Is the {' and '.join(worse)} acceptable in return for the changes?")

    steps = []
    if not done:
        steps.append("Run the test again: Run all with confirm_run = True (a scheduled run repeats on its next start). "
                     "The partial results above are kept.")
    if NC and NC["status"] == "differences":
        steps.append("Check the measures that differ in Details for analysts > Number check (values per date and "
                     "business unit, rows in the window, cause).")
        if by_cause.get("rows_differ"):
            steps.append(f"Refresh {B} and run the dry run again (Step 5) to see whether the differences remain.")
        if by_cause.get("logic_changed") or by_cause.get("dependency_changed"):
            steps.append("Confirm the new definitions with the business owner of the numbers.")
    if NC and NC["status"] == "skipped" and NC["reason"] and "set number_check" in NC["reason"]:
        steps.append("Set number_check_bu_column / number_check_date_column in Step 1 and run the dry run again.")
    if done:
        if cap == "unclear":
            steps.append("Run the comparison again to see whether the server CPU difference repeats: it comes from one "
                         "traced pass per model.")
        if levels and max(levels) < 3:
            steps.append("Re-run with 3–5 concurrent users if that load on the models is approved.")
        ja, jb = M["journeys"]["A"], M["journeys"]["B"]
        if len(M["load_ok"]) and (speed in ("unclear", "few") or min(len(ja), len(jb)) < 5
                                  or min(len(M["dur"]["A"]), len(M["dur"]["B"])) < 200):
            steps.append("Add iterations for narrower confidence ranges and more A/B round pairs.")

    notes = []
    if any(k[0].endswith("¹") for k in kpis):
        notes.append(f"¹ Sum of the waits after each click of one full replay of the {M['n_common']} recorded queries "
                     "(first query start to last query end of a click; the queries of a click run in parallel as captured; "
                     "think time is excluded)." if page else
                     f"¹ Sum of the query times of one full replay of the {M['n_common']} recorded queries, run one after "
                     "another. In the report, visuals on a page load in parallel, so the real waiting time is shorter; "
                     "the relative change is what matters.")
    if any(k[0].endswith("²") for k in kpis):
        notes.append("² Measured in a traced single-user pass after the warm-up (one measurement per query and model). "
                     "Server CPU time is the main driver of the capacity units (CU) a query consumes. Its 95% range "
                     "resamples the queries: a wide range means that a few queries carry the difference.")
    howto = [f"Clear = outside ±{float(noise_band_pct):g}%, the 95% range excludes no change, at least "
             f"{ROUND_PAIR_MIN} A/B round pairs with {ROUND_PAIR_AGREE:.0%} agreeing in direction, and larger than "
             f"{A}'s own round-to-round spread. Fewer than {MIN_SAMPLES} measurements = no verdict.",
             "Measured warm: every query ran once before the measurement, as when report pages are viewed repeatedly "
             "between data refreshes. Not measured: the first view right after a data refresh (to measure it, run the "
             "comparison again right after both models refresh)."]
    if t0 is not None:
        window = f"{t0:%Y-%m-%d %H:%M}–{t1:%H:%M} UTC" if t1 is not None else f"from {t0:%Y-%m-%d %H:%M} UTC"
        howto.append(f"Exact CU: Capacity Metrics app, Timepoint page → Interactive operations, User = the account that ran "
                     f"this notebook, {window}; compare Total CU (s) of the two semantic models.")
    return {"title": f"Executive summary: {B} vs {A}" + (f" ({M['test_name']})" if M["meta"] is not None else ""),
            "A": A, "B": B, "context": " · ".join(context), "fresh": fresh, "blockers": blockers, "trust": trust,
            "trend": trend_line(M.get("trend"), LBL) if M["run_mode"] == "scheduled" else "",
            "bottom": sentences, "kpis": kpis, "pros": pros, "cons": cons, "neither": neither,
            "trade": trade, "questions": questions, "steps": steps, "notes": notes, "howto": howto}


def _summary_text(S):
    """The executive summary as plain text (email / Teams)."""
    lines = [S["title"], S["context"]] + ([f"Data: {S['fresh']}"] if S["fresh"] else []) + [""]
    if S["blockers"]:
        lines.append("BLOCKERS (resolve or explain before merging)")
        for head, sub, _ in S["blockers"]:
            lines += [f"- {head}"] + [f"    {s}" for s in sub]
        lines.append("")
    t = S["trust"]
    lines.append(f"Trust: {t['passed']} of {t['total']} checks passed" + (":" if t["failing"] else "."))
    lines += [f"- {txt}" for txt, _ in t["failing"]]
    lines += ["", "Bottom line: " + " ".join(S["bottom"])] + ([f"Trend: {S['trend']}"] if S["trend"] else [])
    lines += ["", f"| What it means for users and capacity | {S['A']} | {S['B']} | Change |", "|---|---|---|---|"]
    lines += [f"| {l_} | {a_} | {b_} | {t_} |" for l_, a_, b_, t_, _ in S["kpis"]]
    lines += ["", f"Pros ({S['B']} better)"] + ([f"- {p}" for p in S["pros"]] or ["- none found"])
    lines += ["", f"Cons ({S['B']} worse)"] + ([f"- {c}" for c in S["cons"]] or ["- none found"])
    lines += [""] + S["neither"] + ["", "Trade-offs and risks"]
    lines += [f"{i}. {t_}: {d_}" for i, (t_, d_) in enumerate(S["trade"], start=1)]
    if S["questions"]:
        lines += ["", "Questions to resolve"] + [f"- {q}" for q in S["questions"]]
    if S["steps"]:
        lines += ["", "Next steps"] + [f"- {s}" for s in S["steps"]]
    if S["notes"]:
        lines += [""] + S["notes"]
    lines += ["", "How to read"] + [f"- {h}" for h in S["howto"]]
    return "\n".join(lines)


def _summary_html(S):
    esc = _html.escape
    li = lambda items: "".join(f"<li style='margin:3px 0'>{x}</li>" for x in items)  # noqa: E731
    kp_rows = [[esc(l_), esc(a_), esc(b_), f"<span style='color:{CLASS_COLOR[v_]};font-weight:600'>{esc(t_)}</span>"]
               for l_, a_, b_, t_, v_ in S["kpis"]]
    box = "border:1px solid #ddd;border-radius:6px;padding:8px 12px;flex:1"
    blockers = ""
    if S["blockers"]:
        blockers = ("<div style='background:#ffebee;color:#b71c1c;border-left:5px solid #c62828;padding:8px 12px;"
                    "margin-bottom:8px'><b>Blockers: resolve or explain before merging</b><ul style='margin:4px 0;"
                    "padding-left:20px'>"
                    + li(esc(h) + (f"<ul style='margin:2px 0;padding-left:18px;color:#333'>{li(esc(s) for s in sub)}</ul>"
                                   if sub and not hide else "") for h, sub, hide in S["blockers"]) + "</ul></div>")
    t = S["trust"]
    ok_all = not t["failing"]
    trust = (f"<div style='background:{'#f1f8e9' if ok_all else '#fff8e1'};border-left:5px solid {GREEN if ok_all else AMBER};"
             f"padding:6px 10px;margin-bottom:8px'><b>Trust: {t['passed']} of {t['total']} checks passed</b>"
             + (f"<ul style='margin:2px 0;padding-left:20px'>"
                + li(f"<span style='color:{RED if strong else '#7a4f00'}'>{esc(txt)}</span>" for txt, strong in t["failing"])
                + "</ul>" if t["failing"] else "") + "</div>")
    return (
        f"<div style='{_FONT};border:2px solid #333;border-radius:8px;padding:14px 18px;margin:12px 0;max-width:1150px'>"
        f"<div style='font-size:19px;font-weight:700'>Executive summary: <span style='color:{COLOR['B']}'>{esc(S['B'])}</span>"
        f" vs <span style='color:{COLOR['A']}'>{esc(S['A'])}</span></div>"
        f"<div style='color:#555;margin:3px 0 2px 0'>{esc(S['context'])}</div>"
        + (f"<div style='color:#555;margin:0 0 10px 0'>Data: {esc(S['fresh'])}</div>" if S["fresh"] else "<div style='margin-bottom:8px'></div>")
        + blockers + trust
        + f"<div style='background:#eef4fb;border-left:5px solid {COLOR['A']};padding:10px 12px;font-size:15px;"
          f"margin-bottom:10px'><b>Bottom line:</b> {esc(' '.join(S['bottom']))}</div>"
        + (f"<div style='border-left:5px solid {COLOR['A']};padding:4px 12px;margin:-6px 0 10px 0'><b>Trend:</b> "
           f"{esc(S['trend'])}</div>" if S["trend"] else "")
        + _html_table(["What it means for users and capacity", f"<span style='color:{COLOR['A']}'>{esc(S['A'])}</span>",
                       f"<span style='color:{COLOR['B']}'>{esc(S['B'])}</span>", "Change"], kp_rows)
        + "<div style='display:flex;gap:18px;margin-top:12px'>"
          f"<div style='{box}'><b style='color:{GREEN}'>Pros ({esc(S['B'])} better)</b>"
          f"<ul style='margin:4px 0;padding-left:20px'>{li(esc(p) for p in S['pros']) or li(['none found'])}</ul></div>"
          f"<div style='{box}'><b style='color:{RED}'>Cons ({esc(S['B'])} worse)</b>"
          f"<ul style='margin:4px 0;padding-left:20px'>{li(esc(c) for c in S['cons']) or li(['none found'])}</ul></div></div>"
        + "".join(f"<div style='color:#333;margin-top:6px'>{esc(s)}</div>" for s in S["neither"])
        + "<div style='display:flex;gap:18px;margin-top:12px'>"
          f"<div style='{box}'><b>Trade-offs and risks</b>"
          f"<ol style='margin:4px 0;padding-left:20px'>{li(f'<b>{esc(t_)}:</b> {esc(d_)}' for t_, d_ in S['trade'])}</ol></div>"
          f"<div style='{box}'>"
        + (f"<b>Questions to resolve</b><ul style='margin:4px 0;padding-left:20px'>{li(esc(q) for q in S['questions'])}</ul>"
           if S["questions"] else "")
        + (f"<b>Next steps</b><ul style='margin:4px 0;padding-left:20px'>{li(esc(s) for s in S['steps'])}</ul>"
           if S["steps"] else "") + "</div></div>"
        + "".join(f"<div style='color:#777;font-size:11px;margin-top:4px'>{esc(n_)}</div>" for n_ in S["notes"])
        + "<details style='margin-top:8px'><summary style='cursor:pointer;color:#1f77b4'>How to read (what counts as clear, "
          "what was measured, exact CU)</summary>"
        + "".join(f"<div style='color:#555;font-size:12px;margin:4px 0'>{esc(h)}</div>" for h in S["howto"]) + "</details>"
        + "<details style='margin-top:8px'><summary style='cursor:pointer;color:#1f77b4'>Copy as text (for email or "
          f"Teams)</summary><pre style='white-space:pre-wrap;font-size:12px;background:#f7f7f7;padding:8px'>{esc(S['text'])}"
          "</pre></details></div>")


def data_quality_notes(M):
    """(text, strong) facts that limit how far the numbers can be trusted (the long form of the trust line)."""
    LBL, notes = M["labels"], []
    lo = M["load_ok"]
    for c in sorted(lo["concurrency"].unique()) if len(lo) else []:
        n_min = min(int(((lo["concurrency"] == c) & (lo["model_key"] == k)).sum()) for k in "AB")
        if n_min < SAMPLES_MIN_LEVEL:
            notes.append((f"Level {c}: only {n_min} successful samples for one model; treat percentiles as indicative.", False))
    if M["planned_q"]:
        pq, eq = M["planned_q"], M["executed_q"]
        if any(eq[k] < pq[k] for k in "AB"):
            notes.append((f"Coverage: {sum(eq.values()):,} of {sum(pq.values()):,} planned load queries were executed "
                          f"({LBL['A']} {eq['A']:,} of {pq['A']:,}, {LBL['B']} {eq['B']:,} of {pq['B']:,}; rounds "
                          f"{LBL['A']} {M['rounds_run']['A']}, {LBL['B']} {M['rounds_run']['B']} of "
                          f"{M['rounds_planned_per_model']} per model).", True))
        if M["rounds_run"]["A"] != M["rounds_run"]["B"]:
            notes.append(("The A,B / B,A alternation is incomplete (different number of rounds per model), so drift "
                          "between rounds is not cancelled.", True))
    for k, (s, c, n) in M["round_drift"].items():
        if s > ROUND_DRIFT_PCT:
            notes.append((f"Round drift: {LBL[k]}'s {n} rounds at {c} user(s) differ by {s:.0f}% in median latency.", False))
    for k, (matched, total) in M["trace"].items():
        if total - matched:
            notes.append((f"Server trace for {LBL[k]}: {total - matched} of {total} queries without server events "
                          "(best-effort trace).", False))
    for k, v in M["first_touch"].items():
        if not _isnan(v) and v > COLD_MS:
            notes.append((f"{LBL[k]} was cold at the start (first touch {fmt_ms(v)}). The warm-up absorbed this; load "
                          "statistics exclude the warm-up.", False))
    if M["status"] != "completed":
        notes.append((f"Run status is '{M['status']}': {M['message']}", False))
    rb = M["refresh"]["by_model"]
    if any(rb[k]["before"] or rb[k]["after"] for k in "AB"):
        notes.append(("Last refresh / Direct Lake framing before the test (UTC): "
                      + "; ".join(f"{LBL[k]} {(rb[k]['before'] or 'unknown')[:16].replace('T', ' ')}" for k in "AB")
                      + ". A refresh clears a model's caches and unloads Direct Lake columns, so a recently refreshed "
                        "model starts colder.", False))
        for k in M["refresh"]["refreshed"]:
            notes.append((f"{LBL[k]} was refreshed / reframed during the test (at {rb[k]['after']} UTC): its queries after "
                          "that moment ran with cleared caches. For a clean comparison, re-run at a time without refreshes.", True))
    if M["capacity"] is False:
        notes.append(("The workspaces ran on DIFFERENT capacities: speed and cost differences may come from the capacity, "
                      "not from the model.", True))
    elif M["capacity"] is None:
        notes.append(("Capacity could not be verified; a fair comparison needs both workspaces on the same capacity. "
                      "Rounds alternated A,B / B,A to cancel drift.", False))
    return notes


def render_report(M):
    """The report: header, executive summary, the wait-per-click chart, and one collapsed section with the details.
    Everything goes through emit(), so the saved report file shows exactly the same."""
    LBL, params, NC = M["labels"], M["params"], M["number_check"]
    load_ok, sm = M["load_ok"], M["sm"]
    esc = _html.escape
    page = M["replay_mode"] != "sequential"
    excluded_note = (f", {_plural(len(M['failing']), 'failing query', 'failing queries')} excluded" if M["failing"] else "")

    # ---------- 1. header ----------
    meta, ex = M["meta"], M["ex"]
    model_rows = [[f"<b style='color:{COLOR[k]}'>{k} = {esc(LBL[k])}</b>",
                   esc(str(meta[f'{k.lower()}_workspace'])) if meta is not None else "",
                   esc(str(meta[f'{k.lower()}_model'])) if meta is not None else ""] for k in "AB"]
    if page:
        par = [int(v) for v in params.get("click_parallelism", {}).values()]
        replay = (f"page-load replay: the queries of a click run in parallel as captured (up to {max(par) if par else '?'} at "
                  f"a time), think time {params.get('think_time_between_actions_sec', '')}s between clicks")
    else:
        replay = (f"sequential replay: think time {params.get('think_time_between_queries_sec', '')}s between queries, "
                  f"{params.get('think_time_between_actions_sec', '')}s between actions")
    emit(f"<div style='{_FONT}'><h2 style='margin:4px 0'>{esc(str(M['test_name']))}</h2>"
         f"<div style='color:#555'>comparison_id <code>{esc(M['cid'])}</code> | status <b>{esc(M['status'])}</b> | "
         f"{esc(M['run_mode'] or 'interactive')} run | approval fingerprint <code>{esc(M['fingerprint'] or '-')}</code> | "
         f"started {fmt_ts(meta['started_utc']) if meta is not None else '-'} | "
         f"finished {fmt_ts(meta['finished_utc']) if meta is not None else '-'}</div><br>"
         + _html_table(["Model", "Workspace", "Semantic model"], model_rows)
         + f"<div style='color:#555;margin-top:6px'>Query file <code>{esc(str(params.get('query_file', '')))}</code> | "
           f"{_plural(M['n_q'], 'query', 'queries')} replayed per virtual user | concurrency levels "
           f"{esc(str(params.get('concurrency_levels', '')))} | iterations {params.get('iterations', '')} | {esc(replay)} | "
           f"rounds alternate A,B / B,A | executions: load {len(M['load_all']):,}, warm-up {int((ex['phase'] == 'warmup').sum()):,}, "
           f"trace {int((ex['phase'] == 'trace').sum()):,}</div></div>")

    # ---------- 2. executive summary (+ the number check when it found differences) ----------
    try:
        S = summary_sections(M)
        S["text"] = _summary_text(S)
        globals()["EXEC_SUMMARY_TEXT"] = S["text"]
        emit(_summary_html(S))
        if NC and NC["status"] == "differences":
            emit(f"<div style='{_FONT};max-width:1150px;border:2px solid {RED};border-radius:8px;padding:8px 14px'>"
                 f"<b>Number check (controlled filter)</b>" + numcheck_html(NC, LBL, compact=True) + "</div>")
    except Exception as exc:
        print(f"(executive summary skipped: {str(exc)[:300]})")

    # ---------- 3. details for analysts (collapsed; built here, shown at the end) ----------
    details, toc = [], []
    h4 = f"<h4 style='{_FONT};font-size:15px;margin:14px 0 4px 0'>"

    def detail(name, html_, short=None):
        if html_:
            toc.append(short or name)
            details.append(f"{h4}{esc(name)}</h4>{html_}")

    detail("Number check (controlled filter)", numcheck_html(NC, LBL), "number check")

    rows, colors = [], {}

    def add(metric, k, fmt=fmt_ms, words=("faster", "slower"), delta_fmt=None, reading=None):
        a, b, ci = k["a"], k["b"], k["ci"]
        if reading is None and fmt is fmt_ms and not _isnan(a) and not _isnan(b) and abs(b - a) < 1.0:
            reading = ("≈ no meaningful change (< 1 ms)", GREY)
        text, color = reading if reading else _reading(k, LBL["B"], words)
        delta = "-" if (_isnan(a) or _isnan(b)) else (delta_fmt(b - a) if delta_fmt else
                                                     ("+" if b - a >= 0 else "-") + fmt(abs(b - a)))
        c = k.get("cons")
        pairs = (f"{_pairs_short(k, words)}" + (f"; A spread {c['drift']:.0f}%" if c and not _isnan(c["drift"]) else "")) if c else "-"
        i = len(rows)
        rows.append([metric, fmt(a), fmt(b), delta, _fmt_rel(k["rel"]), _fmt_ci(ci), pairs or "-", text])
        colors[(i, 4)] = color
        colors[(i, 7)] = color

    K = M["kpis"]
    if len(load_ok):
        for name, label in (("p50", "median"), ("p90", "p90"), ("p95", "p95")):
            add(f"Query latency under load - {label}, all levels pooled (n = {K[name]['n_a']:,} / {K[name]['n_b']:,})", K[name])
        add("Query latency under load - max (single run)", K["max"])
        what = "waits after each click" if page else "queries"
        add(f"Click-path wait: sum of the {what} of one full replay of {M['n_common']} queries, median "
            f"(n = {K['journey']['n_a']} / {K['journey']['n_b']}{excluded_note})", K["journey"])
    if "err_pct" in K:
        er = {"A": K["err_pct"]["a"], "B": K["err_pct"]["b"]}
        if er["A"] == 0 and er["B"] == 0:
            reading = ("≈ no errors in either model", GREY)
        elif abs(er["B"] - er["A"]) < 0.1:
            reading = ("≈ same error rate", GREY)
        else:
            worse = er["B"] > er["A"]
            arrow = "▲" if worse else "▼"
            reading = (f"{arrow} {LBL['B']} {'more' if worse else 'fewer'} errors", RED if worse else GREEN)
        add("Error rate under load", K["err_pct"], fmt=lambda v: f"{v:.2f}%", delta_fmt=lambda d: f"{d:+.2f} pp", reading=reading)
    if "first_touch" in K:
        add("Cold first touch: connect + first query (single measurement)", K["first_touch"])
    if "warmup" in K:
        add(f"First run of all queries, single user (warm-up, {_plural(K['warmup']['n'], 'query', 'queries')})", K["warmup"])
    if "cpu" in K:
        add(f"Server CPU time, all queries, single user ({_plural(K['cpu']['n'], 'query', 'queries')})", K["cpu"],
            words=("cheaper", "costlier"))
        if "server_dur" in K:
            add("Server duration, all queries, single user", K["server_dur"])
        if "se_queries" in K:
            add("Storage-engine queries, single user", K["se_queries"], fmt=lambda v: "-" if _isnan(v) else f"{v:,.0f}",
                words=("fewer", "more"), delta_fmt=lambda d: f"{d:+,.0f}")
    detail(f"At a glance: {LBL['B']} compared with {LBL['A']}",
           _html_table(["Metric", esc(LBL["A"]), esc(LBL["B"]), "Δ", "Δ %", "95% CI of Δ %", "A/B round pairs", "Reading"],
                       rows, colors)
           + f"<div style='{_FONT};color:#555;margin-top:4px'>Green/red only shows the direction of a clear change; it is "
             f"not a merge verdict. Changes inside ±{noise_band_pct}% are grey (no meaningful change). A change is "
             f"<span style='color:{AMBER};font-weight:600'>amber</span> (not clear) when its 95% confidence interval "
             f"crosses 0, when fewer than {ROUND_PAIR_AGREE:.0%} of at least {ROUND_PAIR_MIN} A/B round pairs agree, or "
             f"when it is smaller than {esc(LBL['A'])}'s own round-to-round spread. Single measurements stay grey.</div>",
           "scorecard with confidence intervals")

    if M["by_level"]:
        lv_rows, lv_colors = [], {}
        for lv in M["by_level"]:
            i = len(lv_rows)
            p50, p90_ = lv["p50"], lv["p90"]
            lv_rows.append([f"{_plural(lv['level'], 'user')}", f"{lv['n_a']:,} / {lv['n_b']:,}", fmt_ms(p50["a"]), fmt_ms(p50["b"]),
                            f"{_fmt_rel(p50['rel'])} {_fmt_ci(p50['ci'])}", fmt_ms(p90_["a"]), fmt_ms(p90_["b"]),
                            f"{_fmt_rel(p90_['rel'])} {_fmt_ci(p90_['ci'])}", _pairs_short(p90_) or "-",
                            f"{lv['errors']['A']} / {lv['errors']['B']}"])
            lv_colors[(i, 4)], lv_colors[(i, 7)] = CLASS_COLOR[p50["cls"]], CLASS_COLOR[p90_["cls"]]
        detail("By concurrency level", _html_table(
            ["Level", "n (A / B)", f"median {esc(LBL['A'])}", f"median {esc(LBL['B'])}", "Δ median [CI]",
             f"p90 {esc(LBL['A'])}", f"p90 {esc(LBL['B'])}", "Δ p90 [CI]", "p90 round pairs", "errors (A / B)"],
            lv_rows, lv_colors), "per concurrency level")

    # ---------- 4. charts: wait per click in the main view, server CPU in the details ----------
    plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})

    def chart_actions():
        acts = M["actions"]
        names = []
        for a in acts:
            extra = f", {a['parallel']} at a time" if page and a.get("parallel") and int(a["parallel"]) > 1 else ""
            names.append(f"{a['action_seq']}. {a['label']} ({_plural(a['n_queries'], 'query', 'queries')}{extra})")
        fig, ax = plt.subplots(figsize=(10, 0.55 * len(acts) + 1.6))
        y, h = np.arange(len(acts)), 0.38
        unit, div = _unit(M["clicks"]["wait_ms"])
        medv = {k: [a["a" if k == "A" else "b"] / div for a in acts] for k in "AB"}
        ax.barh(y - h / 2, medv["A"], height=h, color=COLOR["A"], label=LBL["A"])
        ax.barh(y + h / 2, medv["B"], height=h, color=COLOR["B"], label=LBL["B"])
        for i, a in enumerate(acts):
            ax.text(max(medv["A"][i], medv["B"][i]) * 1.01, i, f" {_fmt_rel(a['rel'])}", va="center", fontsize=9,
                    fontweight="bold", color=CLASS_COLOR[a["cls"]])
        ax.set_yticks(y)
        ax.set_yticklabels(names)
        ax.invert_yaxis()
        ax.set_xlabel(f"Typical wait after the click ({unit}): " + ("first query start to last query end"
                      if page else "the click's queries added up") + ", middle value of all runs")
        ax.set_title("How long a user waits after each click (lower is better; coloured = clear change)"
                     + (f" ({excluded_note.lstrip(', ')})" if excluded_note else ""))
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3, axis="x")
        plt.tight_layout()

    def chart_cpu():
        p = pivot_by_model(sm, "cpu_ms").dropna()
        if p.empty:
            return ("No server CPU data (server trace not run or not permitted)." if sm.empty or "cpu_ms" not in sm.columns
                    else "No server CPU data matched for both models.")
        x, y = p["A"].clip(lower=1), p["B"].clip(lower=1)
        rel = (p["B"] - p["A"]) / p["A"].replace(0, np.nan) * 100
        fig, ax = plt.subplots(figsize=(7.5, 6.5))
        ax.scatter(x, y, c=[_rel_color(r) for r in rel], s=38, alpha=0.8, edgecolor="white")
        lim = [min(x.min(), y.min()) * 0.8, max(x.max(), y.max()) * 1.25]
        ax.plot(lim, lim, "--", color="#333", lw=1)
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlim(lim)
        ax.set_ylim(lim)
        plain = matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:,.0f}" if v >= 1 else f"{v:g}")
        for axis in (ax.xaxis, ax.yaxis):
            axis.set_major_locator(matplotlib.ticker.LogLocator(base=10, subs=(1.0, 2.0, 5.0)))
            axis.set_major_formatter(plain)
            axis.set_minor_formatter(matplotlib.ticker.NullFormatter())
        top5 = list((p["B"] - p["A"]).abs().sort_values(ascending=False).index[:5])
        for i, s in enumerate(top5):
            ax.annotate(f"#{s}", (x[s], y[s]), fontsize=8, fontweight="bold", xytext=(14, -6 - 12 * i),
                        textcoords="offset points", arrowprops={"arrowstyle": "-", "color": "#999", "lw": 0.6})
        qinfo = M["qinfo"]
        caption = ("Largest server-CPU changes: " + "; ".join(
            f"{query_label(qinfo[s], 60) if s in qinfo else '#' + str(s)}: {p.loc[s, 'A']:,.0f} -> {p.loc[s, 'B']:,.0f} ms"
            for s in top5)) if top5 else None
        ax.set_xlabel(f"{LBL['A']}: server CPU per query (ms, log scale)")
        ax.set_ylabel(f"{LBL['B']}: server CPU per query (ms, log scale)")
        ax.set_title(f"Server CPU per query (single user, one run each)\nabove the line = {LBL['B']} costs more capacity")
        ax.grid(alpha=0.3)
        plt.tight_layout()
        return caption

    if len(load_ok) and M["actions"]:
        emit(_chart_html("per action", chart_actions))
    detail("Server CPU per query",
           f"<div style='{_FONT};color:#333;margin:2px 0 6px 0;max-width:1100px'>How to read: each dot is one query, "
           f"measured once per model with a single user, so a single dot can be noise. Server CPU is the work the model "
           f"does, and it drives the capacity (CU) the report uses. On the dashed line both models use the same CPU; "
           f"<b>above</b> the line {esc(LBL['B'])} uses more, <b>below</b> it uses less. Both axes use a log scale "
           "(1, 10, 100, 1,000 ms). The 5 largest changes are numbered (#) and listed under the chart.</div>"
           + _chart_html("server CPU", chart_cpu), "server CPU per query")

    # ---------- 5. a plain per-visual table ----------
    DETAIL = M["per_query"]
    globals()["REPORT_DETAIL"] = DETAIL
    if len(DETAIL) and len(load_ok):
        n_runs = float(np.median([r for r in DETAIL["runs"] if r])) if (DETAIL["runs"] > 0).any() else 0
        mA, mB = f"median {LBL['A']}", f"median {LBL['B']}"
        same_txt = {"identical": "yes", "values differ": "no, different values", "rows differ": "no, different number of rows",
                    "error": "could not compare (error)", "same rows": "same number of rows", "n/a": "not checked"}
        uses = set(M["attrib"]["with"]) if M["attrib"] else set()
        order = DETAIL.assign(_d=(DETAIL[mB] - DETAIL[mA]).abs()).sort_values("_d", ascending=False, na_position="last")
        simple, s_colors = [], {}
        for _, r in order.iterrows():
            txt, col = r["change"], CLASS_COLOR[r["change_cls"]]
            for k in "AB":
                n_fail = r[f"errors {LBL[k]}"]
                if not _isnan(n_fail) and n_fail > 0:
                    txt += f"; failed {int(n_fail)}× in {LBL[k]}"
                    col = RED if k == "B" else col
            visual = str(r["visual"]) + (f" ({r['type']})" if r["type"] and r["type"] != r["visual"] else "")
            res = same_txt.get(r["result"], str(r["result"]))
            if M["attrib"] and r["result"] in ("values differ", "rows differ"):
                res += "; uses a measure that differs" if int(r["#"]) in uses else "; uses only matching measures"
            s_colors[(len(simple), 5)] = col
            simple.append([str(int(r["#"])), esc(str(r["action"])), esc(visual), _fmt_short(r[mA]), _fmt_short(r[mB]),
                           esc(txt), esc(res)])
        detail(
            "Each visual: how long a user waits",
            f"<div style='{_FONT};color:#333;margin:2px 0 6px 0;max-width:1100px'>How to read: during the load test, each "
            f"visual's query ran about {n_runs:.0f} times in each model. <b>Typical wait</b> is the middle of those runs: "
            "half of the runs were faster, half were slower. <b>No clear difference</b> means the gap is small compared "
            "with how much the runs varied, or the A/B round pairs disagree, so it may be chance. <b>Same numbers?</b> says "
            "whether both models returned the same result for the whole query (Step 4) and whether the query uses a "
            "measure that differs in the number check. The biggest changes are at the top.</div>"
            f"<div style='max-height:600px;overflow:auto'>"
            + _html_table(["#", "Click (user action)", "Visual", f"{esc(LBL['A'])}: typical wait",
                           f"{esc(LBL['B'])}: typical wait", "Change", "Same numbers?"], simple, s_colors)
            + "</div>", "each visual's wait")

    if len(M["err_table"]):
        n_err = int(M["err_table"]["count"].sum())
        detail(f"Errors ({n_err})", f"<div style='{_FONT}'>{M['err_table'].to_html(index=False)}</div>", f"{n_err} errors")

    notes = data_quality_notes(M)
    if notes:
        detail(f"Data-quality notes ({len(notes)})", f"<ul style='{_FONT}'>"
               + "".join(f"<li>{'<b>' if strong else ''}{esc(t)}{'</b>' if strong else ''}</li>" for t, strong in notes) + "</ul>",
               f"{len(notes)} data-quality notes")

    # ---------- 6. one collapsed section: everything an analyst may want, out of the reader's way ----------
    if details:
        emit(f"<details style='{_FONT};margin-top:14px;border:1px solid #bbb;border-radius:6px;padding:8px 14px;max-width:1150px'>"
             f"<summary style='cursor:pointer;font-size:15px;font-weight:600;color:#1f77b4'>Details for analysts "
             f"<span style='font-weight:normal;color:#555'>(click to open: {esc(', '.join(toc))})</span></summary>"
             + "".join(details) + "</details>")


def approval_block():
    """The end of a dry run: everything an approver needs in one place, then what to do."""
    NCd = globals().get("NUMCHECK")
    lines = [
        "APPROVAL SUMMARY (dry run: nothing was loaded or saved)",
        f"  Models      : A = {a_label} ({a_workspace} / {a_model}); B = {b_label} ({b_workspace} / {b_model})",
        f"  Load        : {ROUNDS_PLANNED} rounds = levels {CONCURRENCY} x {iterations} iteration(s) x 2 models, alternating "
        f"A,B / B,A; {sum(CONCURRENCY) * int(iterations) * 2 * len(QUERIES) if run_load_test else 0:,} query executions",
        f"  Peak        : {PEAK_QUERIES} concurrent queries per model ({max(CONCURRENCY)} users x {MAX_PARALLEL} parallel; "
        f"guard {max_concurrent_queries_guard})",
        f"  Duration    : ~ {fmt_hms(EST_SECONDS)} (estimate)",
        "  Numbers     : " + (numcheck_line(NCd, {"A": a_label, "B": b_label}) if NCd else
                              "number check not run" if not run_number_check else "number check did not complete (see Step 5)"),
        f"  Fingerprint : {APPROVAL_FINGERPRINT} (these settings + the query file)",
        "",
        "Next:",
        "  - To run the test now: set confirm_run = True in Step 1 and Run all.",
        f'  - To run it on a schedule: set schedule_approval = "{APPROVAL_FINGERPRINT}" in Step 1, save the notebook and '
        "create the schedule in the Fabric UI (README: Scheduled runs). Any change to these settings or the query file "
        "needs a new dry run and approval.",
    ]
    print("\n".join(lines))


REPORT_CID = _pick_comparison_id()
REPORT_METRICS = None
if REPORT_CID is None:
    if globals().get("APPROVAL_FINGERPRINT") and not globals().get("RUN_ENABLED"):
        approval_block()
    else:
        print("Dry run only (confirm_run = False): nothing was measured, so there is no report (the number check is in Step 5).")
    print("\n- To open a saved test: set report_comparison_id = '<comparison_id>' (Step 9 lists them) and run Steps 1, 2 and 8.")
else:
    print(f"Reporting comparison_id: {REPORT_CID}")
    REPORT_META, REPORT_DATA = load_comparison(REPORT_CID)
    _own = REPORT_CID == globals().get("COMPARISON_ID") and bool(globals().get("RUN_ENABLED"))
    if REPORT_DATA["lt_executions"].empty:
        if REPORT_META is not None and REPORT_META["status"] == "skipped":
            print(f"This scheduled run was skipped: {REPORT_META['message']}")
        else:
            print(f"No query executions stored for {REPORT_CID} (preflight only, or the run failed before the warm-up in Step 4).")
        _nc = numcheck_from_profile(REPORT_DATA["lt_model_profile"])
        if _nc is not None:
            display(HTML(f"<div style='{_FONT};max-width:1150px'>" + numcheck_html(_nc, {
                k: str(REPORT_META[f"{k.lower()}_label"]) if REPORT_META is not None else k for k in "AB"}) + "</div>"))
    else:
        REPORT_METRICS = compute_metrics(REPORT_CID, REPORT_META, REPORT_DATA)
        if _own and save_report_files:
            REPORT_FILE = f"{REPORTS_DIR}/{REPORT_CID}.html"
        if _own:   # one summary row per comparison, written by the run that made it (trend in Step 9)
            try:
                write_table(pd.DataFrame([summary_row(REPORT_METRICS)]), "lt_run_summary")
            except Exception as _exc:
                print(f"  (could not save the run summary: {_err_text(_exc)[:200]})")
        try:   # the trend (read-only): one line in the main view of a scheduled run, and in the result JSON
            _H = history_frame()
            _key = _H.loc[_H["comparison_id"] == REPORT_CID, "key"]
            REPORT_METRICS["trend"] = trend_view(_H, _key.iloc[0]) if len(_key) else None
        except Exception as _exc:
            print(f"  (trend skipped: {_err_text(_exc)[:200]})")
        render_report(REPORT_METRICS)
        if REPORT_FILE:
            try:
                write_lakehouse_file(REPORT_FILE, "<!doctype html><html><head><meta charset='utf-8'><title>"
                                     + _html.escape(f"{REPORT_METRICS['test_name']} - {REPORT_CID}") + "</title></head>"
                                     f"<body style='{_FONT}'>" + "".join(REPORT_HTML) + "</body></html>")
                print(f"Report saved: {lakehouse_name}/{REPORT_FILE}")
            except Exception as _exc:
                REPORT_FILE = None
                print(f"  (could not save the report file: {_err_text(_exc)[:200]})")

## Step 9 - History
One row per saved comparison, newest first; then the trend of the runs with the same approval fingerprint.

In [ ]:
# Step 9 - History
banner("STEP 9 - HISTORY")


def _collapse_skipped(H):
    """Consecutive skipped runs (newest first) become one row 'skipped xN' with the latest reason."""
    rows, run = [], []
    for r in H.to_dict("records") + [None]:
        if r is not None and r["status"] == "skipped":
            run.append(r)
            continue
        if run:
            first = dict(run[0])
            if len(run) > 1:
                first["status"] = f"skipped ×{len(run)}"
                first["message"] = f"{run[0]['message']} (oldest of these: {fmt_ts(run[-1]['started'])})"
            rows.append(first)
            run = []
        if r is not None:
            rows.append(r)
    return pd.DataFrame(rows)


def _hist_style(df, pct_cols, grey_rows=()):
    sty = df.style.format({c: _fmt_rel for c in pct_cols})
    smap = sty.map if hasattr(sty, "map") else sty.applymap
    sty = smap(lambda v: "" if _isnan(v) else ("color:#757575" if abs(v) < float(noise_band_pct)
                                               else ("color:#1b5e20;font-weight:600" if v < 0 else "color:#b71c1c;font-weight:600")),
               subset=pct_cols)
    if "numbers" in df.columns:
        sty = smap(lambda v: "color:#b71c1c;font-weight:600" if str(v).endswith("differ") else "", subset=["numbers"])
    if len(grey_rows):
        sty = sty.apply(lambda r: ["color:#9e9e9e" if r.name in grey_rows else "" for _ in r], axis=1)
    return sty


def chart_trend(TV, LBL):
    T = TV["rows"]
    fig, ax = plt.subplots(figsize=(9, 3.6))
    x = np.arange(len(T))
    comp = T["comparable"].astype(bool).to_numpy()
    for col, k in (("p90_a", "A"), ("p90_b", "B")):
        y = pd.to_numeric(T[col], errors="coerce").to_numpy(dtype=float)
        ax.plot(x, y, color=COLOR[k], lw=1.2, alpha=0.6, label=f"{LBL[k]} p90")
        ax.scatter(x[comp], y[comp], color=COLOR[k], s=36, zorder=3)
        ax.scatter(x[~comp], y[~comp], facecolors="white", edgecolors=COLOR[k], s=36, zorder=3)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{pd.Timestamp(t):%m-%d %H:%M}" for t in T["started"]], rotation=30, ha="right", fontsize=8)
    ax.set_ylabel("p90 query latency under load (ms)")
    ax.set_title("p90 per run, same approval fingerprint (hollow = not comparable)")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()


HISTORY_ALL = history_frame()
if HISTORY_ALL.empty:
    print("No comparisons stored yet.")
else:
    _H = _collapse_skipped(HISTORY_ALL)
    _note = [("" if s == "completed" else str(m)[:120]) + ("" if c or str(s).startswith("skipped") else
                                                              (" | not comparable: " + ", ".join(nc) if nc else ""))
             for s, m, c, nc in zip(_H["status"], _H["message"], _H["comparable"], _H["not_comparable"])]
    HISTORY = pd.DataFrame({
        "started": _H["started"].map(fmt_ts), "status": _H["status"], "run": _H["run"], "fingerprint": _H["fingerprint"],
        "comparison_id": _H["comparison_id"], "query file": _H["query file"], "A": _H["A"], "B": _H["B"],
        "levels × iterations": [f"{lv} × {it}" if lv else "" for lv, it in zip(_H["levels"], _H["iterations"])],
        "numbers": _H["numbers"],
        "p90 A": _H["p90_a"].map(fmt_ms) if "p90_a" in _H else "-", "p90 B": _H["p90_b"].map(fmt_ms) if "p90_b" in _H else "-",
        "Δ% p90": [rel_delta(a, b) for a, b in zip(_H.get("p90_a", []), _H.get("p90_b", []))],
        "Δ% median": [rel_delta(a, b) for a, b in zip(_H.get("p50_a", []), _H.get("p50_b", []))],
        "Δ% server CPU": [rel_delta(a, b) for a, b in zip(_H.get("cpu_a", []), _H.get("cpu_b", []))],
        "note": _note,
    })
    try:
        display(_hist_style(HISTORY, ["Δ% p90", "Δ% median", "Δ% server CPU"]))
    except Exception:
        display(HISTORY)
    print("One row per saved comparison, newest first (consecutive skipped scheduled runs are one row). Compare rows only "
          "when they have the same fingerprint (same models, settings and query file). 'numbers' is the Step 5 number "
          "check (- = saved before it existed).")
    print("To open one comparison: set report_comparison_id = '<comparison_id>' in Step 1 (Parameters), "
          "then run Steps 1, 2 and 8.")

    # ---- trend: the runs with the same fingerprint as the report (or the latest run) ----
    _runs = HISTORY_ALL[HISTORY_ALL["status"] != "skipped"]
    _base = globals().get("REPORT_CID")
    _base = _base if _base in set(_runs["comparison_id"]) else (_runs["comparison_id"].iloc[0] if len(_runs) else None)
    TREND = trend_view(HISTORY_ALL, _runs.loc[_runs["comparison_id"] == _base, "key"].iloc[0]) if _base else None
    if TREND is not None and len(TREND["rows"]) >= 2:
        _T = TREND["rows"]
        _lbl = {"A": str(_T.iloc[-1]["A"]).split(":")[0], "B": str(_T.iloc[-1]["B"]).split(":")[0]}
        print(f"\nTrend: {_plural(len(_T), 'run')} with fingerprint {TREND['key']} (oldest first; grey = not comparable, "
              f"left out of the range).")
        _TT = pd.DataFrame({
            "started": _T["started"].map(fmt_ts), "status": _T["status"], "run": _T["run"],
            "comparable": ["yes" if c else "no: " + ", ".join(nc or ["unknown"]) for c, nc in zip(_T["comparable"], _T["not_comparable"])],
            f"p90 {_lbl['A']}": _T["p90_a"].map(fmt_ms), f"p90 {_lbl['B']}": _T["p90_b"].map(fmt_ms),
            "Δ% p90": [rel_delta(a, b) for a, b in zip(_T["p90_a"], _T["p90_b"])],
            "Δ% server CPU": [rel_delta(a, b) for a, b in zip(_T["cpu_a"], _T["cpu_b"])],
            "source": _T["source"], "comparison_id": _T["comparison_id"]})
        try:
            display(_hist_style(_TT, ["Δ% p90", "Δ% server CPU"], [i for i, c in enumerate(_T["comparable"]) if not c]))
        except Exception:
            display(_TT)
        print("Trend: " + trend_line(TREND, _lbl))
        display(HTML(_chart_html("trend", lambda: chart_trend(TREND, _lbl))))
    elif TREND is not None:
        print(f"\nTrend: only 1 run with fingerprint {TREND['key']} so far.")

## Step 10 - Text summary
The executive summary and the exact numbers as plain text. Copy the whole output to share it (chat, ticket, email).

In [ ]:
# Step 10 - Text summary (for sharing)
# Plain-text version of the report for sharing the exact numbers (chat, ticket, email). Copy the whole output.
import contextlib  # noqa: E402


def _text_summary(M):
    A, B, LBL, P, K = M["labels"]["A"], M["labels"]["B"], M["labels"], M["params"], M["kpis"]
    ex, prof, meta = M["ex"], M["prof"], M["meta"]
    pd.set_option("display.width", 250)
    pd.set_option("display.max_colwidth", 45)

    def _h(title):
        print(f"\n=== {title} ===")

    S = summary_sections(M)
    print(_summary_text(S))
    _h("Number check (controlled filter)")
    print("\n".join(numcheck_text(M["number_check"], LBL)))

    _h("Run")
    if meta is not None:
        print(f"comparison_id: {M['cid']}\nstatus: {M['status']} | {M['message']}")
        print(f"run mode: {M['run_mode'] or 'interactive'} | approval fingerprint: {M['fingerprint'] or '-'}")
        print(f"A = {A}: {meta['a_workspace']} / {meta['a_model']}\nB = {B}: {meta['b_workspace']} / {meta['b_model']}")
    print(f"replay: {M['replay_mode']} | params: " + ", ".join(f"{k}={P.get(k)!r}" for k in (
        "concurrency_levels", "iterations", "think_time_between_queries_sec", "think_time_between_actions_sec",
        "include_user_actions") if k in P))

    _h("Load test: query latency (successful queries)")
    if M["load_ok"].empty:
        print("no successful load-test executions in this comparison")
    else:
        rows = []
        for lv in M["by_level"] + [None]:
            p50, p90_ = (lv["p50"], lv["p90"]) if lv else (K["p50"], K["p90"])
            mx = lv["max"] if lv else (K["max"]["a"], K["max"]["b"])
            e = lv["errors"] if lv else M["err_n"]
            n = (lv["n_a"], lv["n_b"]) if lv else (K["p50"]["n_a"], K["p50"]["n_b"])
            rows.append({"users": lv["level"] if lv else "all", "n A/B": f"{n[0]}/{n[1]}",
                         f"median {A}": fmt_ms(p50["a"]), f"median {B}": fmt_ms(p50["b"]),
                         "Δ median [95% CI]": f"{_fmt_rel(p50['rel'])} {_fmt_ci(p50['ci'])}",
                         f"p90 {A}": fmt_ms(p90_["a"]), f"p90 {B}": fmt_ms(p90_["b"]),
                         "Δ p90 [95% CI]": f"{_fmt_rel(p90_['rel'])} {_fmt_ci(p90_['ci'])}",
                         "p90 round pairs": _pairs_short(p90_) or "-",
                         f"max {A}": fmt_ms(mx[0]), f"max {B}": fmt_ms(mx[1]), "errors A/B": f"{e['A']}/{e['B']}"})
        print(pd.DataFrame(rows).to_string(index=False))
        j = K["journey"]
        what = "waits after each click" if M["replay_mode"] != "sequential" else "queries"
        print(f"\nClick-path wait (sum of the {what} of one full replay of {M['n_common']} queries, median; "
              f"{_plural(len(M['failing']), 'failing query', 'failing queries')} excluded): {A} {fmt_ms(j['a'])} | "
              f"{B} {fmt_ms(j['b'])} | Δ {_fmt_rel(j['rel'])} {_fmt_ci(j['ci'])} | replays {j['n_a']}/{j['n_b']}")
        if M["actions"]:
            print("Wait per click (median): " + "; ".join(
                f"{a['action_seq']}. {a['label']} {fmt_ms(a['a'])} -> {fmt_ms(a['b'])} ({_fmt_rel(a['rel'])}, {a['cls']})"
                for a in M["actions"]))

    _h("Single user, same queries in both models")
    for title, key, f in (("First run (warm-up) duration", "warmup", fmt_ms), ("Server CPU (trace)", "cpu", fmt_ms),
                          ("Server duration (trace)", "server_dur", fmt_ms),
                          ("Storage-engine queries (trace)", "se_queries", lambda v: "-" if _isnan(v) else f"{v:,.0f}")):
        k = K.get(key, {"a": float("nan"), "b": float("nan"), "rel": float("nan"), "ci": NAN2, "n": 0})
        print(f"{title:<32} {A} {f(k['a']):>10} | {B} {f(k['b']):>10} | Δ {_fmt_rel(k['rel']):>8} {_fmt_ci(k['ci'])} | "
              f"{_plural(k['n'], 'query', 'queries')}")
    if M["trace"]:
        print("trace matched: " + ", ".join(f"{LBL[k]} {m}/{t}" for k, (m, t) in M["trace"].items()))

    _h("Model state")
    for t in ("first_touch_ms", "last_refresh_utc", "capacity_id"):
        for _, r in prof[prof["item_type"] == t].sort_values(["captured_utc", "model_key"]).iterrows():
            v = r["value_text"] if not _isnan(r["value_text"]) else fmt_ms(r["value_num"])
            print(f"{t:<18} {r['model_label']:<10} {str(r['item_name']):<26} {v}")

    _h("Model definition differences (B vs A)")
    print("none" if M["defs"].empty else M["defs"].to_string(index=False))

    _h(f"Tables whose row count differs by more than {ROWS_LIST_PCT}%")
    tl = M["tables_listed"]
    if M["tables"].empty:
        print("not measured")
    else:
        tl = tl.assign(ratio=tl["ratio"].round(2)).rename(columns={"A": f"rows {A}", "B": f"rows {B}", "ratio": "ratio B/A"})
        print("none" if tl.empty else tl.to_string(float_format=lambda v: f"{v:,.2f}" if v < 100 else f"{v:,.0f}"))

    _h("Query results (warm-up, whole queries)")
    R = M["results"]
    print(f"{R['n']} queries: identical {R['identical']}, different row count {R['rows_diff']}, same rows but different "
          f"values {R['values_diff']}, errors {R['errors']}" if R["n"] else "not measured")
    at = _attrib_text(M)
    if at:
        print(f"Of the queries with different results: {at}.")

    det = M["per_query"]
    if len(det):
        mA, mB, cA, cB = f"median {A}", f"median {B}", f"CPU {A}", f"CPU {B}"
        fm = {mA: fmt_ms, mB: fmt_ms, cA: fmt_ms, cB: fmt_ms, "Δ% median": _fmt_rel, "Δ% CPU": _fmt_rel}
        cols = ["#", "action", "visual", "type", "result", mA, mB, "Δ% median", cA, cB, "Δ% CPU"]
        n = int(top_n_queries)
        by_lat = det.assign(_k=(det[mB] - det[mA]).abs()).sort_values("_k", ascending=False)
        if len(det) <= n:   # one table is enough when it holds every query
            _h(f"All {_plural(len(det), 'query', 'queries')}, by change of median latency under load")
            print(by_lat[cols].to_string(index=False, formatters=fm))
        else:
            _h(f"Top {n} queries by change of median latency under load")
            print(by_lat.head(n)[cols].to_string(index=False, formatters=fm))
            _h(f"Top {n} queries by change of server CPU")
            print(det.assign(_k=(det[cB] - det[cA]).abs()).sort_values("_k", ascending=False).head(n)[cols]
                  .to_string(index=False, formatters=fm))

    _h("Errors")
    err = ex[~ex["success"]]
    print("none" if err.empty else err.assign(error=err["error"].astype(str).str.slice(0, 150),
                                               **{"class": err["error"].map(error_class)})
          .groupby(["phase", "model_label", "class", "error"]).size().reset_index(name="count").to_string(index=False))


def _json_safe(v):
    if isinstance(v, dict):
        return {str(k): _json_safe(x) for k, x in v.items()}
    if isinstance(v, (list, tuple)):
        return [_json_safe(x) for x in v]
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (float, np.floating)):
        return None if math.isnan(v) else round(float(v), 3)
    if isinstance(v, (pd.Timestamp, datetime)):
        return v.isoformat()
    return v


def _trend_json(TV):
    """The trend for a pipeline / alert: p90 of B and A vs the range of the last comparable runs (None: < 3 of them)."""
    if TV is None or not TV["flags"]:
        return None
    out = {"fingerprint": TV["key"], "fingerprint_runs": len(TV["rows"]),
           "comparable": int(TV["rows"]["comparable"].astype(bool).sum()), "reference_runs": TV["n_ref"]}
    for col, k in (("p90_b", "b"), ("p90_a", "a")):
        if col in TV["flags"]:
            state, lo, hi, v = TV["flags"][col]
            out.update({f"{k}_p90": v, f"{k}_range": [lo, hi], f"{k}_position": state})
    return out


def run_result(M, text_file=None):
    """The machine-readable result (scheduled runs end with it: notebookutils.notebook.exit)."""
    K, NC, T = M["kpis"], M["number_check"], M["trust"]
    row = summary_row(M)
    return _json_safe({
        "status": M["status"], "comparison_id": M["cid"], "test_name": M["test_name"], "run_mode": M["run_mode"] or "interactive",
        "fingerprint": M["fingerprint"], "comparable": row["comparable"],
        "not_comparable": (_nc_json(row["metrics_json"], {}) or {}).get("not_comparable", []),
        "trust": {"passed": sum(t["ok"] for t in T), "total": len(T), "failing": [t["text"] for t in T if not t["ok"]]},
        "blockers": [h for h, _, _ in summary_sections(M)["blockers"]],
        "headline": {k: {"a": K[k]["a"], "b": K[k]["b"], "change_pct": K[k]["rel"], "ci95_pct": list(K[k]["ci"]),
                         "class": K[k]["cls"], "round_pairs": _pairs_short(K[k]) or None}
                     for k in ("p50", "p90", "journey", "cpu") if k in K},
        "number_check": {"status": NC["status"] if NC else None, "flagged": NC["flagged"] if NC else []},
        "errors_under_load": {M["labels"][k]: M["err_class"][k] for k in "AB"},
        "trend": _trend_json(M.get("trend")),
        "report_file": globals().get("REPORT_FILE"), "text_file": text_file,
    })


RUN_RESULT = None
if globals().get("REPORT_CID") is None:
    print("Nothing to summarize: dry run only (confirm_run = False), nothing was measured. To summarize a saved test, "
          "set report_comparison_id = '<comparison_id>' (Step 9 lists them) and run Steps 1, 2, 8 and 10.")
elif globals().get("REPORT_METRICS") is None:
    print(f"Nothing to summarize: comparison {REPORT_CID} has no query executions (it stopped before the warm-up in Step 4).")
else:
    _buf = io.StringIO()
    with contextlib.redirect_stdout(_buf):
        _text_summary(REPORT_METRICS)
    print(_buf.getvalue())
    if REPORT_CID == globals().get("COMPARISON_ID") and globals().get("RUN_ENABLED"):
        _txt = None
        if save_report_files:
            try:
                _txt = write_lakehouse_file(f"{REPORTS_DIR}/{REPORT_CID}.txt", _buf.getvalue())
            except Exception as _exc:
                print(f"  (could not save the text summary: {_err_text(_exc)[:200]})")
        RUN_RESULT = run_result(REPORT_METRICS, _txt)
        if save_report_files:
            try:
                write_lakehouse_file(f"{REPORTS_DIR}/{REPORT_CID}.json", json.dumps(RUN_RESULT, indent=1))
                print(f"Saved: {lakehouse_name}/{REPORTS_DIR}/{REPORT_CID}.html, .txt and .json")
            except Exception as _exc:
                print(f"  (could not save the result file: {_err_text(_exc)[:200]})")
if globals().get("SCHEDULED"):   # a pipeline / Monitor sees this JSON as the notebook's exit value
    notebookutils.notebook.exit(json.dumps(RUN_RESULT or {"status": "no result", "comparison_id": globals().get("COMPARISON_ID")}))